# Coding and Thematic Analysis

Created by [Matt Artz](https://www.mattartz.me/) — Advancing AI Anthropology through computational approaches to qualitative research.

## What This Notebook Does

This notebook analyzes qualitative data by coding it using both deductive and inductive approaches, and then building those codes up into themes. Rather than manually applying codes to hundreds of text segments and then building themes iteratively, you receive coded data as well as suggested themes.

Building on the foundations established by the Qualitative Codebook Builder and Interview Transcript Semantic Chunker, the notebook processes chunked text by applying the pre-defined deductive codes while inductively identifying emergent codes, and then it constructs themes from both approaches.

## Key Features

- **Data Integration**: Imports codebooks and transcripts from other AI Anthropology Toolkit notebooks
- **Three Coding Approaches**: Choose between deductive, inductive, or hybrid coding methodologies
- **Analytical Lens Awareness**: Automatically detects analytical lenses from codebook metadata (42 built-in lenses) and injects lens-specific framing into all AI prompts
- **Epistemic Pluralism**: Run the same data through multiple analytical lenses in parallel, then compare what each lens surfaces — the core of productive multi-perspective analysis
- **Cross-Lens Comparison**: Computes chunk-level agreement scores (Jaccard similarity) across lenses, identifies friction points where lenses diverge, and generates lens code profiles showing what each framework foregrounds
- **Code Application**: Uses Claude AI for coding across text segments
- **Theme Construction**: Builds hierarchical themes from coded data, with convergence tagging when multiple lenses are active
- **Research Context**: Project name, research question, and study description ground all prompts and appear in exports
- **Export Options**: Produces Excel workbooks (with Analysis Metadata and Lens Comparison sheets) and formatted Word documents with themes


## Workflow

1. Import deductive codebook from Qualitative Codebook Builder (CSV format) — supports multi-stance codebooks with `stance` column
2. Import interview chunks from Interview Transcript Processor (CSV format)
3. Configure research context (project name, research question, study description)
4. Review auto-detected analytical lens(es) from codebook metadata
5. Configure analysis parameters including coding approach and AI model
6. Apply deductive codes — in multi-stance mode, coding runs once per stance with lens-specific prompts
7. Discover inductive codes emerging from the data (if selected)
8. Cross-stance comparison: agreement matrix, friction points, lens code profiles (multi-stance mode)
9. Integrate codes and build themes (with cross-lens convergence tagging)
10. Generate visualizations (including lens agreement heatmap, friction distribution, lens profiles)
11. Export results with full lens metadata across all formats

## Applications

This notebook supports qualitative research from dissertation fieldwork to applied research projects. It's particularly useful for computational analysis using the notebooks in my AI Anthropology Toolkit, enabling researchers to maintain rigor while working at scale.

## Methodological Positioning

This represents "computational anthropology" - using AI to enhance rather than replace traditional qualitative methods. The notebook handles the time-intensive coding and thematic analysis process of interpretation and meaning-making.

The epistemic pluralism capability — running the same data through multiple analytical lenses — operationalizes the insight that different theoretical frameworks surface different aspects of social phenomena. Rather than choosing one framework and losing the rest, researchers can systematically compare what each lens foregrounds and obscures. Friction points (where lenses disagree most) are often the most analytically productive areas of the data.

**Important:** AI can assist with pattern recognition and consistency, but human expertise remains essential for contextual understanding.

## Target Audience

Designed for anthropologists and qualitative researchers working with qualitative data—from graduate students managing thesis interviews to research teams processing large datasets for applied projects.

## Technical Approach

Combines natural language processing with anthropological coding principles, using Claude AI to apply codes consistently while discovering emergent patterns through computational analysis.

All lens-aware features are automatically activated when the codebook contains a `stance` column or when multiple codebooks are uploaded in multi-stance mode. If your codebook has no `stance` column, or contains only one stance, the notebook behaves identically to previous versions — zero configuration changes needed.

## Contributing to AI Anthropology

This notebook contributes to the emerging field of AI Anthropology—which combines studying AI as cultural artifact, using AI to enhance ethnographic research, and applying anthropological insights to AI development (Artz, forthcoming). By open-sourcing these notebooks, this work advances the collective capacity of anthropologists to work effectively with computational methods.

## AI Anthropology Toolkit

This notebook is part of a growing suite of computational resources for anthropological research:


- **[Qualitative Codebook Builder](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Qualitative_Codebook_Builder.ipynb)** - AI-assisted development of qualitative coding frameworks
- **[Interview Transcript Semantic Chunker](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Interview_Transcript_Semantic_Chunker.ipynb)** - AI-assisted segmentation of interview transcripts
- **[Coding and Thematic Analysis](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Coding_and_Thematic_Analysis.ipynb)** (this notebook) - AI-assisted coding and thematic analysis of qualitative data

*Additional notebooks will be added to this toolkit as they are developed.*


<br>

---

<br>

## License & Attribution

This project is licensed under the Creative Commons Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) license. You may remix, adapt, and build upon the material for non-commercial purposes, provided you credit Matt Artz and link to the repository.

**Full license details**: https://creativecommons.org/licenses/by-nc/4.0/

If you use or adapt this project in your work, please cite:


> Built with the Coding and Thematic Analysis (Matt Artz, 2026) — https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Coding_and_Thematic_Analysis.ipynb


## Citation

If you use this notebook in your academic research, please cite:


> Artz, Matt. 2026. Coding and Thematic Analysis. Software.
Zenodo. https://doi.org/10.5281/zenodo.16728812

## References
Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

*Install required Python packages and import necessary libraries for mixed-method qualitative analysis. This includes AI integration, data processing, visualization, and export capabilities.*

In [ ]:
# Install required packages
!pip install anthropic pandas numpy matplotlib seaborn wordcloud plotly networkx openpyxl python-docx ipywidgets scikit-learn -q

import pandas as pd
import numpy as np
import re
import json
import time
import os
from collections import Counter, defaultdict
from datetime import datetime
from typing import List, Dict, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx

# Document generation
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.style import WD_STYLE_TYPE

# UI components
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Anthropic for Claude API
import anthropic

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("📊 AI-Assisted Qualitative Coding & Thematic Analysis")
print("=" * 50)
print(f"Setup completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✅ All dependencies installed successfully!")

## Configuration Interface

*Configure all analysis parameters including file uploads, coding approach, AI model selection, and API settings through an intuitive interface.*

In [ ]:
# ── Analytical Lens Registry (42 lenses) ──────────────────────────────────
# Prompt modifiers from the Codebook Builder so the coding notebook can
# inject the right analytical framing when it has a lens name from a CSV.

LENS_REGISTRY = {
    "Affect Theory": """Adopt an AFFECT THEORY analytical lens. Focus on:
- Pre-cognitive intensities, bodily capacities, and affective atmospheres
- How affect circulates between bodies, objects, and environments
- The distinction between affect (pre-personal intensity) and emotion (named feeling)
- Affective labor, contagion, and the politics of feeling
- Codes should capture felt intensities and atmospheric qualities beyond named emotions.""",

    "Anarchist / Anti-authoritarian": """Adopt an ANARCHIST / ANTI-AUTHORITARIAN analytical lens. Focus on:
- Mutual aid, solidarity, and horizontal forms of organizing
- Resistance to hierarchical authority, state power, and institutional coercion
- Prefigurative politics and autonomous community practices
- Direct action, self-governance, and voluntary association
- Codes should illuminate anti-hierarchical practices and grassroots self-organization.""",

    "Business / Organizational": """Adopt a BUSINESS / ORGANIZATIONAL analytical lens. Focus on:
- Organizational culture, workplace norms, and institutional practices
- Management strategies, corporate decision-making, and innovation processes
- How cultural context shapes business practices and economic behavior
- Stakeholder dynamics, corporate social responsibility, and market logics
- Codes should capture organizational processes and workplace cultural dynamics.""",

    "Cognitive": """Adopt a COGNITIVE analytical lens. Focus on:
- Mental models, schemas, and categorization processes
- How cultural knowledge is organized, stored, and retrieved
- Decision-making heuristics, reasoning patterns, and folk theories
- Distributed cognition and the relationship between culture and thought
- Codes should capture cognitive processes, mental models, and reasoning patterns.""",

    "Computational / Digital": """Adopt a COMPUTATIONAL / DIGITAL analytical lens. Focus on:
- Digital platforms, algorithmic mediation, and data-driven systems
- How technology shapes social interaction, identity, and community
- Digital labor, platform economies, and datafication of everyday life
- Methodological innovation in studying online and hybrid social worlds
- Codes should capture digital mediation, platform dynamics, and human-technology entanglements.""",

    "Critical": """Adopt a CRITICAL analytical lens. Focus on:
- Power dynamics, structural inequalities, and systemic constraints
- Ideological assumptions and hegemonic narratives
- Resistance, agency within constraint, and counter-narratives
- How macro-level forces shape individual experience
- Codes should illuminate domination, marginalization, and structural forces.""",

    "Critical Medical": """Adopt a CRITICAL MEDICAL analytical lens. Focus on:
- Biomedical power, medicalization, and the social construction of disease categories
- How structural inequalities produce differential health outcomes
- Patient experience within institutional medical systems
- Pharmaceutical politics, clinical authority, and epistemic hierarchies in healthcare
- Codes should illuminate how medical systems produce and reproduce social inequalities.""",

    "Critical Race": """Adopt a CRITICAL RACE analytical lens. Focus on:
- Racial formations, systemic racism, and the social construction of race
- Intersections of race with class, gender, sexuality, and other axes of power
- Counter-narratives, racial consciousness, and resistance to white supremacy
- Institutional racism, colorblind ideology, and racial microaggressions
- Codes should center racial dynamics and illuminate how racism operates structurally and interpersonally.""",

    "Decolonial": """Adopt a DECOLONIAL analytical lens. Focus on:
- Colonial matrices of power and the persistence of coloniality in knowledge production
- Epistemic violence, subaltern knowledge, and border thinking
- Pluriversal alternatives to Western-centric universalism
- Land, sovereignty, and the politics of who produces legitimate knowledge
- Codes should foreground resistance to colonial epistemologies and center marginalized knowledge systems.""",

    "Design Anthropology": """Adopt a DESIGN ANTHROPOLOGY analytical lens. Focus on:
- Design processes as cultural practices of future-making and world-shaping
- Co-creation, participatory design, and collaborative prototyping
- How designed objects and systems mediate social relations
- The politics of design decisions and their material consequences
- Codes should capture design practices, material interventions, and speculative futures.""",

    "Economic Anthropology": """Adopt an ECONOMIC ANTHROPOLOGY analytical lens. Focus on:
- Diverse economies, exchange systems, and forms of value creation
- Labor, livelihoods, and the social embeddedness of economic activity
- Gift economies, reciprocity, redistribution, and market logics
- How economic practices are culturally constituted and morally evaluated
- Codes should capture economic practices, value systems, and material livelihood strategies.""",

    "Environmental / Political Ecology": """Adopt an ENVIRONMENTAL / POLITICAL ECOLOGY analytical lens. Focus on:
- Human-environment relations and the politics of natural resource access and control
- Environmental knowledge, ecological practices, and landscape management
- Climate change, environmental justice, and uneven ecological vulnerabilities
- How power shapes who benefits from and bears the costs of environmental change
- Codes should capture environmental practices, resource politics, and ecological knowledge systems.""",

    "Evaluation": """Adopt an EVALUATION analytical lens. Focus on:
- Program effectiveness, implementation fidelity, and practical outcomes
- Stakeholder perspectives, needs assessments, and utilization-focused analysis
- What works, for whom, under what conditions, and why
- Actionable findings, recommendations, and evidence for decision-making
- Codes should capture program processes, outcomes, and stakeholder experiences relevant to improvement.""",

    "Feminist": """Adopt a FEMINIST analytical lens. Focus on:
- Gendered power relations, patriarchal structures, and intersecting oppressions
- Embodied experience, reproductive labor, and the politics of care
- How gender norms are produced, performed, and contested
- Feminist epistemology: situated knowledge, standpoint, and reflexivity
- Codes should illuminate gendered dynamics, intersectional power, and women's/marginalized experiences.""",

    "Hermeneutic": """Adopt a HERMENEUTIC analytical lens. Focus on:
- Processes of interpretation and the hermeneutic circle (part-whole relationships)
- How meaning is constituted through dialogue between text, context, and interpreter
- Pre-understanding, prejudice (in the Gadamerian sense), and horizons of meaning
- The conditions under which understanding becomes possible
- Codes should capture interpretive processes, meaning-making, and the contextual conditions of understanding.""",

    "Historical / Archival": """Adopt a HISTORICAL / ARCHIVAL analytical lens. Focus on:
- Historical processes, temporal change, and the longue durée of social phenomena
- Archival sources, documentary evidence, and the politics of the archive
- Collective memory, oral history, and contested narratives of the past
- How present conditions are shaped by historical trajectories and path dependencies
- Codes should capture historical processes, temporal dynamics, and the relationship between past and present.""",

    "Indigenous Methodologies": """Adopt an INDIGENOUS METHODOLOGIES analytical lens. Focus on:
- Relational accountability, reciprocity, and community-centered research ethics
- Indigenous knowledge systems, oral traditions, and land-based epistemologies
- Sovereignty over knowledge production and community self-determination
- Storytelling as method, ceremony as research practice, and place-based knowing
- Codes should center Indigenous ways of knowing, relational ethics, and community sovereignty over knowledge.""",

    "Infrastructure Studies": """Adopt an INFRASTRUCTURE STUDIES analytical lens. Focus on:
- Built systems, technical networks, and their social embedding
- Maintenance, repair, breakdown, and the labor of keeping systems running
- How infrastructure becomes visible only upon failure
- The politics of infrastructure: who is served, who is excluded, and who maintains
- Codes should capture infrastructural arrangements, maintenance practices, and the social life of technical systems.""",

    "Interpretive": """Adopt an INTERPRETIVE analytical lens. Focus on:
- Meaning-making processes and subjective experiences
- How participants interpret and construct their social worlds
- Lived experience, emotions, and sense-making narratives
- Cultural symbols, language use, and interpretive frameworks
- Codes should capture how subjects understand their own experiences.""",

    "Legal / Rights-based": """Adopt a LEGAL / RIGHTS-BASED analytical lens. Focus on:
- Legal frameworks, rights claims, and normative orders
- How law is produced, interpreted, and experienced in everyday life
- Legal pluralism, customary law, and the gaps between law-on-paper and law-in-practice
- Justice, access to legal remedies, and the politics of rights discourse
- Codes should capture legal processes, rights claims, and the lived experience of legal systems.""",

    "Linguistic": """Adopt a LINGUISTIC analytical lens. Focus on:
- Language structure, discourse patterns, and communicative practices
- How language constitutes social reality, identity, and relationships
- Code-switching, register, indexicality, and language ideologies
- The pragmatics of speech: what utterances do in social context
- Codes should capture linguistic patterns, discourse strategies, and the social work of language.""",

    "Material Culture / Object-oriented": """Adopt a MATERIAL CULTURE / OBJECT-ORIENTED analytical lens. Focus on:
- Objects, artifacts, and their social lives and trajectories
- Material agency: how things shape, enable, and constrain human action
- Human-object entanglements, assemblages, and material semiosis
- Craft, making, consumption, and the cultural biography of things
- Codes should capture object-human relationships, material practices, and the agency of things.""",

    "Medical / Health (interpretive)": """Adopt a MEDICAL / HEALTH (INTERPRETIVE) analytical lens. Focus on:
- Illness experience, suffering narratives, and explanatory models of sickness
- Healing practices, therapeutic pluralism, and health-seeking behavior
- The body as a site of cultural inscription and lived experience
- How individuals and communities make sense of health, illness, and care
- Codes should capture illness experiences, healing narratives, and culturally situated health practices.""",

    "Migration / Mobility Studies": """Adopt a MIGRATION / MOBILITY STUDIES analytical lens. Focus on:
- Movement, displacement, and the politics of borders and belonging
- Transnational connections, diasporic identities, and mobile livelihoods
- Immigration regimes, documentation status, and bureaucratic encounters
- How mobility and immobility are structured by race, class, gender, and geopolitics
- Codes should capture mobility practices, border experiences, and transnational social fields.""",

    "Mixed-methods": """Adopt a MIXED-METHODS analytical lens. Focus on:
- Phenomena amenable to both qualitative interpretation and quantitative measurement
- Patterns that can be triangulated across data types and methods
- Convergence and divergence between qualitative and quantitative findings
- Practical outcomes, transferability, and generalizable insights
- Codes should be concrete enough for quantification while preserving qualitative richness.""",

    "Multi-sited": """Adopt a MULTI-SITED analytical lens. Focus on:
- Connections, flows, and circulations across multiple geographic or social sites
- How phenomena manifest differently across locations while remaining connected
- Following people, things, metaphors, or conflicts across institutional and spatial boundaries
- The construction of the field through tracing relations rather than bounding a single site
- Codes should capture cross-site connections, circulations, and the multi-scalar nature of phenomena.""",

    "Multispecies / More-than-human": """Adopt a MULTISPECIES / MORE-THAN-HUMAN analytical lens. Focus on:
- Interspecies relations, symbiosis, and multispecies entanglements
- Non-human agencies, animal perspectives, and more-than-human sociality
- How human worlds are co-constituted with plants, animals, fungi, and microbes
- Ecological intimacies, companion species, and the breakdown of nature/culture divides
- Codes should capture interspecies relationships and extend analytical attention beyond the exclusively human.""",

    "Narrative / Life History": """Adopt a NARRATIVE / LIFE HISTORY analytical lens. Focus on:
- Storytelling practices, narrative structure, and biographical trajectories
- How people construct coherent life stories from fragmented experiences
- Plot, temporality, emplotment, and turning points in personal narratives
- The social and cultural shaping of what stories can be told and how
- Codes should capture narrative structures, life story elements, and the temporal organization of experience.""",

    "Ontological": """Adopt an ONTOLOGICAL analytical lens. Focus on:
- Ontological difference: multiple worlds rather than multiple perspectives on one world
- What exists, what kinds of beings populate the world, and how reality is constituted
- Taking seriously non-Western ontologies without reducing them to cultural belief
- The politics of ontology: whose reality counts and how worlds are made and unmade
- Codes should attend to ontological commitments, world-making practices, and radical difference in what is real.""",

    "Performance / Performativity": """Adopt a PERFORMANCE / PERFORMATIVITY analytical lens. Focus on:
- Social performances, staged interactions, and ritual enactments
- How identities, genders, and social categories are performatively constituted
- The distinction between theatrical performance and performativity (Butler, Austin)
- Embodied practice, gesture, spectacle, and the aesthetics of everyday interaction
- Codes should capture performative acts, ritual practices, and how social realities are enacted into being.""",

    "Phenomenological": """Adopt a PHENOMENOLOGICAL analytical lens. Focus on:
- First-person lived experience and the structures of consciousness
- Pre-reflective and embodied dimensions of experience
- Epoché: bracketing assumptions to attend to phenomena as they appear
- Temporality, spatiality, and intersubjective dimensions of experience
- Codes should capture the essential structures of how phenomena are experienced.""",

    "Political Economy / Marxian": """Adopt a POLITICAL ECONOMY / MARXIAN analytical lens. Focus on:
- Class relations, labor exploitation, and capital accumulation processes
- How economic structures shape social relations, consciousness, and everyday life
- Commodification, alienation, and the production of surplus value
- The relationship between base and superstructure, ideology, and material conditions
- Codes should illuminate class dynamics, labor processes, and structural economic forces.""",

    "Postcolonial": """Adopt a POSTCOLONIAL analytical lens. Focus on:
- Colonial legacies and their ongoing effects on formerly colonized societies
- Subaltern voices, representation, and the question of who speaks for whom
- Hybridity, mimicry, and the ambivalence of colonial encounters
- Imperial knowledge systems and their persistent influence on contemporary institutions
- Codes should illuminate colonial residues, subaltern perspectives, and the enduring asymmetries of imperial power.""",

    "Practice Theory": """Adopt a PRACTICE THEORY analytical lens. Focus on:
- Embodied routines, habitual dispositions, and practical know-how
- How social structures are reproduced and transformed through everyday practices
- Habitus, fields, and the interplay of structure and agency (Bourdieu)
- Material arrangements, bodily competences, and shared understandings that compose practices
- Codes should capture recurring practices, embodied routines, and how social order is produced through doing.""",

    "Psychoanalytic": """Adopt a PSYCHOANALYTIC analytical lens. Focus on:
- Unconscious processes, repression, and the psychic life of social structures
- Desire, fantasy, identification, and their role in cultural reproduction
- Transference, projection, and affective dynamics in social relationships
- How collective anxieties, traumas, and defenses shape cultural formations
- Codes should capture unconscious dynamics, psychic investments, and the emotional undercurrents of social life.""",

    "Psychological": """Adopt a PSYCHOLOGICAL analytical lens. Focus on:
- Individual experience, mental processes, and psychological well-being
- Motivation, coping strategies, and adaptive behavior
- Self-concept, identity formation, and developmental trajectories
- Cognitive and emotional responses to social and environmental conditions
- Codes should capture individual psychological processes, subjective states, and person-level experiences.""",

    "Public / Engaged": """Adopt a PUBLIC / ENGAGED analytical lens. Focus on:
- Public impact, civic engagement, and the social responsibility of research
- Community partnerships, participatory processes, and collaborative knowledge production
- How research can serve communities and contribute to social change
- Translation of findings for public audiences and policy-relevant insights
- Codes should capture publicly relevant dynamics, community needs, and opportunities for engaged scholarship.""",

    "Queer Theory": """Adopt a QUEER THEORY analytical lens. Focus on:
- Heteronormativity, compulsory heterosexuality, and the regulation of desire
- Gender fluidity, non-binary identities, and the instability of sexual categories
- How deviance, transgression, and non-normativity challenge social order
- Queer temporalities, kinship formations, and anti-normative world-making
- Codes should illuminate heteronormative assumptions, gender/sexual fluidity, and queer modes of being.""",

    "Semiotic": """Adopt a SEMIOTIC analytical lens. Focus on:
- Sign systems, codes, and the cultural production of meaning
- How signs (icons, indexes, symbols) mediate social reality
- Intertextuality, connotation, and the layering of cultural meanings
- Semiotic ideologies: beliefs about how signs work and what they represent
- Codes should capture semiotic processes, sign relationships, and the cultural machinery of meaning-making.""",

    "Structuralist / Post-structuralist": """Adopt a STRUCTURALIST / POST-STRUCTURALIST analytical lens. Focus on:
- Deep structures, binary oppositions, and underlying systems of classification
- Discourse, power/knowledge, and how language constitutes social reality (Foucault)
- The instability, deferral, and deconstruction of meaning (Derrida)
- How subjects are constituted through discursive practices and disciplinary regimes
- Codes should capture structural patterns, discursive formations, and the constitutive role of language and classification.""",

    "STS / Actor-Network": """Adopt an STS / ACTOR-NETWORK THEORY analytical lens. Focus on:
- Heterogeneous networks of human and non-human actants
- Translation, enrollment, and stabilization of actor-networks
- How facts, technologies, and social order are co-produced
- Controversies, black-boxing, and obligatory passage points
- Codes should trace associations between actants without privileging human agency.""",

    "Visual / Sensory": """Adopt a VISUAL / SENSORY analytical lens. Focus on:
- Visual culture, image-making, and the politics of representation
- Sensory experience: sight, sound, smell, taste, touch as modes of knowing
- How non-textual media (photographs, film, objects, soundscapes) convey meaning
- Embodied perception, synaesthesia, and culturally shaped sensoriums
- Codes should capture visual and sensory dimensions of experience, including non-verbal and non-textual data.""",
}

# Case-insensitive lookup for lens matching
_LENS_LOOKUP = {name.lower().strip(): name for name in LENS_REGISTRY}


def _resolve_stance_to_lens(stance_key):
    """Map a multi-stance codebook key to a LENS_REGISTRY entry name, if possible."""
    if not stance_key:
        return None
    if stance_key in LENS_REGISTRY:
        return stance_key
    lookup_key = stance_key.lower().strip().replace('_', ' ')
    matched = _LENS_LOOKUP.get(lookup_key)
    if matched:
        return matched
    for registry_key in _LENS_LOOKUP:
        if registry_key.replace(' ', '_') == stance_key.lower() or registry_key.replace('/', '_').replace(' ', '_').lower() == stance_key.lower():
            return _LENS_LOOKUP[registry_key]
    for registry_key, display_name in _LENS_LOOKUP.items():
        if stance_key.lower().replace('_', ' ') in registry_key or registry_key in stance_key.lower().replace('_', ' '):
            return display_name
    print(f"⚠️ Analytical lens '{stance_key}' not found in the registry — using a generic framing. "
          f"Check the lens name against the Codebook Builder.")
    return None


def build_lens_context(lens_name=None, stance_key=None):
    """Build research context + lens framing string for injection into any prompt.

    Args:
        lens_name: Explicit lens display name (from detected_lenses).
        stance_key: A multi-stance codebook key (e.g. 'interpretivist') — will be
                    resolved to a LENS_REGISTRY entry automatically.

    If neither is provided, uses auto-detected lenses from config.
    Returns empty string when no lens info is available (backward compatible).
    """
    parts = []

    rq = config.get('research_question', '').strip()
    sd = config.get('study_description', '').strip()
    pn = config.get('project_name', '').strip()

    if rq or sd or pn:
        parts.append("RESEARCH CONTEXT:")
        if pn:
            parts.append(f"Project: {pn}")
        if rq:
            parts.append(f"Research Question: {rq}")
        if sd:
            parts.append(f"Study Context: {sd}")
        parts.append("")

    resolved_lens = lens_name
    if resolved_lens is None and stance_key:
        resolved_lens = _resolve_stance_to_lens(stance_key)

    detected = config.get('detected_lenses', [])
    modifiers = config.get('lens_prompt_modifiers', {})

    if resolved_lens and resolved_lens in LENS_REGISTRY:
        parts.append(f"ANALYTICAL FRAMEWORK ({resolved_lens}):")
        parts.append(LENS_REGISTRY[resolved_lens])
        parts.append("")
        parts.append("Apply codes with attention to these analytical priorities.")
        parts.append("")
    elif resolved_lens and resolved_lens in modifiers:
        parts.append(f"ANALYTICAL FRAMEWORK ({resolved_lens}):")
        parts.append(modifiers[resolved_lens])
        parts.append("")
        parts.append("Apply codes with attention to these analytical priorities.")
        parts.append("")
    elif resolved_lens is None and len(detected) == 1:
        name = detected[0]
        modifier = LENS_REGISTRY.get(name, modifiers.get(name, ''))
        if modifier:
            parts.append(f"ANALYTICAL FRAMEWORK ({name}):")
            parts.append(modifier)
            parts.append("")
            parts.append("Apply codes with attention to these analytical priorities.")
            parts.append("")
    elif resolved_lens is None and len(detected) > 1:
        names = ', '.join(detected)
        parts.append(f"ANALYTICAL FRAMEWORK (epistemic pluralism across {len(detected)} lenses: {names}):")
        parts.append("This analysis uses multiple analytical lenses to surface how different frameworks foreground different aspects of the data.")
        parts.append("")

    return "\n".join(parts)


def _detect_lenses_from_codebook(codebook_df):
    """Auto-detect analytical lenses from a codebook DataFrame's 'stance' column."""
    detected = []
    modifiers = {}
    lens_codebooks = {}

    if 'stance' not in codebook_df.columns:
        return detected, modifiers, lens_codebooks

    stance_values = codebook_df['stance'].dropna().str.strip()
    stance_values = stance_values[stance_values != '']
    unique_stances = stance_values.unique().tolist()

    for stance_name in unique_stances:
        lookup_key = stance_name.lower().strip()
        matched_name = _LENS_LOOKUP.get(lookup_key)

        if matched_name:
            detected.append(matched_name)
            modifiers[matched_name] = LENS_REGISTRY[matched_name]
            mask = codebook_df['stance'].str.strip().str.lower() == lookup_key
            lens_codebooks[matched_name] = codebook_df[mask].copy()
        else:
            print(f"⚠️ Analytical lens '{stance_name}' not found in the registry — using a generic framing. "
                  f"Check the lens name against the Codebook Builder.")
            detected.append(stance_name)
            modifiers[stance_name] = f"Analytical lens: {stance_name}. Apply codes through this analytical framework."
            mask = codebook_df['stance'].str.strip() == stance_name
            lens_codebooks[stance_name] = codebook_df[mask].copy()

    return detected, modifiers, lens_codebooks


# ── Global configuration storage ──────────────────────────────────────────
config = {
    'codebook_df': None,
    'transcript_df': None,
    'coding_approach': 'hybrid',
    'ai_model': 'claude-sonnet-5',
    'api_key': '',
    'output_folder': '1_coding_and_initial_themes',
    'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S'),
    'use_ai': True,
    'checkpoint_file': 'coding_in_progress.json',
    'chunk_id_column': 'chunk_id',
    # Multi-stance support
    'multi_stance_mode': False,
    'codebook_stances': {},       # stance_key -> DataFrame
    'active_stances': [],          # list of stance keys to process
    'stance_results': {},          # stance_key -> analysis results
    # Research context
    'project_name': '',
    'research_question': '',
    'study_description': '',
    # Lens awareness
    'detected_lenses': [],           # list of lens display names found in codebook(s)
    'lens_codebooks': {},            # {lens_name: codebook_df_subset}
    'lens_prompt_modifiers': {},     # {lens_name: prompt_modifier_string}
}

# Create output folder
os.makedirs(config['output_folder'], exist_ok=True)

# ── Research Context Widgets ──────────────────────────────────────────────
project_name_input = widgets.Text(
    placeholder='e.g., Healthcare Worker Experience Study',
    description='',
    layout=widgets.Layout(width='100%')
)

research_question_input = widgets.Textarea(
    placeholder='e.g., How do healthcare practitioners experience and adapt to AI-driven diagnostic tools?',
    description='',
    layout=widgets.Layout(width='100%', height='70px')
)

study_description_input = widgets.Textarea(
    placeholder='e.g., Semi-structured interviews with 30 nurses and physicians at three urban hospitals...',
    description='',
    layout=widgets.Layout(width='100%', height='55px')
)

# Lens display (read-only, auto-populated after codebook load)
lens_status_display = widgets.HTML(
    value="<span style='color: #8B8C89; font-size: 13px;'>Upload a codebook to detect analytical lenses</span>"
)

# ── Analysis Mode Toggle ──
analysis_mode = widgets.RadioButtons(
    options=['Single Codebook', 'Multi-Stance Codebooks'],
    value='Single Codebook',
    description='',
    style={'description_width': 'initial'}
)

# ── Single codebook upload ──
codebook_upload = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Codebook CSV:',
    button_style='primary',
    style={'button_color': '#6096BA'}
)

# ── Multi-stance codebook upload ──
multi_codebook_upload = widgets.FileUpload(
    accept='.csv',
    multiple=True,
    description='Codebook CSVs:',
    button_style='primary',
    style={'button_color': '#6096BA'}
)

transcript_upload = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Transcript CSV:',
    button_style='primary',
    style={'button_color': '#6096BA'}
)

# Coding approach selection
coding_approach = widgets.RadioButtons(
    options=['Deductive Only', 'Inductive Only', 'Hybrid (Both)'],
    value='Hybrid (Both)',
    description='',
    style={'description_width': 'initial'}
)

# AI model selection
ai_model_dropdown = widgets.Dropdown(
    options=[
        ('Claude Sonnet 5 (Recommended)', 'claude-sonnet-5'),
        ('Claude Haiku 4.5 (Fast & Affordable)', 'claude-haiku-4-5'),
        ('Claude Opus 4.8 (Most Capable)', 'claude-opus-4-8')
    ],
    value='claude-sonnet-5',
    description='Claude Model:',
    style={'description_width': 'initial'}
)

# API key input
api_key_input = widgets.Password(
    placeholder='Enter your Anthropic API key',
    description='API Key:',
    style={'description_width': 'initial'}
)

# Status output
status_output = widgets.Output()

# Stance detection output (for multi-stance mode)
stance_status = widgets.HTML(value='')

# Containers that toggle based on mode
single_codebook_box = widgets.VBox([codebook_upload])
multi_codebook_box = widgets.VBox([multi_codebook_upload, stance_status])
multi_codebook_box.layout.display = 'none'

def on_mode_change(change):
    """Toggle between single and multi-stance codebook upload widgets."""
    if change['new'] == 'Multi-Stance Codebooks':
        single_codebook_box.layout.display = 'none'
        multi_codebook_box.layout.display = ''
        config['multi_stance_mode'] = True
    else:
        single_codebook_box.layout.display = ''
        multi_codebook_box.layout.display = 'none'
        config['multi_stance_mode'] = False

analysis_mode.observe(on_mode_change, names='value')

def _detect_stance_from_csv(df, filename):
    """Auto-detect stance from CSV stance column or filename."""
    if 'stance' in df.columns:
        unique_stances = df['stance'].dropna().unique()
        if len(unique_stances) == 1:
            return str(unique_stances[0]).strip().lower().replace(' ', '_')

    import re
    name = os.path.splitext(filename)[0].lower()
    match = re.search(r'codebook[_\s-]+(.+)', name)
    if match:
        return match.group(1).strip().replace(' ', '_')

    return re.sub(r'[^a-z0-9_]', '_', name).strip('_')

def _update_stance_status():
    """Update the stance detection display."""
    if not config['codebook_stances']:
        stance_status.value = ''
        return

    items = []
    for key in config['active_stances']:
        df = config['codebook_stances'][key]
        lens_name = _resolve_stance_to_lens(key)
        lens_label = f' → <em>{lens_name}</em>' if lens_name else ''
        items.append(f'<li><strong>{key}</strong>: {len(df)} codes{lens_label}</li>')

    stance_status.value = (
        '<div style="background:#E7ECEF; padding:10px; border-radius:10px; margin-top:8px; border-left:5px solid #274C77;">'
        f'<strong style="color:#274C77;">Detected {len(config["active_stances"])} stances:</strong>'
        f'<ul style="margin:5px 0 0 15px; padding:0;">{"".join(items)}</ul>'
        '</div>'
    )

# Process buttons
test_button = widgets.Button(
    description='🧪 Test Setup',
    button_style='info',
    style={'button_color': '#A3CEF1'}
)

process_button = widgets.Button(
    description='✅ Apply Configuration',
    button_style='success',
    style={'button_color': '#6096BA'}
)

def test_setup(b):
    with status_output:
        clear_output(wait=True)
        print("🧪 Testing configuration...")

        if config['multi_stance_mode']:
            codebook_ok = bool(multi_codebook_upload.value)
        else:
            codebook_ok = bool(codebook_upload.value)
        transcript_ok = bool(transcript_upload.value)
        api_ok = bool(api_key_input.value)

        mode_label = "Multi-Stance" if config['multi_stance_mode'] else "Single"
        print(f"📊 Codebook file(s) [{mode_label}]: {'✅' if codebook_ok else '❌'}")
        print(f"📝 Transcript file: {'✅' if transcript_ok else '❌'}")
        print(f"🔑 API Key: {'✅' if api_ok else '❌'}")

        if codebook_ok and transcript_ok and api_ok:
            print("\n✅ All tests passed! Ready to start analysis.")
        else:
            print("\n❌ Please address the issues above before proceeding.")

def load_files(b):
    with status_output:
        clear_output(wait=True)

        try:
            # ── Load codebook(s) ──
            if config['multi_stance_mode']:
                if not multi_codebook_upload.value:
                    print("❌ Please upload codebook CSV files")
                    return

                config['codebook_stances'] = {}
                config['active_stances'] = []

                for filename, file_info in multi_codebook_upload.value.items():
                    content = file_info['content']
                    df = pd.read_csv(pd.io.common.BytesIO(content))

                    required_cols = ['code_label', 'definition']
                    missing = [c for c in required_cols if c not in df.columns]
                    if missing:
                        print(f"❌ {filename}: Missing required columns: {missing}")
                        return

                    stance_key = _detect_stance_from_csv(df, filename)
                    config['codebook_stances'][stance_key] = df
                    config['active_stances'].append(stance_key)
                    print(f"✅ Loaded codebook for stance '{stance_key}': {len(df)} codes")

                if len(config['active_stances']) < 2:
                    print("⚠️ Only 1 stance detected — cross-stance comparison requires at least 2.")
                    print("   Analysis will still run, but comparison features will be skipped.")

                # Detect lenses from multi-stance codebooks
                config['detected_lenses'] = []
                config['lens_prompt_modifiers'] = {}
                config['lens_codebooks'] = {}
                for sk in config['active_stances']:
                    lens_name = _resolve_stance_to_lens(sk)
                    if lens_name:
                        config['detected_lenses'].append(lens_name)
                        config['lens_prompt_modifiers'][lens_name] = LENS_REGISTRY.get(lens_name, '')
                        config['lens_codebooks'][lens_name] = config['codebook_stances'][sk]
                    else:
                        config['detected_lenses'].append(sk)
                        config['lens_prompt_modifiers'][sk] = f"Analytical lens: {sk}. Apply codes through this analytical framework."
                        config['lens_codebooks'][sk] = config['codebook_stances'][sk]

                _update_stance_status()
                _update_lens_status_display()

                first_key = config['active_stances'][0]
                config['codebook_df'] = config['codebook_stances'][first_key]
                print(f"\n📊 {len(config['active_stances'])} stance codebooks loaded")

            else:
                if codebook_upload.value:
                    content = list(codebook_upload.value.values())[0]['content']
                    config['codebook_df'] = pd.read_csv(pd.io.common.BytesIO(content))
                    print(f"✅ Codebook loaded: {len(config['codebook_df'])} codes")

                    required_cols = ['code_label', 'definition']
                    missing_required = [col for col in required_cols if col not in config['codebook_df'].columns]
                    if missing_required:
                        print(f"❌ Error: Missing required columns in codebook: {missing_required}")
                        return

                    # Auto-detect analytical lenses from stance column
                    detected, modifiers, lens_codebooks = _detect_lenses_from_codebook(config['codebook_df'])
                    config['detected_lenses'] = detected
                    config['lens_prompt_modifiers'] = modifiers
                    config['lens_codebooks'] = lens_codebooks

                    # If multiple lenses detected in single codebook, enable multi-stance mode
                    if len(detected) > 1:
                        config['multi_stance_mode'] = True
                        config['codebook_stances'] = {}
                        config['active_stances'] = []
                        for lens_name in detected:
                            stance_key = lens_name.lower().replace(' ', '_').replace('/', '_')
                            config['codebook_stances'][stance_key] = lens_codebooks[lens_name]
                            config['active_stances'].append(stance_key)
                        _update_stance_status()

                    _update_lens_status_display()
                else:
                    print("❌ Please upload a codebook CSV file")
                    return

            # ── Load transcripts ──
            if transcript_upload.value:
                content = list(transcript_upload.value.values())[0]['content']
                config['transcript_df'] = pd.read_csv(pd.io.common.BytesIO(content))
                print(f"✅ Transcripts loaded: {len(config['transcript_df'])} chunks")

                if 'text' not in config['transcript_df'].columns:
                    print("❌ Error: Transcript file must have a 'text' column")
                    return

                id_candidates = ['chunk_id', 'id', 'segment_id', 'ID', 'Chunk_ID', 'chunk_ID', 'index']
                config['chunk_id_column'] = None
                for col in id_candidates:
                    if col in config['transcript_df'].columns:
                        config['chunk_id_column'] = col
                        break

                if config['chunk_id_column'] is None:
                    config['transcript_df']['chunk_id'] = [f"chunk_{i}" for i in range(len(config['transcript_df']))]
                    config['chunk_id_column'] = 'chunk_id'
                    print(f"⚠️ No ID column found — created 'chunk_id' from row index")
                else:
                    print(f"✅ Using '{config['chunk_id_column']}' as chunk identifier")
            else:
                print("❌ Please upload a transcript CSV file")
                return

            # ── Update configuration ──
            config['coding_approach'] = coding_approach.value.lower()
            config['ai_model'] = ai_model_dropdown.value
            config['api_key'] = api_key_input.value
            config['project_name'] = project_name_input.value.strip()
            config['research_question'] = research_question_input.value.strip()
            config['study_description'] = study_description_input.value.strip()

            if not config['api_key']:
                print("❌ Please enter your Anthropic API key")
                return

            print("\n✅ Configuration complete! Ready to start analysis.")
            print(f"\n📁 Output folder: {config['output_folder']}")
            print(f"🔧 Coding approach: {config['coding_approach']}")
            print(f"🧠 AI Model: {ai_model_dropdown.label}")
            if config['research_question']:
                print(f"🔬 Research question: {config['research_question'][:80]}...")
            if config['multi_stance_mode']:
                print(f"🔀 Multi-stance mode: {len(config['active_stances'])} stances ({', '.join(config['active_stances'])})")
            if config['detected_lenses']:
                print(f"🔍 Detected lenses: {', '.join(config['detected_lenses'])}")

        except Exception as e:
            print(f"❌ Error loading files: {e}")
            import traceback
            traceback.print_exc()


def _update_lens_status_display():
    """Update the lens status display widget based on detected lenses."""
    detected = config.get('detected_lenses', [])

    if len(detected) == 0:
        lens_status_display.value = (
            "<span style='color: #8B8C89; font-size: 13px;'>"
            "No analytical lenses detected (no <code>stance</code> column). "
            "Coding will use standard prompts.</span>"
        )
    elif len(detected) == 1:
        lens_status_display.value = (
            f"<span style='color: #274C77; font-size: 13px;'>"
            f"🔍 Detected analytical lens: <strong>{detected[0]}</strong></span>"
        )
    else:
        names_html = ', '.join(f"<strong>{n}</strong>" for n in detected)
        lens_status_display.value = (
            f"<span style='color: #274C77; font-size: 13px;'>"
            f"🔍 Detected {len(detected)} analytical lenses (epistemic pluralism): {names_html}<br>"
            f"<span style='color: #6096BA; font-size: 12px;'>"
            f"Multi-lens mode: coding will run once per lens, then compare results.</span></span>"
        )


test_button.on_click(test_setup)
process_button.on_click(load_files)

# ── UI Layout ──

header_section = widgets.HTML("""
<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; border-left: 5px solid #274C77; margin-bottom: 20px;">
    <h2 style="color: #274C77; margin: 0 0 10px 0;">🎯 Qualitative Coding & Thematic Analysis</h2>
    <p><strong>Welcome!</strong> This notebook helps social scientists systematically apply codes to interview transcripts and qualitative data using AI-powered analysis.</p>

    <h4 style="color: #274C77; margin: 15px 0 10px 0;">📋 How to Use:</h4>
    <ol style="margin: 0; padding-left: 20px;">
        <li><strong>Context:</strong> Enter your research question and study description (optional but recommended)</li>
        <li><strong>Upload:</strong> Add your codebook and chunked transcript CSV files</li>
        <li><strong>Configure:</strong> Choose coding approach and AI model</li>
        <li><strong>Process:</strong> Click Apply Configuration, then run the analysis</li>
    </ol>
    <p style="color: #6096BA; font-size: 13px; margin-top: 10px;">
        <strong>Analytical Lenses:</strong> If your codebook was built with the Codebook Builder using one or more analytical lenses,
        they will be auto-detected from the <code>stance</code> column. Multi-lens codebooks enable epistemic pluralism —
        coding runs once per lens, then cross-lens comparison surfaces where frameworks agree or diverge.
    </p>
</div>
""")

guide_section = widgets.HTML("""
<div style="background-color: #A3CEF1; padding: 20px; border-radius: 10px; border-left: 5px solid #274C77; margin-bottom: 20px;">
    <h4 style="color: #274C77; margin: 0 0 15px 0;">📚 Configuration Guide</h4>

    <div style="display: flex; gap: 20px; flex-wrap: wrap;">
        <div style="flex: 1; min-width: 300px;">
            <strong>• Analysis Modes:</strong>
            <ul style="margin: 5px 0 15px 20px;">
                <li><strong>Single Codebook:</strong> Standard analysis with one codebook — if it contains a <code>stance</code> column with multiple lenses, multi-lens mode activates automatically</li>
                <li><strong>Multi-Stance Codebooks:</strong> Upload codebooks from the Codebook Builder (one per epistemological stance) for parallel multi-stance analysis with cross-stance comparison</li>
            </ul>

            <strong>• Coding Approaches:</strong>
            <ul style="margin: 5px 0 15px 20px;">
                <li><strong>Deductive:</strong> Apply pre-defined codes from your codebook — best when you have established theoretical frameworks</li>
                <li><strong>Inductive:</strong> Generate new codes by discovering emergent themes — ideal for exploratory research</li>
                <li><strong>Hybrid:</strong> Combine both approaches to apply existing codes and discover new themes (recommended)</li>
            </ul>

            <strong>• AI Models:</strong>
            <ul style="margin: 5px 0 15px 20px;">
                <li><strong>Claude Sonnet 5:</strong> Balanced speed and capability (recommended)</li>
                <li><strong>Claude Haiku 4.5:</strong> Fast and affordable, suitable for large datasets</li>
                <li><strong>Claude Opus 4.8:</strong> Most capable model for nuanced qualitative analysis</li>
            </ul>
        </div>

        <div style="flex: 1; min-width: 300px;">
            <strong>• Input File Requirements:</strong>
            <ul style="margin: 5px 0 15px 20px;">
                <li><strong>Codebook CSV:</strong> From Codebook Builder (or manual). Required: <code>code_label</code>, <code>definition</code>. Optional: <code>stance</code> (for lens detection), <code>inclusion_criteria</code>, <code>exclusion_criteria</code>, <code>example_1</code>, <code>example_2</code></li>
                <li><strong>Multi-Stance CSVs:</strong> Multiple codebook CSVs from the Codebook Builder, each containing a stance column or named codebook_<em>stance</em>.csv</li>
                <li><strong>Transcript CSV:</strong> From the Interview Transcript Semantic Chunker notebook with chunk_id and text columns</li>
                <li><strong>Format:</strong> UTF-8 encoded CSV files with proper headers</li>
            </ul>
        </div>
    </div>
</div>
""")

# Research context section
research_context_section = widgets.VBox([
    widgets.HTML("""
    <div style="background-color: #E7ECEF; padding: 15px; border-radius: 10px; border-left: 5px solid #274C77; margin-bottom: 5px;">
        <h4 style="color: #274C77; margin: 0 0 8px 0;">🔬 Research Context <span style="color: #8B8C89; font-weight: normal; font-size: 12px;">(optional — improves analysis quality)</span></h4>
    </div>
    """),
    widgets.HTML("<span style='color: #274C77; font-size: 13px; font-weight: bold;'>Project Name</span>"),
    project_name_input,
    widgets.HTML("<span style='color: #274C77; font-size: 13px; font-weight: bold;'>Research Question</span>"),
    research_question_input,
    widgets.HTML("<span style='color: #274C77; font-size: 13px; font-weight: bold;'>Study Description</span>"),
    study_description_input,
    widgets.HTML("<span style='color: #274C77; font-size: 13px; font-weight: bold; margin-top: 5px;'>Analytical Lens(es)</span>"),
    lens_status_display,
], layout=widgets.Layout(width='100%', margin='0 0 15px 0'))

# File uploads section with mode toggle and actions
file_uploads_section = widgets.VBox([
    widgets.HTML("<h4 style='color: #274C77;'>📁 Analysis Mode & File Uploads</h4>"),
    analysis_mode,
    single_codebook_box,
    multi_codebook_box,
    widgets.HTML("<br>"),
    transcript_upload,
    widgets.HTML("<br>"),
    widgets.HTML("<h4 style='color: #274C77;'>🚀 Actions</h4>"),
    widgets.HBox([process_button, test_button], layout=widgets.Layout(gap='10px'))
])

coding_section = widgets.VBox([
    widgets.HTML("<h4 style='color: #274C77;'>🔍 Coding Approach</h4>"),
    coding_approach
])

ai_section = widgets.VBox([
    widgets.HTML("<h4 style='color: #274C77;'>🧠 AI Configuration</h4>"),
    ai_model_dropdown,
    widgets.HTML("<br>"),
    api_key_input
])

main_config = widgets.HBox([
    file_uploads_section,
    coding_section,
    ai_section
], layout=widgets.Layout(justify_content='space-between', width='75%'))

display(widgets.VBox([
    header_section,
    guide_section,
    research_context_section,
    main_config,
    status_output
]))

## Core Analysis Classes

*Define the core classes for deductive coding, AI integration, and analysis functions that power the coding process.*

In [ ]:
def api_call_with_backoff(client, max_retries=3, **kwargs):
    """Call client.messages.create with retries and exponential backoff.

    Waits 2 ** attempt seconds between attempts and re-raises the final
    exception once max_retries is exhausted.
    """
    for attempt in range(max_retries):
        try:
            return client.messages.create(**kwargs)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait_seconds = 2 ** attempt
            print(f"   ⏳ API call failed ({e}) — retrying in {wait_seconds}s "
                  f"(attempt {attempt + 1}/{max_retries})")
            time.sleep(wait_seconds)


def match_code_to_list(code, valid_codes):
    """Match a returned code against a list of valid codes.

    Accepts exact matches, case-insensitive matches, and truncation recovery
    when the code is a prefix of exactly one valid code and is at least 10
    characters long (or at least 60% of that code's length). Returns the
    matched valid code, or None when no reliable match exists.
    """
    code_clean = code.strip()
    if not code_clean:
        return None

    if code_clean in valid_codes:
        return code_clean

    lower_map = {c.lower(): c for c in valid_codes}
    if code_clean.lower() in lower_map:
        return lower_map[code_clean.lower()]

    prefix_matches = [c for c in valid_codes if c.startswith(code_clean)]
    if len(prefix_matches) == 1:
        full_code = prefix_matches[0]
        if len(code_clean) >= 10 or len(code_clean) >= 0.6 * len(full_code):
            return full_code

    return None


class DeductiveCoder:
    """
    Handles deductive coding using a predefined codebook.
    """

    def __init__(self, codebook_df, stance_key=None):
        self.codebook_df = codebook_df
        self.stance_key = stance_key
        self.code_dict = self._build_code_dict()
        self.coding_history = []

    def _build_code_dict(self):
        """Build a dictionary structure from the codebook."""
        code_dict = {}

        for _, row in self.codebook_df.iterrows():
            code = row['code_label']

            # Build examples list, handling missing columns
            examples = []
            if 'example_1' in row.index and pd.notna(row.get('example_1', '')):
                examples.append(row['example_1'])
            if 'example_2' in row.index and pd.notna(row.get('example_2', '')):
                examples.append(row['example_2'])
            if 'example_3' in row.index and pd.notna(row.get('example_3', '')):
                examples.append(row['example_3'])

            code_dict[code] = {
                'definition': row['definition'],
                'inclusion': row.get('inclusion_criteria', '') if pd.notna(row.get('inclusion_criteria', '')) else '',
                'exclusion': row.get('exclusion_criteria', '') if pd.notna(row.get('exclusion_criteria', '')) else '',
                'examples': examples
            }

        return code_dict

    def display_codebook_summary(self):
        """Display an organized summary of the codebook."""
        print("\n📚 CODEBOOK REFERENCE")
        print("=" * 60)

        for code, details in self.code_dict.items():
            print(f"\n🏷️  {code}")
            print(f"   Definition: {details['definition']}")
            if details['inclusion']:
                print(f"   ✓ Include: {details['inclusion']}")
            if details['exclusion']:
                print(f"   ✗ Exclude: {details['exclusion']}")

    def validate_codes(self, codes_list):
        """Validate that provided codes exist in the codebook.

        Accepts exact matches, case-insensitive matches, and truncation
        recovery when a returned code is an unambiguous prefix of exactly
        one valid code. Anything else is treated as invalid and handled
        by the retry path.
        """
        valid_codes = []
        invalid_codes = []

        all_codes = list(self.code_dict.keys())

        for code in codes_list:
            code_clean = code.strip()
            if not code_clean:
                continue

            matched = match_code_to_list(code_clean, all_codes)
            if matched:
                valid_codes.append(matched)
            else:
                invalid_codes.append(code_clean)

        # Remove duplicates while preserving order
        valid_codes = list(dict.fromkeys(valid_codes))

        return {
            'valid': valid_codes,
            'invalid': invalid_codes,
            'all_valid': len(invalid_codes) == 0
        }


class ClaudeAutoCoder:
    """
    Automated coding using Claude API or alternative methods.
    """

    def __init__(self, api_key: str, codebook_df: pd.DataFrame, coder: DeductiveCoder, use_ai: bool = True, stance_key: str = None):
        self.use_ai = use_ai
        self.codebook_df = codebook_df
        self.coder = coder
        self.stance_key = stance_key
        self.coding_history = []
        self.failed_chunks = []  # chunk ids that failed after retries

        if use_ai:
            self.client = anthropic.Anthropic(api_key=api_key)
            self.coding_prompt = self._build_coding_prompt()
        else:
            self.client = None
            self.coding_prompt = None

    def _build_coding_prompt(self) -> str:
        """Build the system prompt for Claude with the complete codebook and optional lens context."""
        lens_context = build_lens_context(stance_key=self.stance_key)

        codebook_text = "DEDUCTIVE CODING CODEBOOK:\n\n"

        for code, details in self.coder.code_dict.items():
            codebook_text += f"CODE: {code}\n"
            codebook_text += f"Definition: {details['definition']}\n"
            if details['inclusion']:
                codebook_text += f"Include when: {details['inclusion']}\n"
            if details['exclusion']:
                codebook_text += f"Exclude when: {details['exclusion']}\n"
            codebook_text += "\n"

        # Create a simple list of valid code labels for reference
        valid_codes_list = ", ".join(self.coder.code_dict.keys())

        prompt = f"""You are a qualitative research assistant specializing in deductive coding. Your task is to analyze text segments and identify which codes from the codebook apply.

{lens_context}{codebook_text}

CODING INSTRUCTIONS:
1. Read each text segment carefully
2. Apply ALL relevant codes from the codebook that match the content
3. **CRITICAL: You may ONLY use codes from the VALID CODES list below. Do NOT invent, modify, or abbreviate code names.**
4. Return codes as a comma-separated list (e.g., "CODE1,CODE2,CODE3")
5. If no codes apply, return "NO_CODES"
6. Be consistent - similar content should receive similar codes
7. Match codes exactly as written - do not paraphrase or create similar-sounding codes

VALID CODES (use ONLY these exact codes):
{valid_codes_list}

Return only the comma-separated codes from the valid codes list above. No explanations."""

        return prompt

    def _save_checkpoint(self, df, processed_count, total_count, phase):
        """Save checkpoint for fault tolerance as CSV."""
        stance_suffix = f"_{self.stance_key}" if self.stance_key else ""
        # Save data as CSV
        checkpoint_path = f"{config['output_folder']}/deductive_coding_checkpoint{stance_suffix}.csv"
        df.to_csv(checkpoint_path, index=False)

        # Save metadata separately
        meta_path = f"{config['output_folder']}/deductive_coding_checkpoint{stance_suffix}_meta.txt"
        with open(meta_path, 'w') as f:
            f.write(f"timestamp: {datetime.now().isoformat()}\n")
            f.write(f"phase: {phase}\n")
            f.write(f"processed: {processed_count}\n")
            f.write(f"total: {total_count}\n")
            f.write(f"percentage: {round((processed_count / total_count) * 100, 1)}%\n")

    def _finalize_checkpoint(self, df, phase):
        """Save final output and clean up checkpoint files."""
        stance_suffix = f"_{self.stance_key}" if self.stance_key else ""
        checkpoint_path = f"{config['output_folder']}/deductive_coding_checkpoint{stance_suffix}.csv"
        final_path = f"{config['output_folder']}/deductive_coding_final{stance_suffix}.csv"

        # Save final data
        df.to_csv(final_path, index=False)
        print(f"   💾 Final deductive coding saved: {final_path}")

        # Remove checkpoint file (keep final)
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)

        # Update metadata
        meta_path = f"{config['output_folder']}/deductive_coding_checkpoint{stance_suffix}_meta.txt"
        if os.path.exists(meta_path):
            with open(meta_path, 'a') as f:
                f.write(f"status: complete\n")
                f.write(f"completed_at: {datetime.now().isoformat()}\n")

    def _code_batch_parallel(self, df: pd.DataFrame, text_column: str,
                              chunk_id_column: str, max_workers: int,
                              checkpoint_interval: int) -> pd.DataFrame:
        """Code chunks in parallel using ThreadPoolExecutor."""
        from concurrent.futures import ThreadPoolExecutor, as_completed
        import threading

        total_chunks = len(df)
        stance_label = f" [{self.stance_key}]" if self.stance_key else ""
        print(f"⚡ Parallel processing enabled ({max_workers} workers){stance_label}")
        print(f"   Processing {total_chunks} chunks...")
        self.failed_chunks = []

        # Auto-detect chunk_id_column
        if chunk_id_column is None:
            chunk_id_column = config.get('chunk_id_column', 'chunk_id')
            if chunk_id_column not in df.columns:
                df = df.reset_index()
                chunk_id_column = 'index'

        # Initialize columns
        if 'Deductive_Codes' not in df.columns:
            df['Deductive_Codes'] = ''
        if 'Coding_Status' not in df.columns:
            df['Coding_Status'] = ''

        # Thread-safe counter and lock
        completed = [0]
        lock = threading.Lock()

        def code_chunk(args):
            idx, row = args
            text = row[text_column]
            chunk_id = row[chunk_id_column] if chunk_id_column in row else str(idx)
            result = self.code_single_chunk(text, chunk_id)
            return idx, result

        # Load checkpoint and merge already-coded rows; submit only unprocessed chunks
        results = {}
        checkpoint = self._load_checkpoint()
        already_coded = {}
        if checkpoint and str(checkpoint.get('phase', '')).startswith('deductive'):
            for record in checkpoint.get('data', []):
                if str(record.get('Coding_Status', '')) == 'Deductive_Coded':
                    cid = str(record.get(chunk_id_column, ''))
                    codes = record.get('Deductive_Codes', '')
                    already_coded[cid] = str(codes) if pd.notna(codes) else ''

        work_items = []
        for idx, row in df.iterrows():
            cid = str(row[chunk_id_column]) if chunk_id_column in df.columns else str(idx)
            if cid in already_coded:
                df.at[idx, 'Deductive_Codes'] = already_coded[cid]
                df.at[idx, 'Coding_Status'] = 'Deductive_Coded'
                results[idx] = {'codes': already_coded[cid], 'valid': True, 'error': None}
            else:
                work_items.append((idx, row))

        start_idx = total_chunks - len(work_items)
        if start_idx > 0:
            print(f"📂 Resuming from checkpoint: {start_idx}/{total_chunks} already processed")

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(code_chunk, item): item[0] for item in work_items}

            for future in as_completed(futures):
                idx = futures[future]
                try:
                    result_idx, result = future.result()
                    results[result_idx] = result

                    with lock:
                        completed[0] += 1
                        pct = ((start_idx + completed[0]) / total_chunks) * 100
                        status = "✅" if result['valid'] else "⚠️"
                        print(f"\r   Coding chunk {start_idx + completed[0]}/{total_chunks} ({pct:.1f}%) {status}", end="")

                        if result.get('error'):
                            print(f" {result['error']}")

                        # Checkpoint periodically
                        if completed[0] % checkpoint_interval == 0:
                            # Apply results so far
                            for r_idx, r_result in results.items():
                                df.at[r_idx, 'Deductive_Codes'] = r_result['codes']
                                df.at[r_idx, 'Coding_Status'] = 'Deductive_Coded'
                            self._save_checkpoint(df, start_idx + completed[0], total_chunks, 'deductive_parallel')

                except Exception as e:
                    print(f"\n   Error on chunk {idx}: {e}")
                    results[idx] = {'codes': '', 'valid': False, 'error': str(e)}

        print()  # New line after progress

        # Apply all results
        for idx, result in results.items():
            df.at[idx, 'Deductive_Codes'] = result['codes']
            df.at[idx, 'Coding_Status'] = 'Deductive_Coded'

        # Finalize
        self._finalize_checkpoint(df, 'deductive_coding')
        print(f"\n✅ Deductive coding completed (parallel)!")
        print(f"   Processed: {total_chunks} chunks")
        if self.failed_chunks:
            print(f"⚠️ {len(self.failed_chunks)} chunks failed after retries: {self.failed_chunks}")

        return df


    def _load_checkpoint(self):
        """Load checkpoint if it exists (CSV format)."""
        stance_suffix = f"_{self.stance_key}" if self.stance_key else ""
        checkpoint_path = f"{config['output_folder']}/deductive_coding_checkpoint{stance_suffix}.csv"
        meta_path = f"{config['output_folder']}/deductive_coding_checkpoint{stance_suffix}_meta.txt"

        if os.path.exists(checkpoint_path) and os.path.exists(meta_path):
            try:
                # Read metadata
                meta = {}
                with open(meta_path, 'r') as f:
                    for line in f:
                        if ':' in line:
                            key, val = line.strip().split(':', 1)
                            meta[key.strip()] = val.strip()

                # Read data
                data_df = pd.read_csv(checkpoint_path)

                return {
                    'phase': meta.get('phase', ''),
                    'processed': int(meta.get('processed', 0)),
                    'total': int(meta.get('total', 0)),
                    'data': data_df.to_dict(orient='records')
                }
            except Exception as e:
                print(f"Could not load checkpoint: {e}")
                return None
        return None

    def code_single_chunk(self, text: str, chunk_id: str = None) -> Dict:
        """Code a single text chunk."""
        if pd.isna(text) or str(text).strip() == '':
            return {
                'chunk_id': chunk_id,
                'codes': '',
                'valid': True,
                'error': None
            }

        if self.use_ai:
            return self._code_with_ai(text, chunk_id)
        else:
            return self._code_without_ai(text, chunk_id)

    def _code_with_ai(self, text: str, chunk_id: str, retry_count: int = 0) -> Dict:
        """Code using Claude API with retry on invalid codes."""
        max_retries = 1  # One retry attempt for invalid codes

        try:
            messages = [{
                "role": "user",
                "content": f"Code this text: {text}"
            }]

            response = api_call_with_backoff(
                self.client,
                model=config['ai_model'],
                max_tokens=150,
                temperature=0.1,
                system=self.coding_prompt,
                messages=messages
            )

            raw_codes = response.content[0].text.strip()

            if raw_codes.upper() == "NO_CODES" or raw_codes == "":
                return {
                    'chunk_id': chunk_id,
                    'codes': '',
                    'valid': True,
                    'error': None
                }

            # Validate codes
            codes_list = [code.strip() for code in raw_codes.split(',') if code.strip()]
            validation = self.coder.validate_codes(codes_list)

            # If there are invalid codes and we haven't retried yet, try again with feedback
            if validation['invalid'] and retry_count < max_retries:
                valid_codes_str = ", ".join(self.coder.code_dict.keys())
                retry_prompt = f"""The following codes you returned are NOT in the valid codebook: {validation['invalid']}

Please re-analyze the text and ONLY use codes from this list:
{valid_codes_str}

Text to code: {text}

Return ONLY valid codes from the list above, comma-separated. If none apply, return NO_CODES."""

                retry_response = api_call_with_backoff(
                    self.client,
                    model=config['ai_model'],
                    max_tokens=150,
                    temperature=0.0,  # More deterministic on retry
                    system=self.coding_prompt,
                    messages=[{"role": "user", "content": retry_prompt}]
                )

                retry_codes = retry_response.content[0].text.strip()

                if retry_codes.upper() == "NO_CODES" or retry_codes == "":
                    # On retry, if model says no codes, use the valid codes from first attempt
                    return {
                        'chunk_id': chunk_id,
                        'codes': ','.join(validation['valid']),
                        'valid': True,
                        'error': None
                    }

                # Validate retry codes
                retry_codes_list = [code.strip() for code in retry_codes.split(',') if code.strip()]
                retry_validation = self.coder.validate_codes(retry_codes_list)

                # Combine valid codes from both attempts (deduplicated)
                all_valid = list(dict.fromkeys(validation['valid'] + retry_validation['valid']))

                return {
                    'chunk_id': chunk_id,
                    'codes': ','.join(all_valid),
                    'valid': len(retry_validation['invalid']) == 0,
                    'error': f"Invalid codes after retry: {retry_validation['invalid']}" if retry_validation['invalid'] else None
                }

            return {
                'chunk_id': chunk_id,
                'codes': ','.join(validation['valid']),
                'valid': validation['all_valid'],
                'error': f"Invalid codes removed: {validation['invalid']}" if validation['invalid'] else None
            }

        except Exception as e:
            self.failed_chunks.append(chunk_id)
            return {
                'chunk_id': chunk_id,
                'codes': '',
                'valid': False,
                'error': f"API Error: {str(e)}"
            }

    def _code_without_ai(self, text: str, chunk_id: str) -> Dict:
        """Simple keyword matching for non-AI coding."""
        text_lower = text.lower()
        matched_codes = []

        for code, details in self.coder.code_dict.items():
            # Simple keyword matching from definition and examples
            keywords = []
            keywords.extend(details['definition'].lower().split())
            keywords.extend([ex.lower() for ex in details['examples'] if ex])

            # Check if any keyword appears in text
            if any(keyword in text_lower for keyword in keywords if len(keyword) > 3):
                matched_codes.append(code)

        return {
            'chunk_id': chunk_id,
            'codes': ','.join(matched_codes),
            'valid': True,
            'error': None
        }

    def code_batch(self, df: pd.DataFrame, text_column: str = 'text',
                   chunk_id_column: str = None, delay_seconds: float = 0.5,
                   use_parallel: bool = True, max_workers: int = 5) -> pd.DataFrame:
        """Code a batch of text chunks with optional parallel processing."""
        from concurrent.futures import ThreadPoolExecutor, as_completed
        import threading

        total_chunks = len(df)
        checkpoint_interval = 15

        # Use parallel processing for faster coding (if enabled and enough chunks)
        if use_parallel and self.use_ai and total_chunks >= 10:
            return self._code_batch_parallel(df, text_column, chunk_id_column, max_workers, checkpoint_interval)

        # Auto-detect chunk_id_column
        if chunk_id_column is None:
            chunk_id_column = config.get('chunk_id_column', 'chunk_id')

        stance_label = f" [{self.stance_key}]" if self.stance_key else ""
        print(f"\n🤖 Starting Deductive Coding{stance_label}")
        print(f"Processing {total_chunks} text chunks")
        print("-" * 50)

        # Add columns for deductive coding
        if 'Deductive_Codes' not in df.columns:
            df['Deductive_Codes'] = ''
        if 'Coding_Status' not in df.columns:
            df['Coding_Status'] = ''

        # Check for existing checkpoint to resume
        checkpoint = self._load_checkpoint()
        start_idx = 0
        checkpoint_data = []
        if checkpoint and checkpoint.get('phase') == 'deductive_coding':
            start_idx = checkpoint.get('processed', 0)
            checkpoint_data = checkpoint.get('data', [])
            if start_idx > 0 and start_idx < total_chunks:
                print(f"📂 Resuming from checkpoint: {start_idx}/{total_chunks} already processed")

        successful_codes = 0
        failed_codes = 0
        self.failed_chunks = []

        for i, (idx, row) in enumerate(df.iterrows()):
            if i < start_idx:
                # skip chunks already coded before start_idx (checkpoint resume)
                if i < len(checkpoint_data):
                    record = checkpoint_data[i]
                    codes = record.get('Deductive_Codes', '')
                    df.at[idx, 'Deductive_Codes'] = codes if pd.notna(codes) else ''
                    status = record.get('Coding_Status', '')
                    df.at[idx, 'Coding_Status'] = status if pd.notna(status) else ''
                continue

            text = row[text_column]
            chunk_id = row.get(chunk_id_column, f'chunk_{i}') if chunk_id_column in df.columns else f'chunk_{i}'

            print(f"Coding chunk {i+1}/{total_chunks} ({((i+1)/total_chunks*100):.1f}%)", end=" ")

            result = self.code_single_chunk(text, chunk_id)

            # Update DataFrame
            df.at[idx, 'Deductive_Codes'] = result['codes']
            df.at[idx, 'Coding_Status'] = 'Deductive_Coded' if result['codes'] else 'No_Deductive_Codes'

            if result['valid'] and not result['error']:
                successful_codes += 1
                print("✅")
            else:
                failed_codes += 1
                print(f"⚠️  {result['error']}")

            # Save checkpoint periodically
            if (i + 1) % checkpoint_interval == 0:
                self._save_checkpoint(df, i + 1, total_chunks, 'deductive_coding')
                print(f"   💾 Checkpoint saved ({i+1}/{total_chunks})")

            if self.use_ai and i < total_chunks - 1:
                time.sleep(delay_seconds)

        # Finalize checkpoint (keep as persistent output)
        self._finalize_checkpoint(df, 'deductive_coding')

        print(f"\n✅ Deductive coding completed!")
        print(f"Successfully coded: {successful_codes}")
        print(f"Failed/partial: {failed_codes}")
        if self.failed_chunks:
            print(f"⚠️ {len(self.failed_chunks)} chunks failed after retries: {self.failed_chunks}")

        return df

## Inductive Coding Classes

*Define classes for discovering emergent themes and patterns not captured in the deductive codebook.*

In [ ]:
class ClaudeInductiveCoder:
    """
    Use Claude API for inductive coding to discover emergent themes.
    """

    def __init__(self, claude_coder: ClaudeAutoCoder, coded_df: pd.DataFrame, stance_key: str = None):
        self.claude_coder = claude_coder
        self.coded_df = coded_df
        self.client = claude_coder.client
        self.use_ai = claude_coder.use_ai
        self.stance_key = stance_key
        self.discovered_codes = {}

    def generate_inductive_codes(self, sample_size: int = 50) -> Dict:
        """Discover emergent codes from the data."""
        stance_label = f" [{self.stance_key}]" if self.stance_key else ""
        print(f"\n🔍 GENERATING INDUCTIVE CODES{stance_label}")
        print("=" * 40)

        # Sample chunks for analysis
        all_chunks = self.coded_df[self.coded_df['text'].notna()]
        sample_chunks = all_chunks.sample(min(sample_size, len(all_chunks)), random_state=42)

        print(f"📊 Analyzing {len(sample_chunks)} chunks for emergent patterns...")

        if self.use_ai:
            return self._generate_with_ai(sample_chunks)
        else:
            return self._generate_without_ai(sample_chunks)

    def _generate_with_ai(self, sample_chunks: pd.DataFrame) -> Dict:
        """Generate inductive codes using AI."""
        # Prepare chunks with their deductive codes for context
        chunks_text = ""
        for idx, row in sample_chunks.iterrows():
            text = str(row['text'])
            deductive_codes = row.get('Deductive_Codes', '')
            chunks_text += f"\nChunk {row['chunk_id']}:\n{text}\nDeductive codes: {deductive_codes}\n---\n"

        lens_context = build_lens_context(stance_key=self.stance_key)

        inductive_prompt = f"""You are conducting INDUCTIVE CODING on interview transcripts.
{lens_context}Your task is to identify EMERGENT THEMES that are NOT captured by the existing deductive codes.

SAMPLE CHUNKS FOR ANALYSIS:
{chunks_text}

TASK: Identify 8-12 EMERGENT INDUCTIVE CODES that capture important patterns NOT covered by deductive codes.

For each code provide:
**INDUCTIVE CODE: [SHORT_NAME]**
Definition: [Clear description]
Rationale: [Why this is important]
Example: "[Direct quote]"
When to Apply: [Clear criteria]

Ensure there is a blank line between codes."""

        try:
            response = api_call_with_backoff(
                self.client,
                model=config['ai_model'],
                max_tokens=3000,
                temperature=0.4,
                messages=[{
                    "role": "user",
                    "content": inductive_prompt
                }]
            )

            inductive_analysis = response.content[0].text
            self.discovered_codes = self._parse_inductive_codes(inductive_analysis)

            print(f"✅ Found {len(self.discovered_codes)} inductive codes")

            return {
                'inductive_analysis': inductive_analysis,
                'discovered_codes': self.discovered_codes,
                'sample_size': len(sample_chunks)
            }

        except Exception as e:
            print(f"❌ Error in inductive analysis: {e}")
            return {'error': str(e)}

    def _generate_without_ai(self, sample_chunks: pd.DataFrame) -> Dict:
        """Generate inductive codes using frequency analysis."""
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.cluster import KMeans

        # Simple TF-IDF based theme discovery
        texts = sample_chunks['text'].fillna('').tolist()

        try:
            vectorizer = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1, 3))
            tfidf_matrix = vectorizer.fit_transform(texts)

            # Get top terms
            feature_names = vectorizer.get_feature_names_out()

            # Simple clustering
            n_clusters = min(8, len(texts) // 5)
            kmeans = KMeans(n_clusters=n_clusters, random_state=42)
            clusters = kmeans.fit_predict(tfidf_matrix)

            # Generate codes from clusters
            for i in range(n_clusters):
                cluster_terms = []
                cluster_docs = [texts[j] for j in range(len(texts)) if clusters[j] == i]

                if cluster_docs:
                    # Get representative terms
                    cluster_vectorizer = TfidfVectorizer(max_features=5, stop_words='english')
                    cluster_tfidf = cluster_vectorizer.fit_transform(cluster_docs)
                    terms = cluster_vectorizer.get_feature_names_out()

                    code_name = f"THEME_{i+1}_{terms[0].upper()}"
                    self.discovered_codes[code_name] = {
                        'definition': f"Theme related to {', '.join(terms[:3])}",
                        'rationale': "Emerged from clustering analysis",
                        'example': cluster_docs[0][:100] + "...",
                        'application': f"Apply when text mentions {terms[0]}"
                    }

            print(f"✅ Found {len(self.discovered_codes)} inductive themes through clustering")

            return {
                'discovered_codes': self.discovered_codes,
                'sample_size': len(sample_chunks)
            }

        except Exception as e:
            print(f"❌ Error in clustering: {e}")
            # Fallback to simple frequency analysis
            return self._simple_frequency_codes(sample_chunks)

    def _simple_frequency_codes(self, sample_chunks: pd.DataFrame) -> Dict:
        """Fallback method using simple frequency analysis."""
        from collections import Counter
        import re

        # Extract common phrases
        all_text = ' '.join(sample_chunks['text'].fillna('').tolist()).lower()
        words = re.findall(r'\b\w+\b', all_text)

        # Filter common words
        stop_words = {'the', 'is', 'at', 'which', 'on', 'and', 'a', 'an', 'as', 'are', 'was', 'were', 'i', 'you', 'he', 'she', 'it', 'they', 'we'}
        words = [w for w in words if w not in stop_words and len(w) > 3]

        # Get most common words
        word_freq = Counter(words).most_common(20)

        # Create simple codes
        for i, (word, freq) in enumerate(word_freq[:8]):
            code_name = f"FREQ_{word.upper()}"
            self.discovered_codes[code_name] = {
                'definition': f"References to {word} (appeared {freq} times)",
                'rationale': "High frequency term in the data",
                'example': f"Text containing '{word}'",
                'application': f"Apply when text mentions {word}"
            }

        print(f"✅ Found {len(self.discovered_codes)} frequency-based codes")

        return {
            'discovered_codes': self.discovered_codes,
            'sample_size': len(sample_chunks)
        }

    def _parse_inductive_codes(self, analysis_text):
        """Parse inductive codes from Claude's analysis."""
        codes = {}

        if not analysis_text:
            return codes

        # Use regex to find each code block
        code_blocks = re.split(r'\*\*INDUCTIVE CODE: (.+?)\*\*\s*\n', analysis_text, flags=re.DOTALL | re.IGNORECASE)

        if len(code_blocks) < 2:
            return codes

        for i in range(1, len(code_blocks), 2):
            code_name = code_blocks[i].strip()
            block_content = code_blocks[i+1].strip() if i+1 < len(code_blocks) else ''

            codes[code_name] = {
                'definition': '',
                'rationale': '',
                'example': '',
                'application': ''
            }

            # Extract fields
            definition_match = re.search(r'Definition: (.+?)(?:\nRationale:|\nExample:|\nWhen to Apply:|$)', block_content, re.DOTALL)
            if definition_match:
                codes[code_name]['definition'] = definition_match.group(1).strip()

            rationale_match = re.search(r'Rationale: (.+?)(?:\nDefinition:|\nExample:|\nWhen to Apply:|$)', block_content, re.DOTALL)
            if rationale_match:
                codes[code_name]['rationale'] = rationale_match.group(1).strip()

            example_match = re.search(r'Example: (.+?)(?:\nDefinition:|\nRationale:|\nWhen to Apply:|$)', block_content, re.DOTALL)
            if example_match:
                codes[code_name]['example'] = example_match.group(1).strip()

            application_match = re.search(r'When to Apply: (.+?)(?:\nDefinition:|\nRationale:|\nExample:|$)', block_content, re.DOTALL)
            if application_match:
                codes[code_name]['application'] = application_match.group(1).strip()

        return codes

    def _save_inductive_checkpoint(self, processed, total):
        """Save checkpoint for inductive coding as CSV."""
        stance_suffix = f"_{self.stance_key}" if self.stance_key else ""
        # Save data as CSV
        checkpoint_path = f"{config['output_folder']}/inductive_coding_checkpoint{stance_suffix}.csv"
        self.coded_df.to_csv(checkpoint_path, index=False)

        # Save metadata
        meta_path = f"{config['output_folder']}/inductive_coding_checkpoint{stance_suffix}_meta.txt"
        with open(meta_path, 'w') as f:
            f.write(f"timestamp: {datetime.now().isoformat()}\n")
            f.write(f"phase: inductive_coding\n")
            f.write(f"processed: {processed}\n")
            f.write(f"total: {total}\n")
            f.write(f"percentage: {round((processed / total) * 100, 1)}%\n")

    def _delete_inductive_checkpoint(self):
        """Rename inductive checkpoint to indicate completion (keep for reference)."""
        stance_suffix = f"_{self.stance_key}" if self.stance_key else ""
        checkpoint_path = f"{config['output_folder']}/inductive_coding_checkpoint{stance_suffix}.csv"
        if os.path.exists(checkpoint_path):
            final_path = f"{config['output_folder']}/inductive_coding_final{stance_suffix}.csv"
            import shutil
            shutil.copy(checkpoint_path, final_path)
            # Keep the checkpoint file - don't delete

    def apply_inductive_codes(self, delay_seconds: float = 1.0) -> pd.DataFrame:
        """Apply discovered inductive codes to all chunks."""
        stance_label = f" [{self.stance_key}]" if self.stance_key else ""
        print(f"\n📝 APPLYING INDUCTIVE CODES{stance_label}")
        print("-" * 50)

        if not self.discovered_codes:
            print("❌ No inductive codes to apply.")
            return self.coded_df

        # Add inductive coding columns
        if 'Inductive_Codes' not in self.coded_df.columns:
            self.coded_df['Inductive_Codes'] = ''

        total_chunks = len(self.coded_df)
        checkpoint_interval = 25
        successful = 0

        if self.use_ai:
            # Create prompt for applying codes
            codes_text = "\n".join([
                f"{code}: {details['definition']} (Apply when: {details['application']})"
                for code, details in self.discovered_codes.items()
            ])

            apply_prompt = f"""Apply these INDUCTIVE CODES to text chunks.

INDUCTIVE CODES:
{codes_text}

Instructions:
1. Apply ONLY codes that clearly match
2. Return codes as comma-separated list
3. If no codes apply, return "NONE"

Return ONLY the code names."""

            failed_chunks = []
            discarded_count = 0
            discovered_names = list(self.discovered_codes.keys())

            for i, (idx, row) in enumerate(self.coded_df.iterrows()):
                if pd.notna(row['text']):
                    text = str(row['text'])

                    print(f"Applying inductive codes {i+1}/{total_chunks} ({(i+1)/total_chunks*100:.1f}%)", end=" ")

                    try:
                        response = api_call_with_backoff(
                            self.client,
                            model=config['ai_model'],
                            max_tokens=100,
                            temperature=0.1,
                            system=apply_prompt,
                            messages=[{
                                "role": "user",
                                "content": f"Text: {text}"
                            }]
                        )

                        result = response.content[0].text.strip()

                        if result and result.upper() != "NONE":
                            # Validate applied codes against the discovered inductive code set
                            valid_applied = []
                            for c in result.split(','):
                                matched = match_code_to_list(c.strip(), discovered_names)
                                if matched:
                                    valid_applied.append(matched)
                                elif c.strip():
                                    discarded_count += 1
                            valid_applied = list(dict.fromkeys(valid_applied))
                            if valid_applied:
                                self.coded_df.at[idx, 'Inductive_Codes'] = ','.join(valid_applied)
                                successful += 1
                                print("✅")

                        # Save checkpoint periodically
                        if (i + 1) % checkpoint_interval == 0:
                            self._save_inductive_checkpoint(i + 1, total_chunks)
                            print(f"   💾 Inductive checkpoint saved ({i+1}/{total_chunks})")

                        time.sleep(delay_seconds)

                    except Exception as e:
                        failed_chunks.append(row.get(config.get('chunk_id_column', 'chunk_id'), idx))
                        print(f"❌ Error: {e}")
                        continue

            if discarded_count:
                print(f"\n⚠️ Discarded {discarded_count} applied codes not found in the discovered inductive code set")
            if failed_chunks:
                print(f"⚠️ {len(failed_chunks)} chunks failed after retries: {failed_chunks}")
        else:
            # Apply codes using keyword matching
            for idx, row in self.coded_df.iterrows():
                if pd.notna(row['text']):
                    text = str(row['text']).lower()
                    matched_codes = []

                    for code, details in self.discovered_codes.items():
                        # Simple keyword matching
                        keywords = details['definition'].lower().split()
                        if any(keyword in text for keyword in keywords if len(keyword) > 3):
                            matched_codes.append(code)

                    if matched_codes:
                        self.coded_df.at[idx, 'Inductive_Codes'] = ','.join(matched_codes)
                        successful += 1

        print(f"\n✅ Applied inductive codes to {successful} chunks")

        # Update coding status
        for idx, row in self.coded_df.iterrows():
            has_deductive = pd.notna(row.get('Deductive_Codes', '')) and row.get('Deductive_Codes', '') != ''
            has_inductive = pd.notna(row.get('Inductive_Codes', '')) and row.get('Inductive_Codes', '') != ''

            if has_deductive and has_inductive:
                self.coded_df.at[idx, 'Coding_Status'] = 'Both_Deductive_Inductive'
            elif has_deductive:
                self.coded_df.at[idx, 'Coding_Status'] = 'Deductive_Only'
            elif has_inductive:
                self.coded_df.at[idx, 'Coding_Status'] = 'Inductive_Only'
            else:
                self.coded_df.at[idx, 'Coding_Status'] = 'No_Codes'

        return self.coded_df

## Analysis and Theme Building Functions

*Functions to analyze code patterns, build themes, and generate insights from the coded data.*

In [ ]:
def integrate_coding_approaches(df_coded, stance_key=None):
    """
    Create integrated view of all codes for theme building.
    """
    stance_label = f" [{stance_key}]" if stance_key else ""
    print(f"\n🔗 INTEGRATING CODING APPROACHES{stance_label}")
    print("-" * 50)

    # Add integration columns
    df_coded['All_Codes'] = ''
    df_coded['Total_Code_Count'] = 0

    for idx, row in df_coded.iterrows():
        all_codes = []

        # Add deductive codes
        if pd.notna(row.get('Deductive_Codes', '')) and row.get('Deductive_Codes', ''):
            deductive_codes = [c.strip() for c in str(row['Deductive_Codes']).split(',')]
            all_codes.extend(deductive_codes)

        # Add inductive codes (marked with _IND suffix)
        if pd.notna(row.get('Inductive_Codes', '')) and row.get('Inductive_Codes', ''):
            inductive_codes = [c.strip() + '_IND' for c in str(row['Inductive_Codes']).split(',')]
            all_codes.extend(inductive_codes)

        # Update integrated columns
        if all_codes:
            df_coded.at[idx, 'All_Codes'] = ', '.join(all_codes)
            df_coded.at[idx, 'Total_Code_Count'] = len(all_codes)

    # Analysis of integration
    integration_stats = {
        'total_chunks': len(df_coded),
        'coded_chunks': len(df_coded[df_coded['Total_Code_Count'] > 0]),
        'deductive_only': len(df_coded[df_coded['Coding_Status'] == 'Deductive_Only']),
        'inductive_only': len(df_coded[df_coded['Coding_Status'] == 'Inductive_Only']),
        'both_types': len(df_coded[df_coded['Coding_Status'] == 'Both_Deductive_Inductive']),
        'no_codes': len(df_coded[df_coded['Coding_Status'] == 'No_Codes'])
    }

    print("\n✅ Integration Complete:")
    print(f"• Total chunks: {integration_stats['total_chunks']}")
    print(f"• Coded chunks: {integration_stats['coded_chunks']}")
    print(f"• Deductive only: {integration_stats['deductive_only']}")
    print(f"• Inductive only: {integration_stats['inductive_only']}")
    print(f"• Both types: {integration_stats['both_types']}")
    print(f"• No codes: {integration_stats['no_codes']}")

    return df_coded, integration_stats


def analyze_code_patterns(df_coded, stance_key=None):
    """
    Analyze patterns in the integrated codes.
    """
    stance_label = f" [{stance_key}]" if stance_key else ""
    print(f"\n📊 ANALYZING CODE PATTERNS{stance_label}")
    print("-" * 40)

    # Collect all codes
    all_codes_list = []
    deductive_codes_list = []
    inductive_codes_list = []
    code_combinations = defaultdict(int)

    for idx, row in df_coded.iterrows():
        if pd.notna(row['All_Codes']) and row['All_Codes']:
            codes = [c.strip() for c in str(row['All_Codes']).split(',')]
            all_codes_list.extend(codes)

            # Separate deductive and inductive
            deductive = [c for c in codes if not c.endswith('_IND')]
            inductive = [c for c in codes if c.endswith('_IND')]

            deductive_codes_list.extend(deductive)
            inductive_codes_list.extend(inductive)

            # Track combinations
            if len(codes) > 1:
                code_combinations[tuple(sorted(codes))] += 1

    # Calculate frequencies
    all_freq = Counter(all_codes_list)
    deductive_freq = Counter(deductive_codes_list)
    inductive_freq = Counter(inductive_codes_list)

    patterns = {
        'total_code_applications': len(all_codes_list),
        'unique_codes': len(set(all_codes_list)),
        'all_codes_frequency': dict(all_freq),
        'deductive_frequency': dict(deductive_freq.most_common(15)),
        'inductive_frequency': dict(inductive_freq.most_common(15)),
        'frequent_combinations': dict(sorted(code_combinations.items(), key=lambda x: x[1], reverse=True)[:15])
    }

    print(f"\n📈 Pattern Analysis Results:")
    print(f"• Total code applications: {patterns['total_code_applications']}")
    print(f"• Unique codes used: {patterns['unique_codes']}")

    print(f"\n🏆 Top 10 Most Frequent Codes:")
    for code, freq in list(patterns['all_codes_frequency'].items())[:10]:
        code_type = "[IND]" if code.endswith('_IND') else "[DED]"
        print(f"  • {code} {code_type}: {freq} occurrences")

    return patterns



def merge_multi_lens_results(stance_results, transcript_df):
    """Merge per-stance coding results into a unified DataFrame.

    Creates per-lens code columns (e.g. Codes_CriticalRace, Codes_Feminist),
    a union All_Codes column, and per-chunk agreement scores.
    """
    df_merged = transcript_df.copy()

    lens_names = list(stance_results.keys())
    lens_code_cols = {}

    for lens_name in lens_names:
        col_safe = re.sub(r'[^a-zA-Z0-9]', '', lens_name)
        col_name = f"Codes_{col_safe}"
        lens_code_cols[lens_name] = col_name

        lens_df = stance_results[lens_name]['df_coded']
        # Agreement compares all applied codes (deductive + inductive) across lenses.
        combined_codes = []
        for _, lens_row in lens_df.iterrows():
            codes = []
            for source_col in ('Deductive_Codes', 'Inductive_Codes'):
                raw = lens_row.get(source_col, '')
                if pd.notna(raw) and str(raw).strip():
                    codes.extend(c.strip() for c in str(raw).split(',') if c.strip())
            combined_codes.append(', '.join(dict.fromkeys(codes)))
        df_merged[col_name] = combined_codes

    # Build union All_Codes and per-chunk agreement
    all_codes_list = []
    agreement_scores = []
    lens_counts = []

    for idx in df_merged.index:
        per_lens_codes = {}
        for lens_name in lens_names:
            col = lens_code_cols[lens_name]
            raw = df_merged.at[idx, col] if col in df_merged.columns else ''
            if pd.notna(raw) and str(raw).strip():
                codes = set(c.strip() for c in str(raw).split(',') if c.strip())
            else:
                codes = set()
            per_lens_codes[lens_name] = codes

        union_codes = set()
        for codes in per_lens_codes.values():
            union_codes |= codes

        all_codes_list.append(', '.join(sorted(union_codes)) if union_codes else '')

        active_lenses = [name for name, codes in per_lens_codes.items() if codes]
        lens_counts.append(len(active_lenses))

        if len(active_lenses) >= 2:
            jaccard_scores = []
            for i in range(len(lens_names)):
                for j in range(i + 1, len(lens_names)):
                    a = per_lens_codes[lens_names[i]]
                    b = per_lens_codes[lens_names[j]]
                    union = a | b
                    if union:
                        jaccard = len(a & b) / len(union)
                    else:
                        jaccard = 1.0
                    jaccard_scores.append(jaccard)
            agreement_scores.append(np.mean(jaccard_scores))
        else:
            agreement_scores.append(float('nan'))

    df_merged['All_Codes'] = all_codes_list
    df_merged['Total_Code_Count'] = df_merged['All_Codes'].apply(
        lambda x: len([c for c in str(x).split(',') if c.strip()]) if x else 0
    )
    df_merged['Deductive_Codes'] = df_merged['All_Codes']
    df_merged['Coding_Status'] = df_merged['All_Codes'].apply(
        lambda x: 'Deductive_Coded' if x else 'No_Codes'
    )
    df_merged['Lens_Count'] = lens_counts
    df_merged['Agreement_Score'] = agreement_scores

    valid_agreements = [s for s in agreement_scores if not np.isnan(s)]
    mean_agreement = np.mean(valid_agreements) if valid_agreements else float('nan')

    print(f"\n🔗 MULTI-LENS MERGE COMPLETE")
    print(f"• Lenses merged: {', '.join(lens_names)}")
    print(f"• Chunks with codes from ≥2 lenses: {sum(1 for lc in lens_counts if lc >= 2)}")
    if valid_agreements:
        print(f"• Mean cross-lens agreement (Jaccard): {mean_agreement:.3f}")
        low_agreement = sum(1 for s in valid_agreements if s < 0.3)
        print(f"• High-friction chunks (agreement < 0.3): {low_agreement}")

    return df_merged


class CrossLensAnalyzer:
    """Analyze agreement and friction across multiple analytical lenses."""

    def __init__(self, df_coded, lens_names):
        self.df_coded = df_coded
        self.lens_names = lens_names
        self.lens_code_cols = {}
        for name in lens_names:
            col_safe = re.sub(r'[^a-zA-Z0-9]', '', name)
            self.lens_code_cols[name] = f"Codes_{col_safe}"

    def _get_lens_codes(self, idx, lens_name):
        """Get the set of codes for a given chunk and lens."""
        # Agreement compares all applied codes (deductive + inductive) across lenses.
        col = self.lens_code_cols.get(lens_name, '')
        if col not in self.df_coded.columns:
            return set()
        raw = self.df_coded.at[idx, col]
        if pd.notna(raw) and str(raw).strip():
            return set(c.strip() for c in str(raw).split(',') if c.strip())
        return set()

    def compute_lens_overlap_matrix(self):
        """Compute N x N matrix of mean pairwise Jaccard between lenses across all chunks."""
        n = len(self.lens_names)
        matrix = np.zeros((n, n))

        for i in range(n):
            for j in range(n):
                if i == j:
                    matrix[i][j] = 1.0
                    continue
                jaccard_vals = []
                for idx in self.df_coded.index:
                    a = self._get_lens_codes(idx, self.lens_names[i])
                    b = self._get_lens_codes(idx, self.lens_names[j])
                    union = a | b
                    if union:
                        jaccard_vals.append(len(a & b) / len(union))
                if jaccard_vals:
                    matrix[i][j] = np.mean(jaccard_vals)
                else:
                    matrix[i][j] = float('nan')

        return pd.DataFrame(matrix, index=self.lens_names, columns=self.lens_names)

    def identify_friction_points(self, threshold=0.3, top_n=20):
        """Identify chunks where lenses diverge most (agreement < threshold)."""
        if 'Agreement_Score' not in self.df_coded.columns:
            return pd.DataFrame()

        friction = self.df_coded[self.df_coded['Agreement_Score'] < threshold].copy()
        friction = friction.sort_values('Agreement_Score', ascending=True).head(top_n)
        return friction

    def compute_lens_code_profiles(self):
        """For each lens, compute top codes and total applications."""
        profiles = {}
        for lens_name in self.lens_names:
            col = self.lens_code_cols.get(lens_name, '')
            if col not in self.df_coded.columns:
                continue

            all_codes = []
            for val in self.df_coded[col].dropna():
                codes = [c.strip() for c in str(val).split(',') if c.strip()]
                all_codes.extend(codes)

            freq = Counter(all_codes)
            profiles[lens_name] = {
                'total_applications': len(all_codes),
                'unique_codes': len(set(all_codes)),
                'top_codes': dict(freq.most_common(10))
            }

        return profiles

    def generate_friction_report(self, top_n=10):
        """Generate a rich HTML friction report for notebook display."""
        friction = self.identify_friction_points(threshold=0.3, top_n=top_n)

        if friction.empty:
            display(widgets.HTML(
                "<div style='padding: 15px; background: #E7ECEF; border-radius: 8px;'>"
                "<h4 style='color: #274C77;'>✅ No high-friction chunks detected</h4>"
                "<p>All chunks show agreement score ≥ 0.3 across lenses.</p></div>"
            ))
            return friction

        chunk_id_col = config.get('chunk_id_column', 'chunk_id')

        html = "<div style='max-height: 600px; overflow-y: auto;'>"
        html += "<h3 style='color: #274C77;'>🔥 High-Friction Chunks (lenses diverge most)</h3>"
        html += "<p style='color: #6096BA;'>These chunks are analytically interesting — different lenses see different things.</p>"

        for idx, row in friction.iterrows():
            chunk_id = row.get(chunk_id_col, idx)
            text = str(row.get('text', ''))[:300]
            agreement = row.get('Agreement_Score', 0)

            html += f"<div style='border: 1px solid #A3CEF1; border-radius: 8px; padding: 12px; margin: 8px 0; background: white;'>"
            html += f"<strong style='color: #274C77;'>Chunk {chunk_id}</strong> "
            html += f"<span style='color: {'#d32f2f' if agreement < 0.1 else '#f57c00'}; font-weight: bold;'>"
            html += f"Agreement: {agreement:.2f}</span><br>"
            html += f"<p style='font-size: 12px; color: #555; margin: 5px 0;'>{text}...</p>"

            for lens_name in self.lens_names:
                codes = self._get_lens_codes(idx, lens_name)
                codes_str = ', '.join(sorted(codes)) if codes else '<em>no codes</em>'
                html += f"<div style='margin: 2px 0; font-size: 12px;'>"
                html += f"<strong style='color: #6096BA;'>{lens_name}:</strong> {codes_str}</div>"

            html += "</div>"

        html += "</div>"
        display(widgets.HTML(html))
        return friction

    def generate_summary(self):
        """Generate a compact cross-lens summary for prompts and exports."""
        profiles = self.compute_lens_code_profiles()
        overlap = self.compute_lens_overlap_matrix()

        lines = [f"CROSS-LENS COMPARISON ({len(self.lens_names)} lenses):"]

        for name in self.lens_names:
            p = profiles.get(name, {})
            top = list(p.get('top_codes', {}).keys())[:5]
            lines.append(f"  {name}: {p.get('total_applications', 0)} applications, "
                         f"top codes: {', '.join(top)}")

        if 'Agreement_Score' in self.df_coded.columns:
            scores = self.df_coded['Agreement_Score'].dropna()
            if len(scores) > 0:
                lines.append(f"\n  Mean agreement: {scores.mean():.3f}")
                lines.append(f"  High-friction chunks (< 0.3): {(scores < 0.3).sum()}")

        return '\n'.join(lines)

class IntegratedThemeBuilder:
    """
    Build themes from integrated deductive and inductive codes.
    """

    def __init__(self, claude_coder, coded_df, code_patterns, inductive_results,
                 cross_lens_analyzer=None):
        self.claude_coder = claude_coder
        self.coded_df = coded_df
        self.client = claude_coder.client if claude_coder else None
        self.use_ai = claude_coder.use_ai if claude_coder else False
        self.code_patterns = code_patterns
        self.inductive_results = inductive_results
        self.cross_lens_analyzer = cross_lens_analyzer

    def build_themes(self):
        """Build hierarchical themes from the integrated analysis."""
        print("\n🎯 BUILDING THEMES FROM INTEGRATED CODES")
        print("=" * 50)

        if self.use_ai:
            return self._build_themes_with_ai()
        else:
            return self._build_themes_without_ai()

    def _build_themes_with_ai(self):
        """Build themes using AI with optional lens-awareness."""
        pattern_examples = self._get_pattern_examples()

        lens_context = build_lens_context()

        cross_lens_section = ""
        if self.cross_lens_analyzer and config.get('multi_stance_mode'):
            cross_lens_section = f"""
CROSS-LENS ANALYSIS:
{self.cross_lens_analyzer.generate_summary()}

When building themes, note:
- Where themes are supported across multiple lenses (CONVERGENT) vs. primarily one lens (LENS-SPECIFIC)
- Build at least one theme around high-friction chunks where lenses diverge — these reveal what each framework foregrounds or obscures
"""

        theme_prompt = f"""You are a qualitative research expert building THEMES from mixed-method coding results.

{lens_context}CODING OVERVIEW:
- Total code applications: {self.code_patterns['total_code_applications']}
- Unique codes: {self.code_patterns['unique_codes']}

TOP DEDUCTIVE CODES:
{self._format_top_codes(self.code_patterns['deductive_frequency'], 10)}

TOP INDUCTIVE CODES:
{self._format_top_codes(self.code_patterns['inductive_frequency'], 10)}

{cross_lens_section}SAMPLE CODED CHUNKS:
{pattern_examples}

TASK: Create 5-7 HIERARCHICAL THEMES that:
1. Integrate insights from both deductive and inductive codes
2. Have clear main themes with 2-3 sub-themes each
3. Are actionable and relevant
{"4. Tag each theme as CONVERGENT (supported by multiple lenses) or LENS-SPECIFIC (primarily one lens) or FRICTION (lenses disagree on this theme)" if config.get('multi_stance_mode') else ""}

Format each theme as:

THEME [Number]: [Clear, Descriptive Name]
Core Concept: [2-3 sentences explaining what this theme captures]
Sub-themes:
  a) [Sub-theme name]: [Brief description]
  b) [Sub-theme name]: [Brief description]
  c) [Sub-theme name]: [Brief description]
Key Finding: [The main insight this theme reveals, including which codes support it]
Supporting Codes: [List the exact code names that support this theme, e.g., VALUE_BASED_CARE_TRANSFOR, OUTCOME_BASED_SUBSIDY_MODELS_IND]
Evidence Strength: [Strong/Moderate/Emerging] ([Total code count] combined code applications: [CODE_NAME]=[count], [CODE_NAME]=[count], ...)
{"Lens Convergence: [CONVERGENT: supported by X, Y lenses | LENS-SPECIFIC: primarily from X lens | FRICTION: lenses disagree on this theme]" if config.get('multi_stance_mode') else ""}

CRITICAL: The "Supporting Codes" line MUST list the actual code names from the TOP CODES lists above that relate to this theme. Use the exact uppercase code names with underscores.
"""

        try:
            response = api_call_with_backoff(
                self.client,
                model=config['ai_model'],
                max_tokens=4000,
                temperature=0.3,
                messages=[{
                    "role": "user",
                    "content": theme_prompt
                }]
            )

            themes_analysis = response.content[0].text
            print("✅ Themes successfully built!")

            return {
                'themes_analysis': themes_analysis,
                'code_patterns': self.code_patterns,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            }

        except Exception as e:
            print(f"❌ Error building themes: {e}")
            return {'error': str(e)}

    def _build_themes_without_ai(self):
        """Build themes using frequency analysis."""
        themes_text = "THEMES GENERATED FROM FREQUENCY ANALYSIS\n\n"

        # Group codes by frequency
        top_codes = list(self.code_patterns['all_codes_frequency'].items())[:15]

        # Create simple themes based on most frequent codes
        theme_num = 1
        for i in range(0, len(top_codes), 3):
            theme_codes = top_codes[i:i+3]
            if theme_codes:
                main_code = theme_codes[0][0]
                themes_text += f"\nTHEME {theme_num}: {main_code.replace('_', ' ').title()}\n"
                themes_text += f"Core Concept: This theme encompasses patterns related to {main_code} "
                themes_text += f"which appeared {theme_codes[0][1]} times in the data.\n"
                themes_text += "Sub-themes:\n"

                for j, (code, freq) in enumerate(theme_codes[1:], 1):
                    themes_text += f"  {chr(96+j)}) {code}: Frequency: {freq}\n"

                themes_text += f"Key Finding: High frequency of {main_code} suggests its importance.\n"
                themes_text += "Evidence Strength: Strong (based on frequency)\n"
                theme_num += 1

        return {
            'themes_analysis': themes_text,
            'code_patterns': self.code_patterns,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }

    def _get_pattern_examples(self):
        """Get example chunks for major patterns."""
        examples = []

        # Get examples of chunks with both deductive and inductive codes
        mixed_chunks = self.coded_df[self.coded_df['Coding_Status'] == 'Both_Deductive_Inductive'].head(3)

        for idx, row in mixed_chunks.iterrows():
            text_preview = str(row['text'])[:200] + "..."
            ded_codes = row.get('Deductive_Codes', 'None')
            ind_codes = row.get('Inductive_Codes', 'None')
            examples.append(f"\nChunk {row['chunk_id']}:\nText: {text_preview}\nDeductive: {ded_codes}\nInductive: {ind_codes}")

        if config.get('multi_stance_mode') and 'Agreement_Score' in self.coded_df.columns:
            friction = self.coded_df.nsmallest(2, 'Agreement_Score')
            for idx, row in friction.iterrows():
                text_preview = str(row['text'])[:200] + "..."
                agreement_val = row.get('Agreement_Score', float('nan'))
                agreement_str = f"{agreement_val:.3f}" if not np.isnan(agreement_val) else "N/A"
                examples.append(f"\nFriction Chunk {row.get(config.get('chunk_id_column', 'chunk_id'), idx)}:\nText: {text_preview}\nAll codes: {row.get('All_Codes', '')}\nAgreement: {agreement_str}")

        return '\n'.join(examples) if examples else "No examples available"

    def _format_top_codes(self, freq_dict, limit):
        """Format top codes for the prompt."""
        lines = []
        for code, freq in list(freq_dict.items())[:limit]:
            lines.append(f"• {code}: {freq} occurrences")
        return '\n'.join(lines)

## Visualization Functions

*Create visualizations including network graphs, word clouds, and statistical charts to illuminate patterns in the coded data.*

In [ ]:
def create_analysis_visualizations(df_coded, code_patterns, themes, coder,
                                   cross_lens_analyzer=None):
    """
    Create all visualizations for the analysis.
    """
    print("\n📊 CREATING VISUALIZATIONS")
    print("-" * 40)

    # Set color scheme
    colors = {
        'primary': '#274C77',
        'secondary': '#6096BA',
        'accent': '#A3CEF1',
        'neutral': '#8B8C89',
        'background': '#E7ECEF'
    }

    # 1. Coding Method Distribution
    plt.figure(figsize=(10, 6))

    # Calculate actual distribution of coding methods
    has_deductive = df_coded['Deductive_Codes'].notna() & (df_coded['Deductive_Codes'] != '')
    has_inductive = df_coded['Inductive_Codes'].notna() & (df_coded['Inductive_Codes'] != '') if 'Inductive_Codes' in df_coded.columns else pd.Series([False] * len(df_coded))

    both_count = (has_deductive & has_inductive).sum()
    deductive_only = (has_deductive & ~has_inductive).sum()
    inductive_only = (~has_deductive & has_inductive).sum()
    neither = (~has_deductive & ~has_inductive).sum()

    # Build distribution dict with non-zero values only
    total_chunks = len(df_coded)
    distribution = {}
    if both_count > 0:
        distribution[f'Both Deductive & Inductive ({both_count})'] = both_count
    if deductive_only > 0:
        distribution[f'Deductive Only ({deductive_only})'] = deductive_only
    if inductive_only > 0:
        distribution[f'Inductive Only ({inductive_only})'] = inductive_only
    if neither > 0:
        distribution[f'Uncoded ({neither})'] = neither

    if distribution:
        pie_colors = [colors['primary'], colors['secondary'], colors['accent'], colors['neutral']][:len(distribution)]
        plt.pie(distribution.values(), labels=distribution.keys(), autopct='%1.1f%%',
                colors=pie_colors, startangle=90, explode=[0.02] * len(distribution))
        plt.title(f'Distribution of Coding Methods (n={total_chunks} chunks)', fontsize=16, fontweight='bold', color=colors['primary'])
    else:
        plt.text(0.5, 0.5, 'No coded data', ha='center', va='center', fontsize=14)
        plt.title(f'Distribution of Coding Methods (n={total_chunks} chunks)', fontsize=16, fontweight='bold', color=colors['primary'])

    plt.tight_layout()
    plt.savefig(f"{config['output_folder']}/coding_distribution.png", dpi=300, bbox_inches='tight')
    plt.show()

    # 2. Code Frequency Comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

    # Deductive codes
    ded_codes = list(code_patterns['deductive_frequency'].items())[:15]
    if ded_codes:
        codes, freqs = zip(*ded_codes)
        ax1.barh(codes, freqs, color=colors['primary'])
        ax1.set_xlabel('Frequency', fontsize=12)
        ax1.set_title('Top 15 Deductive Codes', fontsize=14, fontweight='bold', color=colors['primary'])
        ax1.invert_yaxis()

    # Inductive codes
    ind_codes = list(code_patterns['inductive_frequency'].items())[:15]
    if ind_codes:
        codes, freqs = zip(*ind_codes)
        codes = [c.replace('_IND', '') for c in codes]
        ax2.barh(codes, freqs, color=colors['secondary'])
        ax2.set_xlabel('Frequency', fontsize=12)
        ax2.set_title('Top 15 Inductive Codes', fontsize=14, fontweight='bold', color=colors['primary'])
        ax2.invert_yaxis()

    plt.tight_layout()
    plt.savefig(f"{config['output_folder']}/code_frequencies.png", dpi=300, bbox_inches='tight')
    plt.show()

    # 3. Code Co-occurrence Heatmap
    create_code_cooccurrence_heatmap(df_coded, code_patterns, colors)

    # 4. Network Graph (if needed)
    if len(code_patterns['all_codes_frequency']) > 5:
        create_code_network_graph(df_coded, colors)

    # Cross-lens visualizations (if multi-stance mode is active)
    if cross_lens_analyzer and config.get('multi_stance_mode'):
        create_cross_lens_visualizations(cross_lens_analyzer, colors)

    print("\n✅ All visualizations created and saved!")


def create_code_cooccurrence_heatmap(df_coded, code_patterns, colors):
    """
    Create heatmap of code co-occurrences.
    """
    # Get top 20 codes overall
    top_codes = list(code_patterns['all_codes_frequency'].keys())[:20]

    # Build co-occurrence matrix
    matrix = pd.DataFrame(0, index=top_codes, columns=top_codes)

    for idx, row in df_coded.iterrows():
        if pd.notna(row['All_Codes']) and row['All_Codes']:
            codes = [c.strip() for c in str(row['All_Codes']).split(',')]
            codes = [c for c in codes if c in top_codes]

            for i, code1 in enumerate(codes):
                for code2 in codes[i+1:]:  # exclude self-pairs so the diagonal is not counted
                    matrix.loc[code1, code2] += 1
                    if code1 != code2:
                        matrix.loc[code2, code1] += 1

    # Create heatmap
    plt.figure(figsize=(14, 12))

    # Custom colormap
    cmap = sns.light_palette(colors['primary'], as_cmap=True)

    sns.heatmap(matrix, cmap=cmap, annot=True, fmt='d',
                cbar_kws={'label': 'Co-occurrence Count'},
                linewidths=0.5)

    plt.title('Code Co-occurrence Heatmap', fontsize=16, fontweight='bold',
              color=colors['primary'], pad=20)
    plt.tight_layout()
    plt.savefig(f"{config['output_folder']}/code_cooccurrence.png", dpi=300, bbox_inches='tight')
    plt.show()


def create_code_network_graph(df_coded, colors, min_cooccurrence=3):
    """
    Create network graph of code relationships.
    """
    G = nx.Graph()

    # Build network
    for idx, row in df_coded.iterrows():
        if pd.notna(row['All_Codes']) and row['All_Codes']:
            codes = [c.strip() for c in str(row['All_Codes']).split(',')]

            # Add nodes with type
            for code in codes:
                if code not in G:
                    node_type = 'inductive' if code.endswith('_IND') else 'deductive'
                    G.add_node(code, type=node_type)

            # Add edges
            for i in range(len(codes)):
                for j in range(i+1, len(codes)):
                    if G.has_edge(codes[i], codes[j]):
                        G[codes[i]][codes[j]]['weight'] += 1
                    else:
                        G.add_edge(codes[i], codes[j], weight=1)

    # Filter edges
    edges_to_remove = [(u, v) for u, v, d in G.edges(data=True)
                       if d['weight'] < min_cooccurrence]
    G.remove_edges_from(edges_to_remove)
    G.remove_nodes_from(list(nx.isolates(G)))

    if len(G.nodes()) > 0:
        # Create visualization
        plt.figure(figsize=(16, 12))
        pos = nx.spring_layout(G, k=3, iterations=50, seed=42)

        # Color nodes by type
        node_colors = [colors['primary'] if G.nodes[node]['type'] == 'deductive' else colors['secondary']
                       for node in G.nodes()]

        # Size nodes by degree
        node_sizes = [G.degree(node) * 100 for node in G.nodes()]

        # Draw network
        nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.7)

        # Draw edges with width based on weight
        edges = G.edges()
        weights = [G[u][v]['weight'] for u, v in edges]
        nx.draw_networkx_edges(G, pos, width=weights, alpha=0.3)

        # Draw labels
        nx.draw_networkx_labels(G, pos, font_size=8)

        plt.title(f'Code Co-occurrence Network (min {min_cooccurrence} co-occurrences)',
                  fontsize=16, fontweight='bold', color=colors['primary'])
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(f"{config['output_folder']}/code_network.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("✅ Code network visualization saved")

        # Export network as GEXF for Gephi
        try:
            gexf_filename = f"{config['output_folder']}/code_cooccurrence_network.gexf"
            nx.write_gexf(G, gexf_filename)
            print(f"✅ GEXF network file saved: {gexf_filename}")

            # Also export as edge list CSV for other tools
            edge_csv = f"{config['output_folder']}/code_cooccurrence_edges.csv"
            with open(edge_csv, 'w') as f:
                f.write("Source,Target,Weight\n")
                for (u, v, d) in G.edges(data=True):
                    f.write(f"{u},{v},{d.get('weight', 1)}\n")
            print(f"✅ Edge list CSV saved: {edge_csv}")
        except Exception as e:
            print(f"⚠️ Could not export network files: {e}")


def create_cross_lens_visualizations(cross_lens_analyzer, colors):
    """Create visualizations for cross-lens comparison."""
    print("\n📊 Creating cross-lens visualizations...")

    # Lens Agreement Heatmap
    overlap_matrix = cross_lens_analyzer.compute_lens_overlap_matrix()

    if not overlap_matrix.empty:
        plt.figure(figsize=(10, 8))
        cmap = sns.light_palette(colors['primary'], as_cmap=True)
        sns.heatmap(overlap_matrix, annot=True, fmt='.2f', cmap=cmap,
                    vmin=0, vmax=1, square=True,
                    cbar_kws={'label': 'Mean Jaccard Similarity'},
                    linewidths=0.5)
        plt.title('Cross-Lens Agreement Matrix', fontsize=16, fontweight='bold',
                  color=colors['primary'], pad=20)
        plt.tight_layout()
        plt.savefig(f"{config['output_folder']}/lens_agreement_heatmap.png", dpi=300, bbox_inches='tight')
        plt.show()
        print("  ✅ Lens agreement heatmap saved")

    # Friction Distribution Histogram
    df = cross_lens_analyzer.df_coded
    if 'Agreement_Score' in df.columns:
        scores = df['Agreement_Score'].dropna()
        if len(scores) > 0:
            plt.figure(figsize=(10, 6))
            plt.hist(scores, bins=20, color=colors['secondary'], edgecolor=colors['primary'], alpha=0.8)
            plt.axvline(x=0.3, color='#d32f2f', linestyle='--', linewidth=2, label='Friction threshold (0.3)')
            plt.xlabel('Agreement Score (Jaccard)', fontsize=12)
            plt.ylabel('Number of Chunks', fontsize=12)
            plt.title('Distribution of Cross-Lens Agreement', fontsize=16, fontweight='bold',
                      color=colors['primary'])
            plt.legend()

            mean_val = scores.mean()
            friction_pct = (scores < 0.3).mean() * 100
            plt.text(0.02, 0.95, f'Mean: {mean_val:.3f}\nFriction chunks: {friction_pct:.1f}%',
                     transform=plt.gca().transAxes, fontsize=11, verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor=colors['background'], alpha=0.8))

            plt.tight_layout()
            plt.savefig(f"{config['output_folder']}/friction_distribution.png", dpi=300, bbox_inches='tight')
            plt.show()
            print("  ✅ Friction distribution saved")

    # Lens Code Profiles (grouped bar chart)
    profiles = cross_lens_analyzer.compute_lens_code_profiles()
    if profiles:
        all_top_codes = set()
        for prof in profiles.values():
            all_top_codes.update(list(prof['top_codes'].keys())[:5])

        if all_top_codes:
            top_codes_sorted = sorted(all_top_codes)[:15]
            lens_names = list(profiles.keys())
            x = np.arange(len(top_codes_sorted))
            width = 0.8 / len(lens_names)

            fig, ax = plt.subplots(figsize=(14, 8))
            lens_colors = [colors['primary'], colors['secondary'], colors['accent'],
                          colors['neutral'], '#E76F51', '#2A9D8F']

            for i, lens_name in enumerate(lens_names):
                freqs = [profiles[lens_name]['top_codes'].get(code, 0) for code in top_codes_sorted]
                ax.bar(x + i * width, freqs, width, label=lens_name,
                       color=lens_colors[i % len(lens_colors)], alpha=0.85)

            ax.set_xticks(x + width * (len(lens_names) - 1) / 2)
            ax.set_xticklabels(top_codes_sorted, rotation=45, ha='right', fontsize=9)
            ax.set_ylabel('Frequency', fontsize=12)
            ax.set_title('Code Application by Analytical Lens', fontsize=16, fontweight='bold',
                        color=colors['primary'])
            ax.legend(title='Analytical Lens')
            plt.tight_layout()
            plt.savefig(f"{config['output_folder']}/lens_code_profiles.png", dpi=300, bbox_inches='tight')
            plt.show()
            print("  ✅ Lens code profiles saved")

## Export Functions

*Export analysis results to Excel and create formatted Word documents with themes.*

In [ ]:
def export_complete_analysis(df_coded, themes, inductive_results, code_patterns,
                           integration_stats, coder, output_subfolder=None,
                           cross_lens_analyzer=None):
    """
    Export all analysis results to Excel and Word.
    Includes lens metadata and cross-lens comparison when multi-lens mode is active.
    """
    print("\n💾 EXPORTING ANALYSIS RESULTS")
    print("-" * 40)

    timestamp = config['timestamp']
    base_folder = config['output_folder']
    if output_subfolder:
        base_folder = os.path.join(base_folder, output_subfolder)
        os.makedirs(base_folder, exist_ok=True)
    excel_filename = f"{base_folder}/coding_analysis_{timestamp}.xlsx"
    word_filename = f"{base_folder}/themes_report_{timestamp}.docx"

    # Export to Excel
    try:
        with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
            # 1. Coded Data
            df_coded.to_excel(writer, sheet_name='Coded_Data', index=False)

            # 2. Coding Summary
            summary_data = {
                'Metric': ['Total Chunks', 'Deductive Only', 'Inductive Only',
                          'Both Types', 'No Codes', 'Total Coded'],
                'Count': [integration_stats['total_chunks'],
                         integration_stats['deductive_only'],
                         integration_stats['inductive_only'],
                         integration_stats['both_types'],
                         integration_stats['no_codes'],
                         integration_stats['coded_chunks']]
            }
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_excel(writer, sheet_name='Coding_Summary', index=False)

            # 3. Deductive Codebook
            if coder and hasattr(coder, 'codebook_df'):
                coder.codebook_df.to_excel(writer, sheet_name='Deductive_Codebook', index=False)

            # 4. Inductive Codes
            if inductive_results and 'discovered_codes' in inductive_results:
                inductive_codes_data = []
                for code, details in inductive_results['discovered_codes'].items():
                    inductive_codes_data.append({
                        'Code': code,
                        'Definition': details.get('definition', ''),
                        'Rationale': details.get('rationale', ''),
                        'Application': details.get('application', ''),
                        'Example': details.get('example', '')
                    })
                if inductive_codes_data:
                    inductive_codes_df = pd.DataFrame(inductive_codes_data)
                    inductive_codes_df.to_excel(writer, sheet_name='Inductive_Codes', index=False)

            # 5. Code Frequencies
            freq_data = []
            for code, freq in code_patterns['all_codes_frequency'].items():
                code_type = 'Inductive' if code.endswith('_IND') else 'Deductive'
                freq_data.append({
                    'Code': code,
                    'Type': code_type,
                    'Frequency': freq
                })
            freq_df = pd.DataFrame(freq_data)
            freq_df.to_excel(writer, sheet_name='Code_Frequencies', index=False)

            # 6. Code Combinations
            if 'frequent_combinations' in code_patterns:
                combo_data = []
                for combo, freq in list(code_patterns['frequent_combinations'].items())[:20]:
                    combo_data.append({
                        'Combination': ' + '.join(combo),
                        'Frequency': freq
                    })
                if combo_data:
                    combo_df = pd.DataFrame(combo_data)
                    combo_df.to_excel(writer, sheet_name='Code_Combinations', index=False)

            # 7. Analysis Metadata
            meta_rows = [
                {'Field': 'Project Name', 'Value': config.get('project_name', '')},
                {'Field': 'Research Question', 'Value': config.get('research_question', '')},
                {'Field': 'Study Description', 'Value': config.get('study_description', '')},
                {'Field': 'Coding Approach', 'Value': config.get('coding_approach', '')},
                {'Field': 'AI Model', 'Value': config.get('ai_model', '')},
                {'Field': 'Use AI', 'Value': str(config.get('use_ai', True))},
                {'Field': 'Timestamp', 'Value': config.get('timestamp', '')},
                {'Field': 'Multi-Stance Mode', 'Value': str(config.get('multi_stance_mode', False))},
            ]
            detected = config.get('detected_lenses', [])
            if detected:
                meta_rows.append({'Field': 'Detected Lenses', 'Value': ', '.join(detected)})
                meta_rows.append({'Field': 'Number of Lenses', 'Value': str(len(detected))})
            active_stances = config.get('active_stances', [])
            if active_stances:
                meta_rows.append({'Field': 'Active Stances', 'Value': ', '.join(active_stances)})
            meta_df = pd.DataFrame(meta_rows)
            meta_df.to_excel(writer, sheet_name='Analysis_Metadata', index=False)

            # 8. Lens Comparison (multi-lens mode only)
            if cross_lens_analyzer and config.get('multi_stance_mode'):
                _export_lens_comparison_sheet(writer, cross_lens_analyzer)

        # Apply formatting to Excel file
        from openpyxl import load_workbook
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter

        wb = load_workbook(excel_filename)

        # Define styles
        header_font = Font(bold=True, color='FFFFFF')
        header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
        header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell_alignment = Alignment(vertical='top', wrap_text=True)
        thin_border = Border(
            left=Side(style='thin'),
            right=Side(style='thin'),
            top=Side(style='thin'),
            bottom=Side(style='thin')
        )

        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]

            # Format headers (first row)
            for col_idx, cell in enumerate(ws[1], 1):
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = header_alignment
                cell.border = thin_border

            # Auto-adjust column widths and format data cells
            for col_idx in range(1, ws.max_column + 1):
                col_letter = get_column_letter(col_idx)
                max_length = 0

                for row_idx in range(1, ws.max_row + 1):
                    cell = ws.cell(row=row_idx, column=col_idx)
                    if row_idx > 1:
                        cell.alignment = cell_alignment
                        cell.border = thin_border

                    try:
                        cell_length = len(str(cell.value)) if cell.value else 0
                        max_length = max(max_length, min(cell_length, 50))
                    except:
                        pass

                # Set column width (min 10, max 60)
                adjusted_width = max(10, min(max_length + 2, 60))
                ws.column_dimensions[col_letter].width = adjusted_width

            # Freeze header row
            ws.freeze_panes = 'A2'

            # Set data-row heights for readability
            for row_idx in range(2, ws.max_row + 1):
                ws.row_dimensions[row_idx].height = 60

        wb.save(excel_filename)
        print(f"✅ Excel file saved with formatting: {excel_filename}")

    except Exception as e:
        print(f"❌ Error exporting Excel: {e}")
        import traceback
        traceback.print_exc()

    # Export themes to Word
    try:
        create_themes_document(themes, word_filename, cross_lens_analyzer=cross_lens_analyzer,
                               integration_stats=integration_stats)
        print(f"✅ Word document saved: {word_filename}")

    except Exception as e:
        print(f"❌ Error creating Word document: {e}")

    print("\n📁 Files created in folder:")
    print(f"   {config['output_folder']}/")
    print("   ├── coding_analysis_[timestamp].xlsx")
    print("   ├── themes_report_[timestamp].docx")
    print("   ├── coding_distribution.png")
    print("   ├── code_frequencies.png")
    print("   ├── code_cooccurrence.png")
    if os.path.exists(f"{config['output_folder']}/code_network.png"):
        print("   ├── code_network.png")
    if config.get('multi_stance_mode'):
        print("   ├── lens_agreement_heatmap.png")
        print("   ├── friction_distribution.png")
        print("   └── lens_code_profiles.png")


def _add_md_para(doc, text, base_style='Normal'):
    """Convert markdown text to Word paragraph."""
    import re

    if not text or not text.strip():
        return

    text = text.strip()

    # Handle headings
    if text.startswith('## '):
        doc.add_heading(text[3:].strip(), level=2)
        return
    elif text.startswith('# '):
        doc.add_heading(text[2:].strip(), level=1)
        return
    elif text.startswith('### '):
        doc.add_heading(text[4:].strip(), level=3)
        return

    # Handle bullets
    if text.startswith('- ') or text.startswith('* '):
        text = text[2:]
        p = doc.add_paragraph(style='List Bullet')
    elif re.match(r'^\d+\.\s', text):
        text = re.sub(r'^\d+\.\s', '', text)
        p = doc.add_paragraph(style='List Number')
    elif text.startswith('a) ') or text.startswith('b) ') or text.startswith('c) '):
        p = doc.add_paragraph(style='List Bullet')
    else:
        p = doc.add_paragraph()

    # Process bold markers
    parts = re.split(r'(\*\*[^*]+\*\*)', text)
    for part in parts:
        if part.startswith('**') and part.endswith('**'):
            run = p.add_run(part[2:-2])
            run.bold = True
        else:
            p.add_run(part)


def _add_md_content(doc, content):
    """Add multi-line markdown content to document."""
    if not content:
        return
    for line in content.split('\n'):
        if line.strip():
            _add_md_para(doc, line)


def create_themes_document(themes, filename, cross_lens_analyzer=None, integration_stats=None):
    """
    Create a formatted Word document with themes.
    """
    doc = Document()

    # Add title
    title = doc.add_heading('Qualitative Analysis Themes Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # Add subtitle/description
    doc.add_paragraph()
    desc = doc.add_paragraph()
    desc.add_run('Initial Coding & Thematic Analysis Results').italic = True
    desc.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph()
    doc.add_paragraph()

    # Add metadata section
    meta_heading = doc.add_paragraph()
    meta_heading.add_run('Analysis Details').bold = True

    doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    doc.add_paragraph(f"Analysis Type: {config['coding_approach'].title()}")
    doc.add_paragraph(f"AI Model: {config['ai_model'] if config['use_ai'] else 'Non-AI (keyword matching)'}")

    # Research context
    if config.get('project_name'):
        doc.add_paragraph(f"Project: {config['project_name']}")
    if config.get('research_question'):
        doc.add_paragraph(f"Research Question: {config['research_question']}")

    # Lens info
    detected = config.get('detected_lenses', [])
    if detected:
        doc.add_paragraph(f"Analytical Lens(es): {', '.join(detected)}")
        if config.get('multi_stance_mode'):
            doc.add_paragraph("Mode: Multi-Lens Epistemic Pluralism")

    # Summary stats passed in from the analysis
    if integration_stats:
        doc.add_paragraph(f"Chunks Analyzed: {integration_stats.get('total_chunks', 'N/A')}")
        doc.add_paragraph(f"Chunks Coded: {integration_stats.get('coded_chunks', 'N/A')}")

    doc.add_paragraph()

    # Add description of what this report contains
    about = doc.add_paragraph()
    about.add_run('About This Report').bold = True
    doc.add_paragraph(
        "This report presents the results of qualitative coding and thematic analysis. "
        "It includes themes identified through mixed-method analysis combining deductive codes "
        "(from a predefined codebook) and inductive codes (emergent patterns discovered in the data). "
        "Each theme includes a core concept, sub-themes, key findings, and supporting evidence."
    )

    # Cross-lens section in themes doc (if applicable)
    if cross_lens_analyzer and config.get('multi_stance_mode'):
        doc.add_page_break()
        doc.add_heading('Cross-Lens Comparison', 1)
        doc.add_paragraph(
            "This analysis applied multiple analytical lenses to the same data. "
            "The following summarizes where lenses converge and diverge."
        )

        summary = cross_lens_analyzer.generate_summary()
        if summary:
            for line in summary.split('\n'):
                line = line.strip()
                if line:
                    _add_md_para(doc, line)

        friction = cross_lens_analyzer.identify_friction_points(top_n=5)
        if not friction.empty:
            doc.add_heading('Top Friction Points', 2)
            doc.add_paragraph(
                "These data segments show the highest disagreement across analytical lenses, "
                "indicating areas where different frameworks foreground different aspects."
            )
            chunk_id_col = config.get('chunk_id_column', 'chunk_id')
            for idx, row in friction.iterrows():
                chunk_id = row.get(chunk_id_col, idx)
                agreement = row.get('Agreement_Score', 0)
                p = doc.add_paragraph()
                p.add_run(f"Chunk {chunk_id} ").bold = True
                p.add_run(f"(Agreement: {agreement:.3f})")
                text = str(row.get('text', ''))
                preview = text[:300] + ('...' if len(text) > 300 else '')
                text_p = doc.add_paragraph()
                text_p.add_run(preview).italic = True
                for lens_name in cross_lens_analyzer.lens_names:
                    codes = cross_lens_analyzer._get_lens_codes(idx, lens_name)
                    codes_str = ', '.join(sorted(codes)) if codes else 'no codes'
                    lp = doc.add_paragraph(style='List Bullet')
                    lp.add_run(f"{lens_name}: ").bold = True
                    lp.add_run(codes_str)
                doc.add_paragraph()

    doc.add_page_break()

    # Add themes
    doc.add_heading('Discovered Themes', 1)

    if themes and 'themes_analysis' in themes:
        themes_text = themes['themes_analysis']

        # Parse and format themes
        lines = themes_text.split('\n')

        import re

        def add_formatted_text(p, text):
            """Add text with bold and italic markers converted to actual formatting."""
            if not text:
                return
            # Process bold first (**text**)
            parts = re.split(r'(\*\*[^*]+\*\*)', text)
            for part in parts:
                if part.startswith('**') and part.endswith('**'):
                    p.add_run(part[2:-2]).bold = True
                else:
                    # Process italic (*text*) - but not ** which is already handled
                    italic_parts = re.split(r'(?<!\*)\*([^*]+)\*(?!\*)', part)
                    for i, ipart in enumerate(italic_parts):
                        if i % 2 == 1:  # Odd indices are the captured italic text
                            run = p.add_run(ipart)
                            run.italic = True
                        elif ipart:
                            p.add_run(ipart)

        for line in lines:
            line = line.strip()
            if not line or line == '---':
                continue

            # Handle # SECTION headings (like # HIERARCHICAL THEMES)
            if line.startswith('# ') and not line.startswith('## '):
                clean_title = line[2:].replace('**', '').strip()
                # Skip if it's just a label/category header
                if clean_title:
                    doc.add_heading(clean_title, 1)
                continue

            # Handle ## THEME X: format
            if line.startswith('## THEME') or line.startswith('THEME'):
                clean_title = line.replace('## ', '').replace('**', '')
                doc.add_heading(clean_title, 2)

            elif line.startswith('**Core Concept:**') or line.startswith('Core Concept:'):
                p = doc.add_paragraph()
                p.add_run('Core Concept: ').bold = True
                text = line.replace('**Core Concept:**', '').replace('Core Concept:', '').strip()
                add_formatted_text(p, text)

            elif line.startswith('**Sub-themes:**') or line.startswith('Sub-themes:'):
                p = doc.add_paragraph()
                p.add_run('Sub-themes:').bold = True

            elif line.strip().startswith(('a)', 'b)', 'c)', 'd)', 'e)')):
                p = doc.add_paragraph(style='List Bullet')
                add_formatted_text(p, line)

            elif line.startswith('**Key Finding:**') or line.startswith('Key Finding:'):
                p = doc.add_paragraph()
                p.add_run('Key Finding: ').bold = True
                text = line.replace('**Key Finding:**', '').replace('Key Finding:', '').strip()
                add_formatted_text(p, text)

            elif line.startswith('**Evidence Strength:**') or line.startswith('Evidence Strength:'):
                p = doc.add_paragraph()
                p.add_run('Evidence Strength: ').bold = True
                text = line.replace('**Evidence Strength:**', '').replace('Evidence Strength:', '').strip()
                add_formatted_text(p, text)
                doc.add_paragraph()  # Add spacing

            elif line.startswith('## '):
                # Other markdown headings
                doc.add_heading(line[3:].replace('**', ''), 2)

            elif line.startswith('- ') or line.startswith('* '):
                p = doc.add_paragraph(style='List Bullet')
                add_formatted_text(p, line[2:])

            elif line and not line.isspace():
                # Other content - add with markdown formatting
                p = doc.add_paragraph()
                add_formatted_text(p, line)

    else:
        doc.add_paragraph("No themes were generated in this analysis.")

    # Save document
    doc.save(filename)


def _export_lens_comparison_sheet(writer, cross_lens_analyzer):
    """Write Lens_Comparison sheet with agreement matrix and friction summary."""
    # Agreement matrix
    overlap_matrix = cross_lens_analyzer.compute_lens_overlap_matrix()
    if not overlap_matrix.empty:
        overlap_matrix.to_excel(writer, sheet_name='Lens_Comparison', startrow=0)

    # Friction summary below the matrix
    friction = cross_lens_analyzer.identify_friction_points(top_n=20)
    if not friction.empty:
        chunk_id_col = config.get('chunk_id_column', 'chunk_id')
        friction_data = []
        for idx, row in friction.iterrows():
            entry = {
                'Chunk_ID': row.get(chunk_id_col, idx),
                'Agreement_Score': round(row.get('Agreement_Score', 0), 3),
                'Text_Preview': str(row.get('text', ''))[:200],
            }
            for lens_name in cross_lens_analyzer.lens_names:
                codes = cross_lens_analyzer._get_lens_codes(idx, lens_name)
                entry[f'Codes_{lens_name}'] = ', '.join(sorted(codes)) if codes else ''
            friction_data.append(entry)

        friction_df = pd.DataFrame(friction_data)
        start_row = (len(overlap_matrix) + 4) if not overlap_matrix.empty else 0
        friction_df.to_excel(writer, sheet_name='Lens_Comparison', startrow=start_row, index=False)

    # Lens profiles
    profiles = cross_lens_analyzer.compute_lens_code_profiles()
    if profiles:
        profile_rows = []
        for lens_name, prof in profiles.items():
            profile_rows.append({
                'Lens': lens_name,
                'Total_Applications': prof.get('total_applications', 0),
                'Unique_Codes': prof.get('unique_codes', 0),
                'Top_5_Codes': ', '.join(list(prof.get('top_codes', {}).keys())[:5])
            })
        profile_df = pd.DataFrame(profile_rows)

        friction_count = len(friction) if not friction.empty else 0
        start_row_profiles = start_row + friction_count + 4 if not friction.empty else (len(overlap_matrix) + 4 if not overlap_matrix.empty else 0)
        profile_df.to_excel(writer, sheet_name='Lens_Comparison', startrow=start_row_profiles, index=False)

## Main Analysis Execution

*Run the complete analysis workflow based on the configured settings. This cell orchestrates all the analysis steps from coding through theme building and export.*

In [ ]:
def run_single_stance_analysis(codebook_df, transcript_df, stance_key=None):
    """
    Run the complete analysis workflow for a single codebook/stance.
    Returns a results dict or None on failure.
    """
    stance_label = f" [{stance_key}]" if stance_key else ""
    output_subfolder = f"stance_{stance_key}" if stance_key else None

    print(f"\n🚀 STARTING QUALITATIVE CODING ANALYSIS{stance_label}")
    print("=" * 60)

    try:
        # Initialize the deductive coder
        coder = DeductiveCoder(codebook_df, stance_key=stance_key)
        coder.display_codebook_summary()

        # Initialize results variables
        df_coded = transcript_df.copy()
        inductive_results = None

        # Determine coding approach
        approach = config['coding_approach'].lower()

        # Step 1: Deductive Coding (if not inductive only)
        if 'inductive only' not in approach:
            print(f"\n📝 PHASE 1: DEDUCTIVE CODING{stance_label}")
            claude_coder = ClaudeAutoCoder(
                api_key=config['api_key'],
                codebook_df=codebook_df,
                coder=coder,
                use_ai=config['use_ai'],
                stance_key=stance_key
            )
            df_coded = claude_coder.code_batch(df_coded)
        else:
            claude_coder = None

        # Step 2: Inductive Coding (if not deductive only)
        if 'deductive only' not in approach:
            print(f"\n📝 PHASE 2: INDUCTIVE CODING{stance_label}")
            if claude_coder is None:
                claude_coder = ClaudeAutoCoder(
                    api_key=config['api_key'],
                    codebook_df=codebook_df,
                    coder=coder,
                    use_ai=config['use_ai'],
                    stance_key=stance_key
                )

            inductive_coder = ClaudeInductiveCoder(claude_coder, df_coded, stance_key=stance_key)
            inductive_results = inductive_coder.generate_inductive_codes(sample_size=60)

            if 'discovered_codes' in inductive_results:
                print(f"\n📋 Discovered Inductive Codes{stance_label}:")
                for code, details in inductive_results['discovered_codes'].items():
                    print(f"\n• {code}")
                    print(f"  Definition: {details['definition']}")
                    print(f"  When to apply: {details['application']}")

            df_coded = inductive_coder.apply_inductive_codes()

        # Step 3: Integrate coding approaches
        print(f"\n📝 PHASE 3: INTEGRATION & ANALYSIS{stance_label}")
        df_coded, integration_stats = integrate_coding_approaches(df_coded, stance_key=stance_key)

        # Step 4: Analyze patterns
        code_patterns = analyze_code_patterns(df_coded, stance_key=stance_key)

        # Step 5: Build themes
        print(f"\n📝 PHASE 4: THEME BUILDING{stance_label}")
        theme_builder = IntegratedThemeBuilder(
            claude_coder=claude_coder,
            coded_df=df_coded,
            code_patterns=code_patterns,
            inductive_results=inductive_results,
            cross_lens_analyzer=None
        )
        themes = theme_builder.build_themes()

        if 'themes_analysis' in themes:
            print(f"\n📋 GENERATED THEMES{stance_label}:")
            print("=" * 60)
            print(themes['themes_analysis'])

        # Step 6: Create visualizations
        print(f"\n📝 PHASE 5: VISUALIZATIONS{stance_label}")
        create_analysis_visualizations(df_coded, code_patterns, themes, coder, cross_lens_analyzer=None)

        # Step 7: Export results
        print(f"\n📝 PHASE 6: EXPORT{stance_label}")
        export_complete_analysis(
            df_coded=df_coded,
            themes=themes,
            inductive_results=inductive_results,
            code_patterns=code_patterns,
            integration_stats=integration_stats,
            coder=coder,
            output_subfolder=output_subfolder,
            cross_lens_analyzer=None
        )

        # Summary
        print(f"\n{'=' * 60}")
        print(f"📊 ANALYSIS COMPLETE{stance_label}")
        print(f"{'=' * 60}")
        print(f"\n✅ Analysis Type: {config['coding_approach']}")
        print(f"✅ Total chunks analyzed: {integration_stats['total_chunks']}")
        print(f"✅ Successfully coded: {integration_stats['coded_chunks']}")
        print(f"✅ Unique codes used: {code_patterns['unique_codes']}")

        return {
            'df_coded': df_coded,
            'themes': themes,
            'code_patterns': code_patterns,
            'integration_stats': integration_stats,
            'inductive_results': inductive_results,
            'stance_key': stance_key
        }

    except Exception as e:
        print(f"\n❌ Error in analysis workflow{stance_label}: {e}")
        import traceback
        traceback.print_exc()
        return None


def run_complete_analysis():
    """
    Run the complete analysis workflow.
    Dispatches to single-stance or multi-stance mode based on config.
    """
    # Check if files are loaded
    if config['transcript_df'] is None:
        print("❌ Please load files using the configuration interface first!")
        return None

    if not config['multi_stance_mode']:
        # ── Single-codebook mode (original behavior) ──
        if config['codebook_df'] is None:
            print("❌ Please load a codebook using the configuration interface first!")
            return None

        results = run_single_stance_analysis(
            codebook_df=config['codebook_df'],
            transcript_df=config['transcript_df']
        )
        if results:
            print(f"\n📁 All results saved to: {config['output_folder']}/")
        return results

    else:
        # ── Multi-stance mode ──
        if not config['codebook_stances']:
            print("❌ Please load codebook files using the configuration interface first!")
            return None

        stances = config['active_stances']
        print(f"\n🔀 MULTI-STANCE ANALYSIS: {len(stances)} stances")
        print(f"   Stances: {', '.join(stances)}")
        print("=" * 60)

        # Stances run sequentially: console output stays readable and API
        # concurrency is bounded by the 5-worker chunk pool within each stance.
        stance_results = {}

        for sk in stances:
            try:
                result = run_single_stance_analysis(
                    codebook_df=config['codebook_stances'][sk],
                    transcript_df=config['transcript_df'],
                    stance_key=sk
                )
                if result:
                    stance_results[sk] = result
                    print(f"\n✅ Stance '{sk}' completed successfully")
                else:
                    print(f"\n⚠️ Stance '{sk}' returned no results")
            except Exception as e:
                print(f"\n❌ Stance '{sk}' failed: {e}")

        config['stance_results'] = stance_results

        # Print summary table
        print("\n" + "=" * 60)
        print("📊 MULTI-STANCE ANALYSIS SUMMARY")
        print("=" * 60)
        print(f"\n{'Stance':<25} {'Codes Applied':<15} {'Chunks Coded':<15} {'Unique Codes':<15}")
        print("-" * 70)
        for sk, res in stance_results.items():
            stats = res['integration_stats']
            patterns = res['code_patterns']
            print(f"{sk:<25} {patterns['total_code_applications']:<15} {stats['coded_chunks']:<15} {patterns['unique_codes']:<15}")

        print(f"\n📁 All results saved to: {config['output_folder']}/")
        for sk in stance_results:
            print(f"   └── stance_{sk}/")

        # Return multi-stance results
        return {
            'multi_stance': True,
            'stance_results': stance_results,
            'stances': stances
        }


# Ready message
print("\n✅ Analysis function ready!")
print("📌 To run the analysis:")
print("1. Upload your files using the configuration interface above")
print("2. Configure your analysis settings")
print("3. Click 'Apply Configuration'")
print("4. Then run: results = run_complete_analysis()")

## Cross-Stance Comparison

*When multiple epistemological stances are applied to the same data, this section compares how each stance coded the data, identifies areas of convergence and divergence, and synthesizes cross-stance insights. This operationalizes the epistemic pluralism central to Multi-Agent Ethnography (MAE).*

In [ ]:
def build_convergence_matrix(stance_results):
    """
    Build a matrix showing which codes each stance applied to each chunk.
    Returns a DataFrame: rows = chunk_ids, columns = stance_key codes, values = 1/0.
    """
    chunk_id_col = config.get('chunk_id_column', 'chunk_id')
    all_chunks = set()
    stance_code_maps = {}

    for sk, res in stance_results.items():
        df = res['df_coded']
        code_map = {}
        for _, row in df.iterrows():
            cid = str(row.get(chunk_id_col, ''))
            all_chunks.add(cid)
            codes = set()
            # Agreement compares all applied codes (deductive + inductive) across lenses.
            if pd.notna(row.get('Deductive_Codes', '')) and row['Deductive_Codes']:
                codes.update(c.strip() for c in str(row['Deductive_Codes']).split(','))
            if pd.notna(row.get('Inductive_Codes', '')) and row['Inductive_Codes']:
                codes.update(c.strip() for c in str(row['Inductive_Codes']).split(','))
            code_map[cid] = codes
        stance_code_maps[sk] = code_map

    return stance_code_maps, sorted(all_chunks)


def calculate_agreement_scores(stance_results):
    """
    Calculate pairwise Jaccard similarity between stances based on code overlap per chunk.
    Returns a DataFrame: stances x stances with similarity scores.
    """
    stance_code_maps, all_chunks = build_convergence_matrix(stance_results)
    stances = list(stance_code_maps.keys())

    scores = {}
    for i, s1 in enumerate(stances):
        for j, s2 in enumerate(stances):
            if i >= j:
                continue
            jaccard_sum = 0
            valid_chunks = 0
            for cid in all_chunks:
                codes1 = stance_code_maps[s1].get(cid, set())
                codes2 = stance_code_maps[s2].get(cid, set())
                union = codes1 | codes2
                if union:
                    jaccard_sum += len(codes1 & codes2) / len(union)
                    valid_chunks += 1
            avg_jaccard = jaccard_sum / valid_chunks if valid_chunks > 0 else 0
            scores[(s1, s2)] = round(avg_jaccard, 3)
            scores[(s2, s1)] = round(avg_jaccard, 3)

    # Build matrix DataFrame
    matrix = pd.DataFrame(index=stances, columns=stances, dtype=float)
    for s in stances:
        matrix.loc[s, s] = 1.0
    for (s1, s2), score in scores.items():
        matrix.loc[s1, s2] = score

    return matrix


def identify_consensus_and_divergent(stance_results):
    """
    Identify codes found by all stances (consensus) vs. unique to one stance (divergent).
    """
    stance_codes = {}
    for sk, res in stance_results.items():
        codes = set()
        freq = res['code_patterns']['all_codes_frequency']
        codes.update(freq.keys())
        stance_codes[sk] = codes

    all_stances = list(stance_codes.keys())

    # Consensus: codes appearing in ALL stances
    consensus = set.intersection(*stance_codes.values()) if stance_codes else set()

    # Divergent: codes unique to exactly one stance
    divergent = {}
    for sk in all_stances:
        others = set.union(*(stance_codes[s] for s in all_stances if s != sk)) if len(all_stances) > 1 else set()
        unique = stance_codes[sk] - others
        if unique:
            divergent[sk] = unique

    # Shared by some (but not all)
    all_codes = set.union(*stance_codes.values()) if stance_codes else set()
    partial = all_codes - consensus
    for sk, unique_set in divergent.items():
        partial -= unique_set

    return {
        'consensus': consensus,
        'divergent': divergent,
        'partial_overlap': partial,
        'stance_codes': stance_codes
    }


def generate_cross_stance_synthesis(stance_results, api_key):
    """
    Use Claude to synthesize patterns across stances.
    """
    client = anthropic.Anthropic(api_key=api_key)

    # Build summary for each stance
    stance_summaries = []
    for sk, res in stance_results.items():
        stats = res['integration_stats']
        patterns = res['code_patterns']
        top_codes = list(patterns['all_codes_frequency'].keys())[:10]
        themes_text = res['themes'].get('themes_analysis', '')[:500]

        stance_summaries.append(
            f"STANCE: {sk}\n"
            f"Coded chunks: {stats['coded_chunks']}/{stats['total_chunks']}\n"
            f"Unique codes: {patterns['unique_codes']}\n"
            f"Top codes: {', '.join(top_codes)}\n"
            f"Themes (excerpt): {themes_text}\n"
        )

    # Get consensus/divergent info
    cd_info = identify_consensus_and_divergent(stance_results)
    consensus_str = ', '.join(cd_info['consensus']) if cd_info['consensus'] else 'None'
    divergent_str = '; '.join(f"{sk}: {', '.join(codes)}" for sk, codes in cd_info['divergent'].items()) if cd_info['divergent'] else 'None'

    prompt = f"""You are a qualitative research methodologist analyzing the results of a multi-stance epistemic analysis — the same dataset coded through different epistemological lenses simultaneously.

STANCE-BY-STANCE RESULTS:
{"---".join(stance_summaries)}

CROSS-STANCE PATTERNS:
Consensus codes (found across all stances): {consensus_str}
Divergent codes (unique to one stance): {divergent_str}

Provide a structured synthesis:

## CONVERGENCE ANALYSIS
What patterns, codes, or themes appear consistently across stances? What does this convergence suggest about the robustness of these findings?

## DIVERGENCE ANALYSIS
What did each stance reveal that others missed? What does this tell us about the analytic affordances of each epistemological lens?

## METHODOLOGICAL INSIGHTS
What does the pattern of convergence and divergence tell us about the data itself and about the value of multi-stance analysis?

## INTEGRATED SYNTHESIS
Provide a 3-5 sentence synthesis of the overall findings, drawing on what each stance contributes to understanding the phenomenon.

Be specific — reference actual codes and themes from the stance summaries above."""

    try:
        response = api_call_with_backoff(
            client,
            model=config['ai_model'],
            max_tokens=3000,
            temperature=0.3,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    except Exception as e:
        print(f"❌ Cross-stance synthesis error: {e}")
        return None


def create_cross_stance_visualizations(stance_results, cross_lens_analyzer=None):
    """
    Create visualizations for cross-stance comparison.
    Includes friction histogram when cross_lens_analyzer is provided.
    """
    import matplotlib.pyplot as plt

    output_folder = config['output_folder']
    cross_folder = os.path.join(output_folder, 'cross_stance')
    os.makedirs(cross_folder, exist_ok=True)

    stances = list(stance_results.keys())

    # 1. Agreement Heatmap
    try:
        agreement_matrix = calculate_agreement_scores(stance_results)
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(agreement_matrix.values.astype(float), cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(len(stances)))
        ax.set_yticks(range(len(stances)))
        ax.set_xticklabels(stances, rotation=45, ha='right')
        ax.set_yticklabels(stances)
        for i in range(len(stances)):
            for j in range(len(stances)):
                val = agreement_matrix.values[i, j]
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        color='white' if float(val) > 0.5 else '#274C77', fontsize=12)
        plt.colorbar(im, label='Jaccard Similarity')
        ax.set_title('Cross-Stance Agreement (Jaccard Similarity)', fontsize=14, color='#274C77')
        plt.tight_layout()
        plt.savefig(f'{cross_folder}/agreement_heatmap.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"✅ Agreement heatmap saved")
    except Exception as e:
        print(f"⚠️ Could not create agreement heatmap: {e}")

    # 2. Unique Codes per Stance Bar Chart
    try:
        cd_info = identify_consensus_and_divergent(stance_results)
        fig, ax = plt.subplots(figsize=(10, 6))

        x_labels = []
        total_counts = []
        unique_counts = []
        consensus_counts = []

        for sk in stances:
            total = len(cd_info['stance_codes'].get(sk, set()))
            unique = len(cd_info['divergent'].get(sk, set()))
            cons = len(cd_info['consensus'])
            x_labels.append(sk)
            total_counts.append(total)
            unique_counts.append(unique)
            consensus_counts.append(cons)

        x = range(len(stances))
        width = 0.25
        ax.bar([i - width for i in x], total_counts, width, label='Total Codes', color='#274C77')
        ax.bar(x, consensus_counts, width, label='Consensus Codes', color='#6096BA')
        ax.bar([i + width for i in x], unique_counts, width, label='Unique to Stance', color='#A3CEF1')

        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=45, ha='right')
        ax.set_ylabel('Number of Codes')
        ax.set_title('Code Distribution Across Stances', fontsize=14, color='#274C77')
        ax.legend()
        plt.tight_layout()
        plt.savefig(f'{cross_folder}/code_distribution.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"✅ Code distribution chart saved")
    except Exception as e:
        print(f"⚠️ Could not create code distribution chart: {e}")

    # 3. Consensus vs Divergent Pie Chart
    try:
        consensus_n = len(cd_info['consensus'])
        divergent_n = sum(len(v) for v in cd_info['divergent'].values())
        partial_n = len(cd_info['partial_overlap'])

        if consensus_n + divergent_n + partial_n > 0:
            fig, ax = plt.subplots(figsize=(8, 6))
            sizes = [consensus_n, partial_n, divergent_n]
            labels = [f'Consensus ({consensus_n})', f'Partial Overlap ({partial_n})', f'Unique to One Stance ({divergent_n})']
            colors = ['#274C77', '#6096BA', '#A3CEF1']
            ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
            ax.set_title('Code Agreement Across Stances', fontsize=14, color='#274C77')
            plt.tight_layout()
            plt.savefig(f'{cross_folder}/consensus_divergent.png', dpi=150, bbox_inches='tight')
            plt.show()
            print(f"✅ Consensus/divergent pie chart saved")
    except Exception as e:
        print(f"⚠️ Could not create consensus chart: {e}")

    # 4. Friction Distribution Histogram (from CrossLensAnalyzer)
    if cross_lens_analyzer is not None:
        try:
            df = cross_lens_analyzer.df_coded
            if 'Agreement_Score' in df.columns:
                scores = df['Agreement_Score'].dropna()
                if len(scores) > 0:
                    fig, ax = plt.subplots(figsize=(10, 6))
                    ax.hist(scores, bins=20, color='#6096BA', edgecolor='#274C77', alpha=0.8)
                    ax.axvline(x=0.3, color='#d32f2f', linestyle='--', linewidth=2, label='Friction threshold (0.3)')
                    ax.set_xlabel('Agreement Score (Jaccard)', fontsize=12)
                    ax.set_ylabel('Number of Chunks', fontsize=12)
                    ax.set_title('Distribution of Cross-Lens Agreement', fontsize=14, color='#274C77')
                    ax.legend()

                    mean_val = scores.mean()
                    friction_pct = (scores < 0.3).mean() * 100
                    ax.text(0.02, 0.95, f'Mean: {mean_val:.3f}\nFriction chunks: {friction_pct:.1f}%',
                            transform=ax.transAxes, fontsize=11, verticalalignment='top',
                            bbox=dict(boxstyle='round', facecolor='#E7ECEF', alpha=0.8))

                    plt.tight_layout()
                    plt.savefig(f'{cross_folder}/friction_distribution.png', dpi=150, bbox_inches='tight')
                    plt.show()
                    print(f"✅ Friction distribution histogram saved")
        except Exception as e:
            print(f"⚠️ Could not create friction histogram: {e}")

        # 5. Lens Code Profiles
        try:
            profiles = cross_lens_analyzer.compute_lens_code_profiles()
            if profiles:
                all_top_codes = set()
                for prof in profiles.values():
                    all_top_codes.update(list(prof['top_codes'].keys())[:5])

                if all_top_codes:
                    top_codes_sorted = sorted(all_top_codes)[:15]
                    lens_names = list(profiles.keys())
                    x = np.arange(len(top_codes_sorted))
                    width = 0.8 / len(lens_names)

                    fig, ax = plt.subplots(figsize=(14, 8))
                    lens_colors = ['#274C77', '#6096BA', '#A3CEF1', '#8B8C89', '#E76F51', '#2A9D8F']

                    for i, lens_name in enumerate(lens_names):
                        freqs = [profiles[lens_name]['top_codes'].get(code, 0) for code in top_codes_sorted]
                        ax.bar(x + i * width, freqs, width, label=lens_name,
                               color=lens_colors[i % len(lens_colors)], alpha=0.85)

                    ax.set_xticks(x + width * (len(lens_names) - 1) / 2)
                    ax.set_xticklabels(top_codes_sorted, rotation=45, ha='right', fontsize=9)
                    ax.set_ylabel('Frequency', fontsize=12)
                    ax.set_title('Code Application by Analytical Lens', fontsize=14, color='#274C77')
                    ax.legend(title='Analytical Lens')
                    plt.tight_layout()
                    plt.savefig(f'{cross_folder}/lens_code_profiles.png', dpi=150, bbox_inches='tight')
                    plt.show()
                    print(f"✅ Lens code profiles chart saved")
        except Exception as e:
            print(f"⚠️ Could not create lens profiles chart: {e}")


def export_cross_stance_analysis(stance_results):
    """
    Export cross-stance comparison to Excel and Word.
    """
    output_folder = config['output_folder']
    cross_folder = os.path.join(output_folder, 'cross_stance')
    os.makedirs(cross_folder, exist_ok=True)
    timestamp = config['timestamp']

    stances = list(stance_results.keys())

    # Calculate all comparison data
    agreement_matrix = calculate_agreement_scores(stance_results)
    cd_info = identify_consensus_and_divergent(stance_results)

    # ── Excel Export ──
    excel_path = f'{cross_folder}/cross_stance_comparison_{timestamp}.xlsx'
    try:
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            # Sheet 1: Agreement Matrix
            agreement_matrix.to_excel(writer, sheet_name='Agreement_Scores')

            # Sheet 2: Consensus Codes
            consensus_data = [{'Code': c, 'Status': 'Consensus (all stances)'} for c in sorted(cd_info['consensus'])]
            if consensus_data:
                pd.DataFrame(consensus_data).to_excel(writer, sheet_name='Consensus_Codes', index=False)

            # Sheet 3: Divergent Codes
            divergent_data = []
            for sk, codes in cd_info['divergent'].items():
                for c in sorted(codes):
                    divergent_data.append({'Code': c, 'Unique_To_Stance': sk})
            if divergent_data:
                pd.DataFrame(divergent_data).to_excel(writer, sheet_name='Divergent_Codes', index=False)

            # Sheet 4: Per-stance summary
            summary_data = []
            for sk, res in stance_results.items():
                stats = res['integration_stats']
                patterns = res['code_patterns']
                summary_data.append({
                    'Stance': sk,
                    'Total_Chunks': stats['total_chunks'],
                    'Coded_Chunks': stats['coded_chunks'],
                    'Unique_Codes': patterns['unique_codes'],
                    'Total_Applications': patterns['total_code_applications'],
                    'Deductive_Only': stats['deductive_only'],
                    'Inductive_Only': stats['inductive_only'],
                    'Both_Types': stats['both_types'],
                    'No_Codes': stats['no_codes']
                })
            pd.DataFrame(summary_data).to_excel(writer, sheet_name='Stance_Summary', index=False)

        # Apply formatting
        from openpyxl import load_workbook
        wb = load_workbook(excel_path)
        header_font = Font(bold=True, color='FFFFFF')
        header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
        for sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            for cell in ws[1]:
                cell.font = header_font
                cell.fill = header_fill
                cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            ws.freeze_panes = 'A2'
        wb.save(excel_path)
        print(f"✅ Cross-stance Excel: {excel_path}")

    except Exception as e:
        print(f"❌ Error exporting cross-stance Excel: {e}")

    # ── Word Export ──
    word_path = f'{cross_folder}/cross_stance_report_{timestamp}.docx'
    try:
        doc = Document()

        title = doc.add_heading('Cross-Stance Comparison Report', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        doc.add_paragraph()
        p = doc.add_paragraph()
        p.add_run('Multi-Stance Epistemic Analysis').italic = True
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER

        doc.add_paragraph()
        p = doc.add_paragraph()
        p.add_run('Stances Analyzed: ').bold = True
        p.add_run(', '.join(stances))

        p = doc.add_paragraph()
        p.add_run('Generated: ').bold = True
        p.add_run(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

        doc.add_page_break()

        # Summary table
        doc.add_heading('Stance Summary', 1)
        for sk, res in stance_results.items():
            stats = res['integration_stats']
            patterns = res['code_patterns']
            doc.add_heading(f'Stance: {sk}', 2)
            doc.add_paragraph(f"Coded chunks: {stats['coded_chunks']} / {stats['total_chunks']}")
            doc.add_paragraph(f"Unique codes: {patterns['unique_codes']}")
            doc.add_paragraph(f"Total code applications: {patterns['total_code_applications']}")

        # Agreement scores
        doc.add_heading('Pairwise Agreement (Jaccard Similarity)', 1)
        for i, s1 in enumerate(stances):
            for j, s2 in enumerate(stances):
                if j > i:
                    score = agreement_matrix.loc[s1, s2]
                    doc.add_paragraph(f"{s1} ↔ {s2}: {score:.3f}", style='List Bullet')

        # Consensus / Divergent
        doc.add_heading('Consensus Codes', 1)
        if cd_info['consensus']:
            doc.add_paragraph(f"Found {len(cd_info['consensus'])} codes shared across all stances:")
            for c in sorted(cd_info['consensus']):
                doc.add_paragraph(c, style='List Bullet')
        else:
            doc.add_paragraph("No codes were shared across all stances.")

        doc.add_heading('Divergent Codes (Unique to One Stance)', 1)
        if cd_info['divergent']:
            for sk, codes in cd_info['divergent'].items():
                doc.add_heading(f'Unique to {sk}', 2)
                for c in sorted(codes):
                    doc.add_paragraph(c, style='List Bullet')
        else:
            doc.add_paragraph("No stance-unique codes found.")

        # AI Synthesis
        doc.add_page_break()
        doc.add_heading('Cross-Stance Synthesis', 1)
        synthesis = generate_cross_stance_synthesis(stance_results, config['api_key'])
        if synthesis:
            _add_md_content(doc, synthesis)
        else:
            doc.add_paragraph("Synthesis could not be generated.")

        doc.save(word_path)
        print(f"✅ Cross-stance Word report: {word_path}")

    except Exception as e:
        print(f"❌ Error exporting cross-stance Word: {e}")

    return {'excel': excel_path, 'word': word_path}


def run_cross_stance_comparison(stance_results):
    """
    Run the full cross-stance comparison pipeline: analysis, visualizations, and export.
    Includes per-chunk agreement via CrossLensAnalyzer and friction analysis.
    Requires at least 2 stances.
    """
    if len(stance_results) < 2:
        print("⚠️ Cross-stance comparison requires at least 2 stances. Skipping.")
        return None

    print("\n" + "=" * 60)
    print("🔀 CROSS-STANCE COMPARISON")
    print("=" * 60)

    # Merge per-stance results into a unified DataFrame with per-chunk agreement
    cross_lens_analyzer = None
    try:
        stances = list(stance_results.keys())
        transcript_df = config['transcript_df']
        df_merged = merge_multi_lens_results(stance_results, transcript_df)

        cross_lens_analyzer = CrossLensAnalyzer(df_merged, stances)

        # Display friction report
        print("\n🔥 Friction Analysis:")
        cross_lens_analyzer.generate_friction_report(top_n=10)
    except Exception as e:
        print(f"⚠️ Could not create cross-lens analysis: {e}")
        import traceback
        traceback.print_exc()

    # Visualizations
    print("\n📊 Creating cross-stance visualizations...")
    create_cross_stance_visualizations(stance_results, cross_lens_analyzer=cross_lens_analyzer)

    # Export
    print("\n💾 Exporting cross-stance analysis...")
    exports = export_cross_stance_analysis(stance_results)

    print("\n✅ Cross-stance comparison complete!")
    return {'exports': exports, 'cross_lens_analyzer': cross_lens_analyzer}


print("Cross-stance comparison functions loaded!")

## Thematic Analysis

*Generate detailed analysis for each identified theme using a blended thick description and applied anthropology framework. Outputs include a consolidated Word document, CSV with theme assignments, and structured JSON for downstream analysis.*

In [ ]:
class ThematicAnalyzer:
    """
    Generates detailed thematic analysis using thick description + applied anthropology framework.
    Outputs: Word document, CSV, and JSON.
    """

    def __init__(self, df_coded, themes_results, code_patterns, integration_stats,
                 inductive_results, coder, api_key):
        self.df_coded = df_coded
        self.themes_results = themes_results
        self.code_patterns = code_patterns
        self.integration_stats = integration_stats
        self.inductive_results = inductive_results
        self.coder = coder
        self.client = anthropic.Anthropic(api_key=api_key)
        self.theme_mappings = {}
        self.detailed_analyses = {}
        self._cached_cross_cutting = None

    def parse_themes(self):
        """Parse themes and their supporting codes from the initial analysis."""
        if not self.themes_results or 'themes_analysis' not in self.themes_results:
            return {}

        themes_text = self.themes_results['themes_analysis']
        theme_mappings = {}

        # Split by THEME markers (handles ## THEME X: format)
        theme_sections = re.split(r'(?:^|\n)(?:##\s*)?THEME\s+(\d+):', themes_text, flags=re.IGNORECASE)

        for i in range(1, len(theme_sections), 2):
            if i + 1 < len(theme_sections):
                theme_num = theme_sections[i].strip()
                section = theme_sections[i + 1]

                # Extract theme title (first line, clean markdown)
                title_match = re.search(r'^\s*([^\n]+)', section)
                if title_match:
                    theme_title = title_match.group(1).strip()
                    theme_title = re.sub(r'\*\*', '', theme_title)  # Remove markdown bold
                    theme_name = f"Theme {theme_num}: {theme_title}"

                    # Extract codes from multiple sources
                    codes = set()

                    # Extract Key Finding (full text, not just codes)
                    key_finding = ''
                    finding_match = re.search(r'\*\*Key Finding[s]?:\*\*\s*([^\n]+(?:\n(?!\*\*|##)[^\n]+)*)', section, re.IGNORECASE)
                    if not finding_match:
                        finding_match = re.search(r'Key Finding[s]?:\s*([^\n]+(?:\n(?!\*\*|##)[^\n]+)*)', section, re.IGNORECASE)
                    if finding_match:
                        key_finding = finding_match.group(1).strip()
                        # Extract codes from key finding
                        found_codes = re.findall(r'\b([A-Z][A-Z0-9_]+(?:_IND)?)\b', key_finding)
                        codes.update(found_codes)

                    # Extract Evidence Strength (full text)
                    evidence_strength = 'Moderate'
                    evidence_detail = ''
                    ev_match = re.search(r'\*\*Evidence Strength:\*\*\s*([^\n]+(?:\n(?!\*\*|##)[^\n]+)*)', section, re.IGNORECASE)
                    if not ev_match:
                        ev_match = re.search(r'Evidence Strength:\s*([^\n]+)', section, re.IGNORECASE)
                    if ev_match:
                        evidence_detail = ev_match.group(1).strip()
                        # Extract strength level
                        strength_level = re.search(r'\*\*(Strong|Moderate|Emerging|Weak)[^*]*\*\*', evidence_detail, re.IGNORECASE)
                        if strength_level:
                            evidence_strength = strength_level.group(1)
                        # Extract codes from evidence section
                        found_codes = re.findall(r'\b([A-Z][A-Z0-9_]+(?:_IND)?)\b', evidence_detail)
                        codes.update(found_codes)

                    # Extract Sub-themes
                    sub_themes = []
                    # Look for sub-themes section - capture until next major section
                    subtheme_match = re.search(r'\*\*Sub-themes:\*\*\s*([\s\S]*?)(?=\*\*Key Finding|\*\*Evidence Strength|---|##|$)', section, re.IGNORECASE)
                    if not subtheme_match:
                        subtheme_match = re.search(r'Sub-themes:\s*([\s\S]*?)(?=Key Finding|Evidence|---|##|$)', section, re.IGNORECASE)
                    if subtheme_match:
                        subtheme_text = subtheme_match.group(1)

                        # Parse a), b), c), d), e) format - handles both newline and inline formats
                        # Split on letter+parenthesis pattern (with optional whitespace/newline before)
                        items = re.split(r'(?:^|\s+)([a-e])\)\s*', subtheme_text)

                        # items will be: ['', 'a', 'content', 'b', 'content', 'c', 'content', ...]
                        # Process pairs (letter, content)
                        i = 1
                        while i < len(items) - 1:
                            letter = items[i]
                            content = items[i + 1].strip() if i + 1 < len(items) else ''

                            if content:
                                # Extract **Name:** Description
                                name_match = re.match(r'\*\*([^:*]+):\*\*\s*(.+)', content, re.DOTALL)
                                if name_match:
                                    name = name_match.group(1).strip()
                                    desc = name_match.group(2).strip()
                                    # Clean up description
                                    desc = re.sub(r'\s+', ' ', desc)
                                    desc = desc.replace('**', '').replace('*', '')
                                    # Remove any trailing letter+paren that got included
                                    desc = re.sub(r'\s+[b-e]\)\s*$', '', desc)
                                    sub_themes.append({
                                        'name': name,
                                        'description': desc[:500]
                                    })
                                else:
                                    # Try without bold markers: Name: Description
                                    name_match2 = re.match(r'([^:]+):\s*(.+)', content, re.DOTALL)
                                    if name_match2:
                                        name = name_match2.group(1).strip().replace('**', '')
                                        desc = name_match2.group(2).strip()
                                        desc = re.sub(r'\s+', ' ', desc).replace('**', '').replace('*', '')
                                        sub_themes.append({
                                            'name': name,
                                            'description': desc[:500]
                                        })
                            i += 2

                        # If no a), b) format found, try bullet format
                        if not sub_themes:
                            bullet_items = re.findall(r'[-•]\s*\*\*([^:*]+):\*\*\s*([^\n]+)', subtheme_text)
                            for name, desc in bullet_items:
                                sub_themes.append({
                                    'name': name.strip(),
                                    'description': desc.strip().replace('**', '')
                                })

                    # Also look for Supporting Codes line if present
                    supp_match = re.search(r'Supporting Codes:\s*([^\n]+)', section, re.IGNORECASE)
                    if supp_match:
                        found_codes = re.findall(r'\b([A-Z][A-Z0-9_]+(?:_IND)?)\b', supp_match.group(1))
                        codes.update(found_codes)

                    # Filter out non-code words that match pattern
                    codes = [c for c in codes if len(c) > 3 and c not in ['THEME', 'STRONG', 'MODERATE', 'EMERGING', 'IND', 'WEAK']]

                    # Extract core insight (handle **Core Concept:** format)
                    insight_match = re.search(r'\*\*Core (?:Insight|Concept):\*\*\s*([^\n]+(?:\n(?!\*\*)[^\n]+)*)', section, re.IGNORECASE)
                    if not insight_match:
                        insight_match = re.search(r'Core (?:Insight|Concept):\s*([^\n]+)', section, re.IGNORECASE)
                    core_insight = insight_match.group(1).strip() if insight_match else ''
                    core_insight = re.sub(r'^\*\*\s*', '', core_insight)  # Remove leading **

                    # Clean markdown from text fields
                    def clean_md(text):
                        return re.sub(r'\*\*([^*]+)\*\*', r'\1', text) if text else ''

                    theme_mappings[theme_name] = {
                        'codes': list(codes),
                        'core_insight': clean_md(core_insight),
                        'sub_themes': sub_themes,
                        'key_finding': clean_md(key_finding),
                        'evidence_strength': evidence_strength,
                        'evidence_detail': clean_md(evidence_detail),
                        'section_text': section
                    }

                    print(f"   Parsed {theme_name}: {len(codes)} codes, {len(sub_themes)} sub-themes")

        self.theme_mappings = theme_mappings
        return theme_mappings

    def get_theme_chunks(self, theme_name):
        """Get all chunks associated with a theme based on its codes."""
        if theme_name not in self.theme_mappings:
            return pd.DataFrame()

        theme_codes = self.theme_mappings[theme_name]['codes']
        chunk_id_col = config.get('chunk_id_column', 'chunk_id')

        matching_indices = []
        for idx, row in self.df_coded.iterrows():
            all_codes = []
            if pd.notna(row.get('Deductive_Codes', '')) and row['Deductive_Codes']:
                all_codes.extend([c.strip() for c in str(row['Deductive_Codes']).split(',')])
            if pd.notna(row.get('Inductive_Codes', '')) and row['Inductive_Codes']:
                all_codes.extend([c.strip() for c in str(row['Inductive_Codes']).split(',')])

            for theme_code in theme_codes:
                if any(theme_code.lower() in code.lower() for code in all_codes):
                    matching_indices.append(idx)
                    break

        return self.df_coded.loc[matching_indices].copy()

    def create_deep_analysis_prompt(self, theme_name, theme_chunks):
        """Create prompt for thematic analysis."""
        theme_info = self.theme_mappings[theme_name]
        chunk_id_col = config.get('chunk_id_column', 'chunk_id')

        sample_size = min(25, len(theme_chunks))
        sample_df = theme_chunks.sample(sample_size, random_state=42) if len(theme_chunks) > sample_size else theme_chunks

        sample_texts = []
        for idx, row in sample_df.iterrows():
            text = str(row.get('text', ''))
            record_id = row.get(chunk_id_col, f'record_{idx}')
            sample_texts.append(f"[{record_id}]: {text}")

        samples_formatted = "\n\n".join(sample_texts)
        codes_str = ', '.join(theme_info['codes'])

        prompt = f"""You are a qualitative research analyst conducting THEMATIC ANALYSIS using a blended thick description and applied anthropology framework.

THEME: {theme_name}
CORE INSIGHT: {theme_info['core_insight']}
SUPPORTING CODES: {codes_str}
EVIDENCE STRENGTH: {theme_info['evidence_strength']}
TOTAL RECORDS IN THEME: {len(theme_chunks)}
ANALYZING: {len(sample_df)} sample records

SAMPLE DATA:
{samples_formatted}

Provide a detailed analysis following this EXACT structure:

## THEME OVERVIEW
[2-3 sentences capturing what this theme represents at a high level]

## MANIFESTATIONS
Identify 2-4 distinct patterns in how this theme appears:
- **[Pattern Name]**: [Description of how this pattern manifests in the data]

## ANALYTICAL INTERPRETATION (ETIC)
Provide the researcher's analytical interpretation:
- **Core Concept**: [2-3 sentences synthesizing what this theme fundamentally represents]
- **Key Finding**: [The main analytical insight this theme reveals]

## PARTICIPANT PERSPECTIVE (EMIC)
How do participants themselves talk about and conceptualize this? What language, metaphors, or frameworks do they use? Write 2-3 sentences describing their framing.

REPRESENTATIVE QUOTES (REQUIRED - include 5-7 direct quotes from the data):
- "[Exact quote from the data above]" — [Record ID, e.g., chunk_001]
- "[Another exact quote]" — [Record ID]
- "[Continue with 5-7 total quotes]" — [Record ID]

IMPORTANT: You MUST include actual quotes from the source data provided. Use the exact text in quotation marks followed by an em-dash (—) and the chunk ID.

## TENSIONS AND CONTRADICTIONS
[Where do participants struggle, disagree, or hold competing views related to this theme?]

## STAKEHOLDER PERSPECTIVES
If different groups are identifiable:
- **[Group 1]**: [How they experience this]
- **[Group 2]**: [How they experience this]

## NEEDS, BARRIERS, AND WORKAROUNDS
- **Expressed Needs**: [What do participants need or want?]
- **Barriers**: [What obstacles do they face?]
- **Workarounds**: [How are participants currently adapting?]

## PRACTICAL IMPLICATIONS
[What does this theme suggest for design, policy, intervention, or action?]

## OPEN QUESTIONS
- [Question that remains unclear or warrants further exploration]

FORMATTING RULES:
- Do NOT use horizontal rules (---) between sections
- Do NOT use just "#" or "•" as list items - always include descriptive text
- Use **bold** for emphasis, *italic* for terms/concepts
- Keep section headers exactly as shown above (## SECTION NAME)
- Each list item should be substantive (at least 10 words)

Be thorough but concise. Ground all observations in the actual data provided."""

        return prompt

    def analyze_theme(self, theme_name, theme_chunks):
        """Run thematic analysis for a single theme."""
        print(f"   Analyzing: {theme_name} ({len(theme_chunks)} records)")

        prompt = self.create_deep_analysis_prompt(theme_name, theme_chunks)

        try:
            response = api_call_with_backoff(
                self.client,
                model=config['ai_model'],
                max_tokens=4000,
                temperature=0.3,
                messages=[{"role": "user", "content": prompt}]
            )

            analysis_text = response.content[0].text
            parsed = self.parse_analysis_sections(analysis_text)

            return {
                'raw_text': analysis_text,
                'parsed': parsed,
                'chunk_count': len(theme_chunks),
                'success': True
            }

        except Exception as e:
            print(f"      Error: {e}")
            return {'success': False, 'error': str(e)}

    def parse_analysis_sections(self, text):
        """Parse the analysis text into structured sections."""
        sections = {
            'overview': '',
            'manifestations': [],
            'analytical_interpretation': {'core_concept': '', 'key_finding': ''},
            'participant_perspective': '',
            'participant_voices': [],
            'tensions_contradictions': '',
            'stakeholder_perspectives': [],
            'needs_barriers_workarounds': {'needs': [], 'barriers': [], 'workarounds': []},
            'practical_implications': [],
            'open_questions': []
        }

        # Extract overview
        overview_match = re.search(r'## THEME OVERVIEW\s*\n(.+?)(?=##|$)', text, re.DOTALL | re.IGNORECASE)
        if overview_match:
            sections['overview'] = overview_match.group(1).strip()

        # Extract manifestations
        manifest_match = re.search(r'## MANIFESTATIONS\s*\n(.+?)(?=##|$)', text, re.DOTALL | re.IGNORECASE)
        if manifest_match:
            patterns = re.findall(r'\*\*([^*]+)\*\*:\s*([^\n]+)', manifest_match.group(1))
            sections['manifestations'] = [{'pattern': p[0].strip(), 'description': p[1].strip()} for p in patterns]

                # Extract analytical interpretation (etic)
        etic_match = re.search(r'## ANALYTICAL INTERPRETATION \(ETIC\)\s*\n(.+?)(?=##|$)', text, re.DOTALL | re.IGNORECASE)
        if etic_match:
            etic_text = etic_match.group(1)
            # Extract core concept
            core_match = re.search(r'\*\*Core Concept\*\*:\s*(.+?)(?=\*\*Key Finding|$)', etic_text, re.DOTALL)
            if core_match:
                sections['analytical_interpretation']['core_concept'] = core_match.group(1).strip()
            # Extract key finding
            finding_match = re.search(r'\*\*Key Finding\*\*:\s*(.+?)(?=##|$)', etic_text, re.DOTALL)
            if finding_match:
                sections['analytical_interpretation']['key_finding'] = finding_match.group(1).strip()

        # Extract participant perspective (emic) and quotes
        emic_match = re.search(r'## PARTICIPANT PERSPECTIVE \(EMIC\)\s*\n(.+?)(?=## TENSIONS|## STAKEHOLDER|$)', text, re.DOTALL | re.IGNORECASE)
        if emic_match:
            emic_text = emic_match.group(1)

            # Extract the framing text (before quotes section)
            framing_match = re.search(r'^(.+?)(?=Select \d|Representative [Qq]uotes|$)', emic_text, re.DOTALL)
            if framing_match:
                sections['participant_perspective'] = framing_match.group(1).strip()
            else:
                sections['participant_perspective'] = emic_text.strip()

            # Extract quotes from the emic section
            quotes = []
            # Robust patterns for quote extraction (handles brackets, hyphens in IDs)
            quote_patterns = [
                # Pattern: - "quote" — [ID] or — ID (handles optional brackets)
                r'-\s*"([^"]{20,})"\s*[—–-]+\s*\[?([A-Za-z0-9_-]+)\]?',
                # Pattern: "quote" — [ID] (no leading dash, longer quotes)
                r'"([^"]{30,})"\s*[—–-]+\s*\[?([A-Za-z0-9_-]+)\]?',
                # Pattern with curly quotes (left/right)
                r'-\s*"([^"]{20,})"\s*[—–-]+\s*\[?([A-Za-z0-9_-]+)\]?',
                r'"([^"]{30,})"\s*[—–-]+\s*\[?([A-Za-z0-9_-]+)\]?',
            ]
            for pattern in quote_patterns:
                try:
                    found = re.findall(pattern, emic_text)
                    for q in found:
                        quote_text = q[0].strip()
                        record_id = q[1].strip()
                        # Avoid duplicates
                        if quote_text and not any(v['quote'] == quote_text for v in quotes):
                            quotes.append({'quote': quote_text, 'record_id': record_id})
                except:
                    pass

            sections['participant_voices'] = quotes

        # Extract participant voices as fallback (if separate section exists)
        if not sections['participant_voices']:
            voices_match = re.search(r'## PARTICIPANT VOICES\s*\n(.+?)(?=##|$)', text, re.DOTALL | re.IGNORECASE)
            if voices_match:
                voices_text = voices_match.group(1)
                quotes = []
                # Try multiple patterns for quotes
                pattern1 = re.findall(r'-\s*["\""]([^\""]+)["\""]\s*[-—]\s*([^\n]+)', voices_text)
                quotes.extend(pattern1)
                pattern2 = re.findall(r'-\s*["\""]([^\""]+)["\""]\s*\(([^)]+)\)', voices_text)
                quotes.extend(pattern2)
                pattern3 = re.findall(r'["\""]([^\""]{20,})["\""]\s*[-—]+\s*([A-Za-z0-9_-]+)', voices_text)
                quotes.extend(pattern3)

                sections['participant_voices'] = [{'quote': q[0].strip(), 'record_id': q[1].strip()} for q in quotes if q[0].strip()]

        # Extract tensions
        tensions_match = re.search(r'## TENSIONS AND CONTRADICTIONS\s*\n(.+?)(?=##|$)', text, re.DOTALL | re.IGNORECASE)
        if tensions_match:
            sections['tensions_contradictions'] = tensions_match.group(1).strip()

        # Extract stakeholder perspectives
        stakeholder_match = re.search(r'## STAKEHOLDER PERSPECTIVES\s*\n(.+?)(?=##|$)', text, re.DOTALL | re.IGNORECASE)
        if stakeholder_match:
            perspectives = re.findall(r'\*\*([^*]+)\*\*:\s*([^\n]+)', stakeholder_match.group(1))
            sections['stakeholder_perspectives'] = [{'group': p[0].strip(), 'perspective': p[1].strip()} for p in perspectives]

        # Extract needs/barriers/workarounds
        nbw_match = re.search(r'## NEEDS,? BARRIERS,? AND WORKAROUNDS\s*\n(.+?)(?=##|$)', text, re.DOTALL | re.IGNORECASE)
        if nbw_match:
            nbw_text = nbw_match.group(1)
            needs_match = re.search(r'\*\*Expressed Needs\*\*:\s*([^\n]+)', nbw_text)
            barriers_match = re.search(r'\*\*Barriers\*\*:\s*([^\n]+)', nbw_text)
            workarounds_match = re.search(r'\*\*Workarounds\*\*:\s*([^\n]+)', nbw_text)

            if needs_match:
                sections['needs_barriers_workarounds']['needs'] = [needs_match.group(1).strip()]
            if barriers_match:
                sections['needs_barriers_workarounds']['barriers'] = [barriers_match.group(1).strip()]
            if workarounds_match:
                sections['needs_barriers_workarounds']['workarounds'] = [workarounds_match.group(1).strip()]

        # Extract practical implications
        implications_match = re.search(r'## PRACTICAL IMPLICATIONS\s*\n(.+?)(?=##|$)', text, re.DOTALL | re.IGNORECASE)
        if implications_match:
            sections['practical_implications'] = [implications_match.group(1).strip()]

        # Extract open questions
        questions_match = re.search(r'## OPEN QUESTIONS\s*\n(.+?)(?=##|$)', text, re.DOTALL | re.IGNORECASE)
        if questions_match:
            questions = re.findall(r'-\s*([^\n]+)', questions_match.group(1))
            sections['open_questions'] = [q.strip() for q in questions if q.strip()]

        return sections

    def run_deep_analysis(self):
        """Run thematic analysis for all themes."""
        print("\n🎯 THEMATIC ANALYSIS")
        print("=" * 50)

        self.parse_themes()

        if not self.theme_mappings:
            print("No themes found to analyze.")
            return None

        print(f"\nFound {len(self.theme_mappings)} themes to analyze")

        for theme_name in self.theme_mappings:
            theme_chunks = self.get_theme_chunks(theme_name)
            if len(theme_chunks) > 0:
                analysis = self.analyze_theme(theme_name, theme_chunks)
                self.detailed_analyses[theme_name] = analysis
                if analysis['success']:
                    print(f"      Complete")
                time.sleep(1)
            else:
                print(f"   {theme_name}: No matching chunks found")

        print("\nThematic analysis complete")
        return self.detailed_analyses

    def generate_cross_cutting_analysis(self):
        """Generate cross-cutting analysis across all themes (computed once, then cached)."""
        if getattr(self, '_cached_cross_cutting', None) is not None:
            return self._cached_cross_cutting

        if not self.detailed_analyses:
            return None

        theme_summaries = []
        for theme_name, analysis in self.detailed_analyses.items():
            if analysis.get('success'):
                theme_summaries.append(f"\n{theme_name}:\n{analysis['parsed'].get('overview', '')}")

        summaries_text = chr(10).join(theme_summaries)

        prompt = f"""Based on these thematic analyses, provide cross-cutting observations:

{summaries_text}

Provide:
## CONNECTIONS BETWEEN THEMES
[How do these themes relate to or influence each other?]

## OVERARCHING PATTERNS
[What patterns appear across multiple themes?]

## KEY TENSIONS
[What major tensions or contradictions exist across the dataset?]

## SYNTHESIS
[2-3 sentences synthesizing the overall findings]"""

        try:
            response = api_call_with_backoff(
                self.client,
                model=config['ai_model'],
                max_tokens=2000,
                temperature=0.3,
                messages=[{"role": "user", "content": prompt}]
            )
            self._cached_cross_cutting = response.content[0].text
            return self._cached_cross_cutting
        except Exception as e:
            print(f"Cross-cutting analysis error: {e}")
            return None

    def _add_markdown_paragraph(self, doc, text, base_style='Normal'):
        """Convert markdown text to formatted Word paragraph."""
        import re

        # Skip horizontal rules
        if text.strip() == '---' or text.strip() == '***':
            return

        # Skip malformed bullets
        if text.strip() == '#' or text.strip() == '•' or not text.strip():
            return

        # Handle heading markers
        if text.startswith('## '):
            doc.add_heading(text[3:].strip().replace('**', ''), level=2)
            return
        elif text.startswith('# '):
            doc.add_heading(text[2:].strip().replace('**', ''), level=1)
            return
        elif text.startswith('### '):
            doc.add_heading(text[4:].strip().replace('**', ''), level=3)
            return

        # Handle bullet points
        if text.startswith('- ') or text.startswith('* '):
            text = text[2:]
            p = doc.add_paragraph(style='List Bullet')
        elif re.match(r'^\d+\.\s', text):
            text = re.sub(r'^\d+\.\s', '', text)
            p = doc.add_paragraph(style='List Number')
        else:
            p = doc.add_paragraph()

        # Process bold and italic markers
        # First handle bold (**text**)
        parts = re.split(r'(\*\*[^*]+\*\*)', text)

        for part in parts:
            if part.startswith('**') and part.endswith('**'):
                # Bold text
                run = p.add_run(part[2:-2])
                run.bold = True
            else:
                # Now handle italic (*text*) within this part
                italic_parts = re.split(r'(?<!\*)\*([^*]+)\*(?!\*)', part)
                for i, ipart in enumerate(italic_parts):
                    if i % 2 == 1:  # Odd indices are the captured italic text
                        run = p.add_run(ipart)
                        run.italic = True
                    else:
                        if ipart:
                            p.add_run(ipart)

    def _add_markdown_content(self, doc, content):
        """Add multi-line markdown content to document."""
        if not content:
            return

        lines = content.split('\n')
        for line in lines:
            line = line.strip()
            # Skip horizontal rules and empty/malformed lines
            if line and line != '---' and line != '#' and line != '•':
                self._add_markdown_paragraph(doc, line)

    def _add_formatted_run(self, paragraph, text):
        """Add text to paragraph with inline markdown formatting (bold, italic)."""
        import re
        if not text:
            return

        # Process bold first (**text**)
        parts = re.split(r'(\*\*[^*]+\*\*)', text)

        for part in parts:
            if part.startswith('**') and part.endswith('**'):
                run = paragraph.add_run(part[2:-2])
                run.bold = True
            else:
                # Process italic (*text*)
                italic_parts = re.split(r'(?<!\*)\*([^*]+)\*(?!\*)', part)
                for i, ipart in enumerate(italic_parts):
                    if i % 2 == 1:
                        run = paragraph.add_run(ipart)
                        run.italic = True
                    elif ipart:
                        paragraph.add_run(ipart)

    def export_word_document(self, filename):
        """Export consolidated Word document with all analyses."""
        doc = Document()

        # Cover Page with Title and Methodology
        title = doc.add_heading('Qualitative Thematic Analysis Report', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER

        # Methodology info on cover page
        doc.add_paragraph()

        cover_info = doc.add_paragraph()
        cover_info.alignment = WD_ALIGN_PARAGRAPH.CENTER

        p1 = doc.add_paragraph()
        p1.add_run('Analysis Timestamp: ').bold = True
        p1.add_run(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

        p2 = doc.add_paragraph()
        p2.add_run('Coding Approach: ').bold = True
        p2.add_run(config['coding_approach'].title())

        p3 = doc.add_paragraph()
        p3.add_run('AI Model: ').bold = True
        p3.add_run(config['ai_model'])

        p4 = doc.add_paragraph()
        p4.add_run('Total Chunks Analyzed: ').bold = True
        p4.add_run(str(self.integration_stats['total_chunks']))

        p5 = doc.add_paragraph()
        p5.add_run('Successfully Coded: ').bold = True
        p5.add_run(str(self.integration_stats['coded_chunks']))

        p6 = doc.add_paragraph()
        p6.add_run('Unique Codes Applied: ').bold = True
        p6.add_run(str(self.code_patterns['unique_codes']))

        p7 = doc.add_paragraph()
        p7.add_run('Themes Identified: ').bold = True
        p7.add_run(str(len(self.theme_mappings)))

        doc.add_page_break()

        # Part 1: Executive Summary
        doc.add_heading('Part 1: Executive Summary', 1)

        doc.add_heading('Key Findings', 2)
        for theme_name, theme_info in self.theme_mappings.items():
            p = doc.add_paragraph(style='List Bullet')
            p.add_run(f"{theme_name}: ").bold = True
            # Use key finding for brief summary
            if theme_info.get('key_finding'):
                p.add_run(theme_info['key_finding'].replace('**', '')[:300])
            elif theme_info.get('core_insight'):
                p.add_run(theme_info['core_insight'].replace('**', '')[:300])
            else:
                p.add_run('Analysis pending.')

        # Cross-cutting insights
        cross_cutting = self.generate_cross_cutting_analysis()
        if cross_cutting:
            doc.add_heading('Cross-Cutting Insights', 2)
            self._add_markdown_content(doc, cross_cutting)

        doc.add_page_break()

        # Part 2: Thematic Overview (from initial analysis)
        doc.add_heading('Part 2: Thematic Overview', 1)
        doc.add_paragraph('This section presents the themes identified in the initial analysis, including core concepts, sub-themes, and key findings.')

        for theme_name, theme_info in self.theme_mappings.items():
            doc.add_heading(theme_name, 2)

            # Core Insight/Concept
            p = doc.add_paragraph()
            p.add_run('Core Concept: ').bold = True
            p.add_run(theme_info['core_insight'].replace('**', ''))

            # Sub-themes (from original analysis)
            if theme_info.get('sub_themes'):
                p_sub = doc.add_paragraph()
                p_sub.add_run('Sub-themes:').bold = True
                for sub in theme_info['sub_themes']:
                    p_item = doc.add_paragraph(style='List Bullet')
                    p_item.add_run(sub['name'] + ': ').bold = True
                    self._add_formatted_run(p_item, sub['description'])

            # Key Finding (from original analysis)
            if theme_info.get('key_finding'):
                p_kf = doc.add_paragraph()
                p_kf.add_run('Key Finding: ').bold = True
                self._add_formatted_run(p_kf, theme_info['key_finding'])

            # Evidence Strength
            p2 = doc.add_paragraph()
            p2.add_run('Evidence Strength: ').bold = True
            p2.add_run(theme_info['evidence_strength'])
            if theme_info.get('evidence_detail') and theme_info['evidence_detail'] != theme_info['evidence_strength']:
                p2.add_run(f" ({theme_info['evidence_detail'].replace('**', '')})")

            # Supporting Codes
            p3 = doc.add_paragraph()
            p3.add_run('Supporting Codes: ').bold = True
            p3.add_run(', '.join(theme_info['codes']))

            doc.add_paragraph()  # spacing

        doc.add_page_break()

        # Part 4: Detailed Theme Analyses
        doc.add_heading('Part 3: Detailed Theme Analyses', 1)
        doc.add_paragraph('This section provides thematic analysis of each theme, combining the original analytical framework with detailed examination of the underlying data.')

        for theme_name, theme_info in self.theme_mappings.items():
            doc.add_heading(theme_name, 2)

            # Get thematic analysis if available
            analysis = self.detailed_analyses.get(theme_name, {})
            has_deep_analysis = analysis.get('success', False)
            parsed = analysis.get('parsed', {}) if has_deep_analysis else {}

            # Include original analysis summary first
            doc.add_heading('Initial Analysis Summary', 3)

            # Core concept from original
            if theme_info.get('core_insight'):
                p = doc.add_paragraph()
                p.add_run('Core Concept: ').bold = True
                p.add_run(theme_info['core_insight'].replace('**', ''))

            # Sub-themes from original
            if theme_info.get('sub_themes'):
                p_sub = doc.add_paragraph()
                p_sub.add_run('Sub-themes:').bold = True
                for sub in theme_info['sub_themes']:
                    p_item = doc.add_paragraph(style='List Bullet')
                    p_item.add_run(sub['name'] + ': ').bold = True
                    self._add_formatted_run(p_item, sub['description'])

            # Key finding from original
            if theme_info.get('key_finding'):
                p_kf = doc.add_paragraph()
                p_kf.add_run('Key Finding: ').bold = True
                p_kf.add_run(theme_info['key_finding'].replace('**', ''))

            # Evidence strength
            p_ev = doc.add_paragraph()
            p_ev.add_run('Evidence Strength: ').bold = True
            p_ev.add_run(theme_info.get('evidence_strength', 'Moderate'))

            # Now the thematic analysis (if available)
            if has_deep_analysis:
                doc.add_heading('Deep Analysis', 3)

                doc.add_heading('Overview', 4)
                self._add_markdown_content(doc, parsed.get('overview', 'No overview available.'))

            doc.add_heading('Manifestations', 4)
            manifestations = parsed.get('manifestations', [])
            if manifestations:
                for m in manifestations:
                    p = doc.add_paragraph(style='List Bullet')
                    p.add_run(m['pattern'] + ': ').bold = True
                    self._add_formatted_run(p, m['description'])
            else:
                doc.add_paragraph('No specific manifestations identified.')

            # Analytical Interpretation (Etic)
            doc.add_heading('Analytical Interpretation (Etic)', 4)
            etic = parsed.get('analytical_interpretation', {})
            if etic.get('core_concept'):
                p = doc.add_paragraph()
                p.add_run('Core Concept: ').bold = True
                self._add_formatted_run(p, etic['core_concept'])
            if etic.get('key_finding'):
                p = doc.add_paragraph()
                p.add_run('Key Finding: ').bold = True
                self._add_formatted_run(p, etic['key_finding'])
            if not etic.get('core_concept') and not etic.get('key_finding'):
                doc.add_paragraph('No analytical interpretation provided.')

            # Participant Perspective (Emic)
            doc.add_heading('Participant Perspective (Emic)', 4)
            emic_content = parsed.get('participant_perspective', '')
            if emic_content:
                # Split to get the framing part (before quotes list)
                framing_parts = emic_content.split('Select ')
                if framing_parts[0].strip():
                    self._add_markdown_content(doc, framing_parts[0].strip())
            else:
                doc.add_paragraph('No participant perspective provided.')

            doc.add_heading('Participant Voices', 4)
            voices = parsed.get('participant_voices', [])
            if voices:
                for voice in voices:
                    p = doc.add_paragraph()
                    p.add_run(f'"{voice["quote"]}"').italic = True
                    p.add_run(f" — {voice['record_id']}")
            else:
                doc.add_paragraph('No direct quotes extracted for this theme.')

            doc.add_heading('Tensions & Contradictions', 4)
            self._add_markdown_content(doc, parsed.get('tensions_contradictions', 'Not available.'))

            doc.add_heading('Stakeholder Perspectives', 4)
            perspectives = parsed.get('stakeholder_perspectives', [])
            if perspectives:
                for sp in perspectives:
                    p = doc.add_paragraph(style='List Bullet')
                    p.add_run(sp['group'] + ': ').bold = True
                    self._add_formatted_run(p, sp['perspective'])
            else:
                doc.add_paragraph('No distinct stakeholder perspectives identified.')

            doc.add_heading('Needs, Barriers & Workarounds', 4)
            nbw = parsed.get('needs_barriers_workarounds', {})
            has_nbw = False

            # Filter function to remove markdown and empty items
            def clean_items(items):
                return [item.replace('**', '').replace('*', '') for item in items if item and item.strip() not in ['#', '•', '-', '']]

            if nbw.get('needs'):
                needs = clean_items(nbw['needs'])
                if needs:
                    p = doc.add_paragraph()
                    p.add_run('Expressed Needs: ').bold = True
                    p.add_run('; '.join(needs))
                    has_nbw = True
            if nbw.get('barriers'):
                barriers = clean_items(nbw['barriers'])
                if barriers:
                    p = doc.add_paragraph()
                    p.add_run('Barriers: ').bold = True
                    p.add_run('; '.join(barriers))
                    has_nbw = True
            if nbw.get('workarounds'):
                workarounds = clean_items(nbw['workarounds'])
                if workarounds:
                    p = doc.add_paragraph()
                    p.add_run('Workarounds: ').bold = True
                    p.add_run('; '.join(workarounds))
                    has_nbw = True
            if not has_nbw:
                doc.add_paragraph('No specific needs, barriers, or workarounds identified.')

            doc.add_heading('Practical Implications', 4)
            implications = parsed.get('practical_implications', [])
            # Filter out malformed items
            valid_implications = [imp for imp in implications if imp and imp.strip() not in ['#', '•', '-', '*', '']]
            if valid_implications:
                for imp in valid_implications:
                    p = doc.add_paragraph(style='List Bullet')
                    self._add_formatted_run(p, imp.replace('**', ''))
            else:
                doc.add_paragraph('No specific practical implications identified.')

            doc.add_heading('Open Questions', 4)
            questions = parsed.get('open_questions', [])
            # Filter out malformed items
            valid_questions = [q for q in questions if q and q.strip() not in ['#', '•', '-', '*', '']]
            if valid_questions:
                for q in valid_questions:
                    p = doc.add_paragraph(style='List Bullet')
                    self._add_formatted_run(p, q.replace('**', ''))
            else:
                doc.add_paragraph('No open questions identified.')

            doc.add_page_break()

        # Appendix
        doc.add_heading('Appendix A: Code Frequency Summary', 1)
        doc.add_paragraph(f"Total code applications: {self.code_patterns['total_code_applications']}")
        doc.add_paragraph(f"Unique codes: {self.code_patterns['unique_codes']}")
        doc.add_paragraph('Top 15 codes by frequency:')
        for code, freq in list(self.code_patterns['all_codes_frequency'].items())[:15]:
            doc.add_paragraph(f"{code}: {freq}", style='List Bullet')

        doc.save(filename)
        return filename

    def export_themed_data(self, filename):
        """Export XLSX with theme assignments for each chunk, professionally formatted."""
        from openpyxl import Workbook
        from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
        from openpyxl.utils.dataframe import dataframe_to_rows

        export_df = self.df_coded.copy()
        export_df['Primary_Theme'] = ''
        export_df['All_Themes'] = ''
        export_df['Theme_Count'] = 0

        for idx, row in export_df.iterrows():
            all_codes = []
            if pd.notna(row.get('Deductive_Codes', '')) and row['Deductive_Codes']:
                all_codes.extend([c.strip() for c in str(row['Deductive_Codes']).split(',')])
            if pd.notna(row.get('Inductive_Codes', '')) and row['Inductive_Codes']:
                all_codes.extend([c.strip() for c in str(row['Inductive_Codes']).split(',')])

            matching_themes = []
            for theme_name, theme_info in self.theme_mappings.items():
                for theme_code in theme_info['codes']:
                    if any(theme_code.lower() in code.lower() for code in all_codes):
                        matching_themes.append(theme_name)
                        break

            if matching_themes:
                export_df.at[idx, 'Primary_Theme'] = matching_themes[0]
                export_df.at[idx, 'All_Themes'] = '; '.join(matching_themes)
                export_df.at[idx, 'Theme_Count'] = len(matching_themes)

        # Create Excel with formatting
        wb = Workbook()
        ws = wb.active
        ws.title = "Coded_Data_With_Themes"

        # Write dataframe
        for r_idx, row in enumerate(dataframe_to_rows(export_df, index=False, header=True), 1):
            for c_idx, value in enumerate(row, 1):
                ws.cell(row=r_idx, column=c_idx, value=value)

        # Styling
        header_fill = PatternFill(start_color="274C77", end_color="274C77", fill_type="solid")
        header_font = Font(bold=True, color="FFFFFF")
        thin_border = Border(
            left=Side(style='thin'),
            right=Side(style='thin'),
            top=Side(style='thin'),
            bottom=Side(style='thin')
        )

        # Apply header formatting
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = thin_border

        # Apply data formatting
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
            for cell in row:
                cell.border = thin_border
                cell.alignment = Alignment(vertical='top', wrap_text=True)

        # Set column widths
        column_widths = {
            'A': 20,  # chunk_id
            'B': 60,  # text
            'C': 30,  # Deductive_Codes
            'D': 18,  # Coding_Status
            'E': 30,  # Inductive_Codes
            'F': 40,  # All_Codes
            'G': 12,  # Total_Code_Count
            'H': 45,  # Primary_Theme
            'I': 60,  # All_Themes
            'J': 12,  # Theme_Count
        }
        for col, width in column_widths.items():
            ws.column_dimensions[col].width = width

        # Freeze header row
        ws.freeze_panes = 'A2'

        # Set row heights for better readability
        for row_num in range(2, ws.max_row + 1):
            ws.row_dimensions[row_num].height = 60

        wb.save(filename)
        return filename

    def export_json(self, filename):
        """Export structured JSON for downstream analysis."""
        output = {
            'metadata': {
                'timestamp': datetime.now().isoformat(),
                'coding_approach': config['coding_approach'],
                'ai_model': config['ai_model'],
                'total_chunks': int(self.integration_stats['total_chunks']),
                'coded_chunks': int(self.integration_stats['coded_chunks']),
                'unique_codes': int(self.code_patterns['unique_codes']),
                'total_code_applications': int(self.code_patterns['total_code_applications'])
            },
            'executive_summary': {
                'key_findings': [],
                'cross_cutting_insights': ''
            },
            'themes': [],
            'codes': {
                'frequencies': {},
                'edges': [],
                'co_occurrences': []
            }
        }

        for theme_name, analysis in self.detailed_analyses.items():
            theme_info = self.theme_mappings.get(theme_name, {})
            if analysis.get('success'):
                output['executive_summary']['key_findings'].append({
                    'theme': theme_name,
                    'core_concept': theme_info.get('core_insight', ''),
                    'key_finding': theme_info.get('key_finding', ''),
                    'deep_analysis_summary': analysis['parsed'].get('overview', '')[:300]
                })

        # Add cross-cutting insights
        cross_cutting = self.generate_cross_cutting_analysis()
        if cross_cutting:
            output['executive_summary']['cross_cutting_insights'] = cross_cutting

        for i, (theme_name, theme_info) in enumerate(self.theme_mappings.items(), 1):
            theme_chunks = self.get_theme_chunks(theme_name)
            analysis = self.detailed_analyses.get(theme_name, {})

            theme_data = {
                'theme_id': i,
                'theme_name': theme_name,
                'initial_analysis': {
                    'core_concept': theme_info.get('core_insight', ''),
                    'sub_themes': theme_info.get('sub_themes', []),
                    'key_finding': theme_info.get('key_finding', ''),
                    'evidence_strength': theme_info.get('evidence_strength', 'Moderate'),
                    'evidence_detail': theme_info.get('evidence_detail', '')
                },
                'supporting_codes': theme_info['codes'],
                'chunk_count': len(theme_chunks),
                'deep_analysis': {}
            }

            if analysis.get('success'):
                # Clean markdown from deep_analysis fields
                def clean_markdown(text):
                    if not text or not isinstance(text, str):
                        return text
                    import re
                    # Remove bold markers
                    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
                    # Remove italic markers
                    text = re.sub(r'(?<!\*)\*([^*]+)\*(?!\*)', r'\1', text)
                    return text

                cleaned_analysis = {}
                for key, value in analysis['parsed'].items():
                    if isinstance(value, str):
                        cleaned_analysis[key] = clean_markdown(value)
                    elif isinstance(value, list):
                        cleaned_list = []
                        for item in value:
                            if isinstance(item, dict):
                                cleaned_item = {k: clean_markdown(v) if isinstance(v, str) else v for k, v in item.items()}
                                cleaned_list.append(cleaned_item)
                            elif isinstance(item, str):
                                cleaned_list.append(clean_markdown(item))
                            else:
                                cleaned_list.append(item)
                        cleaned_analysis[key] = cleaned_list
                    elif isinstance(value, dict):
                        cleaned_analysis[key] = {k: clean_markdown(v) if isinstance(v, str) else v for k, v in value.items()}
                    else:
                        cleaned_analysis[key] = value

                theme_data['deep_analysis'] = cleaned_analysis

            output['themes'].append(theme_data)

        # Compute pairwise co-occurrence edges (like the CSV export)
        edge_weights = {}
        for idx, row in self.df_coded.iterrows():
            if pd.notna(row.get('All_Codes', '')) and row['All_Codes']:
                codes = [c.strip() for c in str(row['All_Codes']).split(',')]
                # Create pairwise edges
                for i in range(len(codes)):
                    for j in range(i+1, len(codes)):
                        edge_key = tuple(sorted([codes[i], codes[j]]))
                        edge_weights[edge_key] = edge_weights.get(edge_key, 0) + 1

        # Add edges to output (min weight 3 to match CSV export)
        for (source_code, target_code), weight in sorted(edge_weights.items(), key=lambda x: -x[1]):
            if weight >= 3:
                output['codes']['edges'].append({
                    'source': source_code,
                    'target': target_code,
                    'weight': int(weight)
                })

        # Add all code frequencies
        for code, freq in self.code_patterns['all_codes_frequency'].items():
            code_type = 'inductive' if code.endswith('_IND') else 'deductive'
            output['codes']['frequencies'][code] = {
                'frequency': int(freq),
                'type': code_type
            }

        if 'frequent_combinations' in self.code_patterns:
            for combo, freq in list(self.code_patterns['frequent_combinations'].items())[:20]:
                output['codes']['co_occurrences'].append({
                    'codes': list(combo),
                    'frequency': int(freq)
                })

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(output, f, indent=2, ensure_ascii=False)

        return filename

    def export_html_dashboard(self, filename):
        """Export interactive HTML dashboard with embedded data."""
        import json
        import re

        # Helper function to clean markdown formatting
        def clean_markdown(text):
            if not text:
                return ''
            text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)  # Remove bold
            text = re.sub(r'(?<!\*)\*([^*]+)\*(?!\*)', r'\1', text)  # Remove italic
            return text

        # Build the same data structure as export_json
        output = {
            'metadata': {
                'timestamp': datetime.now().isoformat(),
                'coding_approach': config['coding_approach'],
                'ai_model': config['ai_model'],
                'total_chunks': int(self.integration_stats['total_chunks']),
                'coded_chunks': int(self.integration_stats['coded_chunks']),
                'unique_codes': int(self.code_patterns['unique_codes']),
                'total_code_applications': int(self.code_patterns['total_code_applications'])
            },
            'executive_summary': {
                'key_findings': [],
                'cross_cutting_insights': ''
            },
            'themes': [],
            'codes': {
                'frequencies': {},
                'edges': [],
                'co_occurrences': []
            }
        }

        for theme_name, analysis in self.detailed_analyses.items():
            theme_info = self.theme_mappings.get(theme_name, {})
            if analysis.get('success'):
                output['executive_summary']['key_findings'].append({
                    'theme': theme_name,
                    'core_concept': theme_info.get('core_insight', ''),
                    'key_finding': theme_info.get('key_finding', ''),
                    'deep_analysis_summary': analysis['parsed'].get('overview', '')[:300]
                })

        # Add cross-cutting insights (computed once and cached on the instance)
        cross_cutting = self.generate_cross_cutting_analysis()
        if cross_cutting:
            output['executive_summary']['cross_cutting_insights'] = cross_cutting

        for i, (theme_name, theme_info) in enumerate(self.theme_mappings.items(), 1):
            theme_chunks = self.get_theme_chunks(theme_name)
            analysis = self.detailed_analyses.get(theme_name, {})

            theme_data = {
                'theme_id': i,
                'theme_name': theme_name,
                'initial_analysis': {
                    'core_concept': theme_info.get('core_insight', ''),
                    'sub_themes': theme_info.get('sub_themes', []),
                    'key_finding': theme_info.get('key_finding', ''),
                    'evidence_strength': theme_info.get('evidence_strength', 'Moderate'),
                    'evidence_detail': theme_info.get('evidence_detail', '')
                },
                'supporting_codes': theme_info['codes'],
                'chunk_count': len(theme_chunks),
                'deep_analysis': {}
            }

            if analysis.get('success'):
                parsed = analysis['parsed']
                theme_data['deep_analysis'] = {
                    'overview': clean_markdown(parsed.get('overview', '')),
                    'manifestations': parsed.get('manifestations', []),
                    'analytical_interpretation': {
                        'core_concept': clean_markdown(parsed.get('analytical_interpretation', {}).get('core_concept', '')),
                        'key_finding': clean_markdown(parsed.get('analytical_interpretation', {}).get('key_finding', ''))
                    },
                    'participant_perspective': clean_markdown(parsed.get('participant_perspective', '')),
                    'participant_voices': parsed.get('participant_voices', []),
                    'tensions_contradictions': clean_markdown(parsed.get('tensions_contradictions', '')),
                    'stakeholder_perspectives': parsed.get('stakeholder_perspectives', []),
                    'needs_barriers_workarounds': parsed.get('needs_barriers_workarounds', {}),
                    'practical_implications': [clean_markdown(imp) for imp in parsed.get('practical_implications', [])],
                    'open_questions': parsed.get('open_questions', [])
                }

            output['themes'].append(theme_data)

        # Add code data (same pattern as export_json)
        for code, freq in self.code_patterns['all_codes_frequency'].items():
            code_type = 'inductive' if code.endswith('_IND') else 'deductive'
            output['codes']['frequencies'][code] = {
                'frequency': int(freq),
                'type': code_type
            }

        # Compute pairwise co-occurrence edges (same as export_json)
        edge_weights = {}
        for idx, row in self.df_coded.iterrows():
            if pd.notna(row.get('All_Codes', '')) and row['All_Codes']:
                codes = [c.strip() for c in str(row['All_Codes']).split(',')]
                for i in range(len(codes)):
                    for j in range(i+1, len(codes)):
                        edge_key = tuple(sorted([codes[i], codes[j]]))
                        edge_weights[edge_key] = edge_weights.get(edge_key, 0) + 1

        # Add edges to output (min weight 3)
        for (source_code, target_code), weight in sorted(edge_weights.items(), key=lambda x: -x[1]):
            if weight >= 3:
                output['codes']['edges'].append({
                    'source': source_code,
                    'target': target_code,
                    'weight': int(weight)
                })

        # Add co_occurrences
        if 'frequent_combinations' in self.code_patterns:
            for combo, freq in list(self.code_patterns['frequent_combinations'].items())[:20]:
                output['codes']['co_occurrences'].append({
                    'codes': list(combo),
                    'frequency': int(freq)
                })

        # Generate JSON data string
        # Escape '</' so embedded data cannot terminate the <script> block
        json_data = json.dumps(output, ensure_ascii=False).replace('</', '<\\/')

        # HTML template - start (before data injection point)
        html_start = r'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Qualitative Analysis Dashboard | AI Anthropology Toolkit</title>

    <!-- Fonts -->
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Source+Serif+4:opsz,wght@8..60,400;8..60,600;8..60,700&family=IBM+Plex+Sans:wght@400;500;600;700&family=IBM+Plex+Mono:wght@400;500&display=swap" rel="stylesheet">

    <!-- No external dependencies needed -->

    <style>
        :root {
            --color-bg: #E7ECEF;
            --color-primary: #274C77;
            --color-accent: #6096BA;
            --color-secondary: #A3CEF1;
            --color-neutral: #8B8C89;
            --color-deductive: #274C77;
            --color-inductive: #6096BA;
            --color-white: #ffffff;
            --font-display: 'Source Serif 4', Georgia, serif;
            --font-body: 'IBM Plex Sans', system-ui, sans-serif;
            --font-mono: 'IBM Plex Mono', monospace;
        }

        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: var(--font-body);
            background: var(--color-bg);
            color: var(--color-primary);
            line-height: 1.6;
            min-height: 100vh;
        }

        /* Layout */
        .app-container {
            display: flex;
            min-height: 100vh;
        }

        /* Sidebar */
        .sidebar {
            width: 260px;
            background: var(--color-primary);
            padding: 1.5rem;
            display: flex;
            flex-direction: column;
            position: fixed;
            height: 100vh;
            overflow-y: auto;
        }

        .sidebar-brand {
            margin-bottom: 2rem;
            padding-bottom: 1rem;
            border-bottom: 1px solid rgba(255,255,255,0.1);
        }

        .sidebar-brand h1 {
            font-family: var(--font-display);
            color: white;
            font-size: 1.25rem;
            font-weight: 600;
            display: flex;
            align-items: center;
            gap: 0.5rem;
        }

        .sidebar-brand p {
            color: rgba(255,255,255,0.6);
            font-size: 0.75rem;
            margin-top: 0.25rem;
        }

        .nav-item {
            display: flex;
            align-items: center;
            gap: 0.75rem;
            padding: 0.75rem 1rem;
            border-radius: 8px;
            color: rgba(255,255,255,0.7);
            cursor: pointer;
            transition: all 0.2s;
            margin-bottom: 0.25rem;
            font-weight: 500;
            font-size: 0.9rem;
        }

        .nav-item:hover {
            background: rgba(255,255,255,0.1);
            color: white;
        }

        .nav-item.active {
            background: var(--color-accent);
            color: white;
        }

        .nav-item svg {
            width: 20px;
            height: 20px;
            flex-shrink: 0;
        }

        /* Main Content */
        .main-content {
            flex: 1;
            margin-left: 260px;
            padding: 2rem;
            overflow-x: hidden;
        }

        .content-wrapper {
            max-width: 1200px;
            margin: 0 auto;
        }

        /* Page Headers */
        .page-header {
            margin-bottom: 2rem;
        }

        .page-header h1 {
            font-family: var(--font-display);
            font-size: 2rem;
            font-weight: 700;
            color: var(--color-primary);
            margin-bottom: 0.5rem;
        }

        .page-header p {
            color: var(--color-neutral);
        }

        /* Cards */
        .card {
            background: white;
            border-radius: 12px;
            padding: 1.5rem;
            box-shadow: 0 1px 3px rgba(0,0,0,0.08);
            border: 1px solid rgba(0,0,0,0.05);
        }

        .card-accent {
            border-left: 4px solid var(--color-accent);
        }

        .card h3 {
            font-family: var(--font-display);
            font-size: 1.1rem;
            font-weight: 600;
            color: var(--color-primary);
            margin-bottom: 0.75rem;
        }

        /* Stats Grid */
        .stats-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 1rem;
            margin-bottom: 2rem;
        }

        .stat-card {
            background: white;
            border-radius: 12px;
            padding: 1.25rem;
            display: flex;
            align-items: center;
            gap: 1rem;
            box-shadow: 0 1px 3px rgba(0,0,0,0.08);
        }

        .stat-icon {
            width: 48px;
            height: 48px;
            border-radius: 12px;
            display: flex;
            align-items: center;
            justify-content: center;
            flex-shrink: 0;
        }

        .stat-icon svg {
            width: 24px;
            height: 24px;
            color: white;
        }

        .stat-content h4 {
            font-size: 0.8rem;
            color: var(--color-neutral);
            text-transform: uppercase;
            letter-spacing: 0.05em;
            margin-bottom: 0.25rem;
        }

        .stat-content .value {
            font-size: 1.75rem;
            font-weight: 700;
            color: var(--color-primary);
            line-height: 1;
        }

        /* Theme Cards */
        .themes-grid {
            display: grid;
            gap: 1rem;
            margin-bottom: 2rem;
        }

        .theme-card {
            background: white;
            border-radius: 12px;
            padding: 1.25rem;
            cursor: pointer;
            transition: all 0.2s;
            border: 1px solid transparent;
            display: flex;
            align-items: flex-start;
            gap: 1rem;
        }

        .theme-card:hover {
            border-color: var(--color-accent);
            box-shadow: 0 4px 12px rgba(39, 76, 119, 0.1);
            transform: translateY(-2px);
        }

        .theme-number {
            width: 36px;
            height: 36px;
            border-radius: 8px;
            background: var(--color-secondary);
            color: var(--color-primary);
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 700;
            flex-shrink: 0;
        }

        .theme-content {
            flex: 1;
            min-width: 0;
        }

        .theme-content h4 {
            font-family: var(--font-display);
            font-size: 1rem;
            font-weight: 600;
            color: var(--color-primary);
            margin-bottom: 0.5rem;
            line-height: 1.3;
        }

        .theme-content p {
            font-size: 0.875rem;
            color: var(--color-neutral);
            display: -webkit-box;
            -webkit-line-clamp: 2;
            -webkit-box-orient: vertical;
            overflow: hidden;
        }

        .theme-meta {
            display: flex;
            gap: 1rem;
            margin-top: 0.75rem;
            font-size: 0.75rem;
        }

        .theme-meta span {
            display: flex;
            align-items: center;
            gap: 0.25rem;
            color: var(--color-neutral);
        }

        .theme-arrow {
            color: var(--color-accent);
            align-self: center;
        }

        /* Theme Detail Page */
        .back-button {
            display: inline-flex;
            align-items: center;
            gap: 0.5rem;
            padding: 0.5rem 1rem;
            background: white;
            border: 1px solid #e5e7eb;
            border-radius: 8px;
            cursor: pointer;
            font-size: 0.875rem;
            color: var(--color-primary);
            margin-bottom: 1.5rem;
            transition: all 0.2s;
        }

        .back-button:hover {
            background: var(--color-bg);
        }

        .theme-detail-header {
            background: white;
            border-radius: 12px;
            padding: 1.5rem;
            margin-bottom: 1.5rem;
            border-left: 4px solid var(--color-accent);
        }

        .theme-detail-header h2 {
            font-family: var(--font-display);
            font-size: 1.5rem;
            font-weight: 700;
            color: var(--color-primary);
            margin-bottom: 0.75rem;
        }

        .evidence-badge {
            display: inline-block;
            padding: 0.25rem 0.75rem;
            border-radius: 20px;
            font-size: 0.75rem;
            font-weight: 600;
            margin-right: 0.5rem;
        }

        .evidence-strong { background: #274C77 !important; color: #ffffff !important; }
        .evidence-moderate { background: #3d5a80 !important; color: #ffffff !important; }
        .evidence-emerging { background: #d4e8f7 !important; color: #274C77 !important; }

        /* Tabs */
        .tabs {
            display: flex;
            gap: 0.5rem;
            margin-bottom: 1.5rem;
            flex-wrap: wrap;
        }

        .tab {
            padding: 0.625rem 1rem;
            border-radius: 8px;
            cursor: pointer;
            font-size: 0.875rem;
            font-weight: 500;
            transition: all 0.2s;
            background: white;
            color: var(--color-neutral);
            border: 1px solid #e5e7eb;
        }

        .tab:hover {
            color: var(--color-primary);
            border-color: var(--color-accent);
        }

        .tab.active {
            background: var(--color-accent);
            color: white;
            border-color: var(--color-accent);
        }

        /* Section Styles */
        .section-grid {
            display: grid;
            gap: 1rem;
        }

        .section-grid-2 {
            grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
        }

        .section-grid-3 {
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
        }

        /* Sub-themes */
        .sub-theme {
            display: flex;
            gap: 0.75rem;
            padding: 1rem;
            background: #f8fafc;
            border-radius: 8px;
            margin-bottom: 0.75rem;
        }

        .sub-theme-letter {
            font-weight: 700;
            color: var(--color-accent);
        }

        .sub-theme h4 {
            font-weight: 600;
            color: var(--color-primary);
            margin-bottom: 0.25rem;
        }

        .sub-theme p {
            font-size: 0.875rem;
            color: var(--color-neutral);
        }

        /* Manifestation Cards */
        .manifestation {
            background: white;
            border-radius: 8px;
            padding: 1rem;
            border: 1px solid #e5e7eb;
        }

        .manifestation h4 {
            font-weight: 600;
            color: var(--color-primary);
            margin-bottom: 0.5rem;
            font-size: 0.9rem;
        }

        .manifestation p {
            font-size: 0.85rem;
            color: var(--color-neutral);
        }

        /* Etic/Emic Sections */
        .interpretation-section {
            background: white;
            border-radius: 12px;
            padding: 1.5rem;
            margin-bottom: 1rem;
        }

        .interpretation-section.etic {
            border-left: 4px solid var(--color-deductive);
        }

        .interpretation-section.emic {
            border-left: 4px solid var(--color-inductive);
            background: #f0f7fc;
        }

        .interpretation-section h3 {
            display: flex;
            align-items: center;
            gap: 0.5rem;
            margin-bottom: 1rem;
        }

        .interpretation-section .label {
            font-size: 0.7rem;
            padding: 0.2rem 0.5rem;
            border-radius: 4px;
            font-weight: 600;
            text-transform: uppercase;
            letter-spacing: 0.05em;
        }

        .etic .label { background: #e7ecef; color: #274C77; }
        .emic .label { background: #d4e8f7; color: #3d6a8a; }

        .interpretation-item {
            margin-bottom: 1rem;
        }

        .interpretation-item strong {
            display: block;
            color: var(--color-primary);
            margin-bottom: 0.25rem;
        }

        .interpretation-item p {
            color: var(--color-neutral);
            font-size: 0.9rem;
        }

        /* Participant Voices */
        .quote-card {
            border-left: 3px solid var(--color-secondary);
            padding: 1rem;
            padding-left: 1.25rem;
            margin-bottom: 1rem;
            background: white;
            border-radius: 0 8px 8px 0;
        }

        .quote-card blockquote {
            font-style: italic;
            color: #4b5563;
            margin-bottom: 0.5rem;
            line-height: 1.6;
        }

        .quote-card cite {
            font-style: normal;
            font-size: 0.8rem;
            color: var(--color-accent);
            font-family: var(--font-mono);
        }

        /* Tensions Section */
        .tensions-card {
            background: #f5f7f9;
            border: 1px solid #8B8C89;
            border-radius: 12px;
            padding: 1.5rem;
        }

        .tensions-card h3 {
            color: #274C77;
            display: flex;
            align-items: center;
            gap: 0.5rem;
        }

        .tensions-card p {
            color: #4a4a4a;
        }

        /* Stakeholder Cards */
        .stakeholder-card {
            background: white;
            border-radius: 12px;
            padding: 1.25rem;
            border: 1px solid #e5e7eb;
        }

        .stakeholder-card h4 {
            font-weight: 600;
            color: var(--color-primary);
            margin-bottom: 0.5rem;
            display: flex;
            align-items: center;
            gap: 0.5rem;
        }

        .stakeholder-card p {
            font-size: 0.875rem;
            color: var(--color-neutral);
        }

        /* Needs/Barriers/Workarounds */
        .nbw-card {
            border-radius: 12px;
            padding: 1.25rem;
        }

        .nbw-card.needs { background: #e7f3fa; }
        .nbw-card.barriers { background: #f1f1f0; }
        .nbw-card.workarounds { background: #e7ecef; }

        .nbw-card h4 {
            font-weight: 600;
            margin-bottom: 0.75rem;
        }

        .nbw-card.needs h4 { color: #274C77; }
        .nbw-card.barriers h4 { color: #5c5c5a; }
        .nbw-card.workarounds h4 { color: #3d6a8a; }

        .nbw-card li {
            font-size: 0.875rem;
            margin-bottom: 0.5rem;
            padding-left: 0.5rem;
        }

        .nbw-card.needs li { color: #274C77; }
        .nbw-card.barriers li { color: #5c5c5a; }
        .nbw-card.workarounds li { color: #3d6a8a; }

        /* Open Questions */
        .open-question {
            padding: 0.75rem 1rem;
            background: #f0f9ff;
            border-radius: 8px;
            margin-bottom: 0.5rem;
            font-size: 0.9rem;
            color: var(--color-primary);
            border-left: 3px solid var(--color-accent);
        }

        /* Coding */
        .chart-container {
            background: white;
            border-radius: 12px;
            padding: 1.5rem;
            margin-bottom: 1.5rem;
        }

        .code-bar {
            display: flex;
            align-items: center;
            gap: 0.75rem;
            margin-bottom: 0.5rem;
            padding: 0.5rem 0;
        }

        .code-bar .name {
            width: 200px;
            font-family: var(--font-mono);
            font-size: 0.75rem;
            color: var(--color-primary);
            text-align: right;
            overflow: hidden;
            text-overflow: ellipsis;
            white-space: nowrap;
        }

        .code-bar .name:hover {
            text-decoration: underline;
        }

        .code-bar .bar-container {
            flex: 1;
            height: 24px;
            background: #f3f4f6;
            border-radius: 4px;
            overflow: hidden;
        }

        .code-bar .bar {
            height: 100%;
            border-radius: 4px;
            transition: width 0.3s ease;
        }

        .code-bar .freq {
            width: 40px;
            font-weight: 600;
            font-size: 0.875rem;
        }

        .legend {
            display: flex;
            justify-content: center;
            gap: 2rem;
            margin-top: 1rem;
        }

        .legend-item {
            display: flex;
            align-items: center;
            gap: 0.5rem;
            font-size: 0.85rem;
            color: var(--color-neutral);
        }

        .legend-dot {
            width: 12px;
            height: 12px;
            border-radius: 3px;
        }

        /* Code Table */
        .code-table {
            width: 100%;
            border-collapse: collapse;
        }

        .code-table th {
            text-align: left;
            padding: 0.75rem 1rem;
            border-bottom: 2px solid #e5e7eb;
            font-weight: 600;
            color: var(--color-primary);
            font-size: 0.85rem;
        }

        .code-table td {
            padding: 0.75rem 1rem;
            border-bottom: 1px solid #f3f4f6;
            font-size: 0.875rem;
        }

        .code-table tr:hover {
            background: #f9fafb;
        }

        .code-table .code-name {
            font-family: var(--font-mono);
            font-size: 0.75rem;
        }

        .type-badge {
            display: inline-block;
            padding: 0.2rem 0.5rem;
            border-radius: 4px;
            font-size: 0.7rem;
            font-weight: 600;
        }

        .type-badge.deductive {
            background: #274C77;
            color: #ffffff;
        }

        .type-badge.inductive {
            background: #6096BA;
            color: #ffffff;
        }

        /* Network Visualization */
        #network-container {
            background: white;
            border-radius: 12px;
            padding: 1rem;
            min-height: 600px;
        }

        #network-svg {
            width: 100%;
            height: 500px;
        }

        .node-label {
            font-family: var(--font-mono);
            font-size: 9px;
            fill: var(--color-primary);
        }

        /* Cross-Cutting Insights */
        .insight-section {
            background: white;
            border-radius: 12px;
            padding: 1.5rem;
            margin-bottom: 1.5rem;
        }

        .insight-section h2 {
            font-family: var(--font-display);
            font-size: 1.25rem;
            color: var(--color-primary);
            margin-bottom: 1rem;
        }

        .insight-item {
            border-left: 4px solid var(--color-secondary);
            padding: 1rem;
            padding-left: 1.25rem;
            margin-bottom: 1rem;
            background: #f8fafc;
            border-radius: 0 8px 8px 0;
        }

        .insight-item h3 {
            font-weight: 600;
            color: var(--color-primary);
            margin-bottom: 0.5rem;
        }

        .insight-item p {
            font-size: 0.9rem;
            color: var(--color-neutral);
            line-height: 1.7;
            margin-bottom: 0.5rem;
        }

        .insight-item p:last-child {
            margin-bottom: 0;
        }

        /* Search Input */
        .search-input {
            padding: 0.625rem 1rem;
            border: 1px solid #e5e7eb;
            border-radius: 8px;
            font-size: 0.875rem;
            width: 200px;
        }

        .search-input:focus {
            outline: none;
            border-color: var(--color-accent);
            box-shadow: 0 0 0 3px rgba(96, 150, 186, 0.2);
        }

        /* Empty State */
        .empty-state {
            text-align: center;
            padding: 2rem;
            color: var(--color-neutral);
            font-style: italic;
        }

        /* Hidden */
        .hidden {
            display: none !important;
        }

        /* Code Detail Page */
        .code-detail-header {
            background: white;
            border-radius: 12px;
            padding: 1.5rem;
            margin-bottom: 1.5rem;
        }

        .code-detail-header h2 {
            font-family: var(--font-mono);
            font-size: 1.25rem;
            color: var(--color-primary);
            margin-bottom: 0.75rem;
            word-break: break-all;
        }

        .code-stats-row {
            display: flex;
            gap: 2rem;
            margin-top: 1rem;
        }

        .code-stat-item {
            display: flex;
            flex-direction: column;
            gap: 0.25rem;
        }

        .code-stat-item .value {
            font-size: 1.75rem;
            font-weight: 700;
            color: var(--color-primary);
        }

        .code-stat-item .label {
            font-size: 0.75rem;
            color: var(--color-neutral);
            text-transform: uppercase;
            letter-spacing: 0.05em;
        }

        .theme-chip {
            display: inline-block;
            padding: 0.5rem 0.75rem;
            background: #f0f7fc;
            border-radius: 8px;
            margin: 0.25rem;
            font-size: 0.85rem;
            color: var(--color-primary);
            cursor: pointer;
            transition: background 0.2s;
            border: 1px solid #e1edf5;
        }

        .theme-chip:hover {
            background: #e1edf5;
        }

        .cooccurrence-item {
            display: flex;
            justify-content: space-between;
            align-items: center;
            padding: 0.75rem 0;
            border-bottom: 1px solid #f3f4f6;
        }

        .cooccurrence-item:last-child {
            border-bottom: none;
        }

        .cooccurrence-code {
            font-family: var(--font-mono);
            font-size: 0.85rem;
            cursor: pointer;
            color: var(--color-primary);
        }

        .cooccurrence-code:hover {
            text-decoration: underline;
        }

        .cooccurrence-weight {
            background: #e7ecef;
            padding: 0.25rem 0.5rem;
            border-radius: 4px;
            font-size: 0.75rem;
            font-weight: 600;
            color: var(--color-primary);
        }

        .code-table tr.clickable {
            cursor: pointer;
        }

        .code-table tr.clickable:hover {
            background: #f0f7fc;
        }

        .code-table tr.clickable .row-arrow {
            color: #8B8C89;
            transition: color 0.2s;
        }

        .code-table tr.clickable:hover .row-arrow {
            color: var(--color-primary);
        }

        /* Thematic Analysis Page */
        .theme-analysis-card {
            background: white;
            border-radius: 12px;
            padding: 1.5rem;
            margin-bottom: 1rem;
            cursor: pointer;
            transition: box-shadow 0.2s, transform 0.2s;
            display: flex;
            gap: 1rem;
            border: 1px solid #e5e7eb;
        }

        .theme-analysis-card:hover {
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);
            transform: translateY(-2px);
        }

        .theme-analysis-number {
            width: 48px;
            height: 48px;
            background: var(--color-primary);
            color: white;
            border-radius: 10px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 700;
            font-size: 1.25rem;
            flex-shrink: 0;
        }

        .theme-analysis-content {
            flex: 1;
            min-width: 0;
        }

        .theme-analysis-content h3 {
            font-family: var(--font-display);
            font-size: 1rem;
            color: var(--color-primary);
            margin-bottom: 0.5rem;
            text-transform: uppercase;
            letter-spacing: 0.02em;
        }

        .theme-analysis-content .core-concept {
            font-size: 0.9rem;
            color: #4a4a4a;
            line-height: 1.6;
            margin-bottom: 0.75rem;
        }

        .theme-analysis-content .key-finding {
            font-size: 0.85rem;
            color: var(--color-primary);
            font-style: italic;
            background: #f0f7fc;
            padding: 0.75rem;
            border-radius: 8px;
            margin-bottom: 0.75rem;
            border-left: 3px solid var(--color-accent);
        }

        .sub-theme-tags {
            display: flex;
            flex-wrap: wrap;
            gap: 0.5rem;
            margin-bottom: 0.75rem;
        }

        .sub-theme-tag {
            background: #e7ecef;
            color: var(--color-primary);
            padding: 0.25rem 0.5rem;
            border-radius: 4px;
            font-size: 0.75rem;
            font-weight: 500;
        }

        .theme-analysis-meta {
            display: flex;
            align-items: center;
            gap: 1rem;
            flex-wrap: wrap;
        }

        .theme-analysis-meta span {
            display: flex;
            align-items: center;
            gap: 0.25rem;
            font-size: 0.8rem;
            color: var(--color-neutral);
        }

        .supporting-codes-row {
            display: flex;
            flex-wrap: wrap;
            gap: 0.35rem;
            margin-top: 0.75rem;
        }

        .code-chip {
            font-family: var(--font-mono);
            font-size: 0.7rem;
            padding: 0.2rem 0.4rem;
            border-radius: 4px;
            cursor: pointer;
            transition: opacity 0.2s;
        }

        .code-chip:hover {
            opacity: 0.8;
        }

        .code-chip.deductive {
            background: rgba(39, 76, 119, 0.1);
            color: #274C77;
        }

        .code-chip.inductive {
            background: rgba(96, 150, 186, 0.15);
            color: #3d6a8a;
        }

        .theme-analysis-arrow {
            display: flex;
            align-items: center;
            color: #8B8C89;
            flex-shrink: 0;
        }

        .theme-analysis-card:hover .theme-analysis-arrow {
            color: var(--color-primary);
        }

        .evidence-summary {
            display: flex;
            gap: 1rem;
            margin-bottom: 1.5rem;
        }

        .evidence-summary-item {
            display: flex;
            align-items: center;
            gap: 0.5rem;
            font-size: 0.85rem;
        }

        .evidence-dot {
            width: 12px;
            height: 12px;
            border-radius: 50%;
        }

        /* Scrollbar */
        ::-webkit-scrollbar {
            width: 8px;
            height: 8px;
        }

        ::-webkit-scrollbar-track {
            background: #f1f1f1;
        }

        ::-webkit-scrollbar-thumb {
            background: #c1c1c1;
            border-radius: 4px;
        }

        ::-webkit-scrollbar-thumb:hover {
            background: #a1a1a1;
        }

        /* Responsive */
        @media (max-width: 768px) {
            .sidebar {
                width: 100%;
                height: auto;
                position: relative;
            }

            .main-content {
                margin-left: 0;
            }

            .app-container {
                flex-direction: column;
            }

            .code-bar .name {
                width: 120px;
            }
        }
    </style>
</head>
<body>
    <div class="app-container">
        <!-- Sidebar -->
        <aside class="sidebar">
            <div class="sidebar-brand">
                <h1>📊 Qualitative Insight</h1>
                <p>AI Anthropology Toolkit</p>
            </div>

            <nav>
                <div class="nav-item active" data-page="dashboard">
                    <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M3 9l9-7 9 7v11a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2z"/><polyline points="9,22 9,12 15,12 15,22"/></svg>
                    Dashboard
                </div>
                <div class="nav-item" data-page="codes">
                    <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M18 20V10M12 20V4M6 20v-6"/></svg>
                    Coding
                </div>
                <div class="nav-item" data-page="themes">
                    <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M19.428 15.428a2 2 0 00-1.022-.547l-2.387-.477a6 6 0 00-3.86.517l-.318.158a6 6 0 01-3.86.517L6.05 15.21a2 2 0 00-1.806.547M8 4h8l-1 1v5.172a2 2 0 00.586 1.414l5 5c1.26 1.26.367 3.414-1.415 3.414H4.828c-1.782 0-2.674-2.154-1.414-3.414l5-5A2 2 0 009 10.172V5L8 4z"/></svg>
                    Thematic Analysis
                </div>
                <div class="nav-item" data-page="network">
                    <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><circle cx="12" cy="12" r="3"/><path d="M12 2v4m0 12v4m-7-10H1m22 0h-4m-2.3-5.7l2.8-2.8m-12 0l2.8 2.8m0 11.4l-2.8 2.8m12 0l-2.8-2.8"/></svg>
                    Concept Network
                </div>
                <div class="nav-item" data-page="insights">
                    <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M12 2v20m0-20c-4.4 0-8 3.6-8 8 0 2.3 1 4.4 2.5 5.9L12 22l5.5-6.1C19 14.4 20 12.3 20 10c0-4.4-3.6-8-8-8z"/></svg>
                    Cross-Cutting Insights
                </div>
            </nav>
        </aside>

        <!-- Main Content -->
        <main class="main-content">
            <div class="content-wrapper">
                <!-- Pages will be rendered here -->
                <div id="page-content"></div>
            </div>
        </main>
    </div>

    <script>
    // ============================================================
    // EMBEDDED DATA - Replace this with your JSON export
    // ============================================================
'''

        # HTML template - end (after data injection point)
        html_end = r'''
    // ============================================================
    // APP STATE
    // ============================================================
    let currentPage = 'dashboard';
    let selectedTheme = null;
    let selectedCode = null;
    let activeTab = 'overview';

    // ============================================================
    // RENDER FUNCTIONS
    // ============================================================

    function escapeHtml(text) {
        if (text === null || text === undefined) return '';
        return String(text)
            .replace(/&/g, '&amp;')
            .replace(/</g, '&lt;')
            .replace(/>/g, '&gt;')
            .replace(/"/g, '&quot;')
            .replace(/'/g, '&#39;');
    }

    function renderDashboard() {
        const { metadata, themes, executive_summary } = analysisData;

        return `
            <div class="page-header">
                <h1>Qualitative Analysis Dashboard</h1>
                <p>AI-assisted thematic analysis of ${metadata.total_chunks} data chunks using ${metadata.coding_approach} coding approach.</p>
            </div>

            <div class="stats-grid">
                <div class="stat-card">
                    <div class="stat-icon" style="background: #274C77">
                        <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9 12h6m-6 4h6m2 5H7a2 2 0 01-2-2V5a2 2 0 012-2h5.586a1 1 0 01.707.293l5.414 5.414a1 1 0 01.293.707V19a2 2 0 01-2 2z"/></svg>
                    </div>
                    <div class="stat-content">
                        <h4>Data Chunks</h4>
                        <div class="value">${metadata.total_chunks}</div>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon" style="background: #4a6fa5">
                        <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M7 7h.01M7 3h5c.512 0 1.024.195 1.414.586l7 7a2 2 0 010 2.828l-7 7a2 2 0 01-2.828 0l-7-7A1.994 1.994 0 013 12V7a4 4 0 014-4z"/></svg>
                    </div>
                    <div class="stat-content">
                        <h4>Unique Codes</h4>
                        <div class="value">${metadata.unique_codes}</div>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon" style="background: #6096BA">
                        <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M19.428 15.428a2 2 0 00-1.022-.547l-2.387-.477a6 6 0 00-3.86.517l-.318.158a6 6 0 01-3.86.517L6.05 15.21a2 2 0 00-1.806.547M8 4h8l-1 1v5.172a2 2 0 00.586 1.414l5 5c1.26 1.26.367 3.414-1.415 3.414H4.828c-1.782 0-2.674-2.154-1.414-3.414l5-5A2 2 0 009 10.172V5L8 4z"/></svg>
                    </div>
                    <div class="stat-content">
                        <h4>Themes Identified</h4>
                        <div class="value">${themes.length}</div>
                    </div>
                </div>
            </div>

            <div style="margin-bottom: 1.5rem;">
                <h2 style="font-family: var(--font-display); font-size: 1.25rem; color: var(--color-primary); margin-bottom: 1rem;">Themes</h2>
                <div class="themes-grid">
                    ${themes.map(theme => `
                        <div class="theme-card" onclick="openTheme(${theme.theme_id})">
                            <div class="theme-number">${theme.theme_id}</div>
                            <div class="theme-content">
                                <h4>${escapeHtml(theme.theme_name.replace(/^Theme \\d+:\\s*/, ''))}</h4>
                                <p>${escapeHtml(theme.initial_analysis.core_concept)}</p>
                                <div class="theme-meta">
                                    <span>
                                        <svg width="14" height="14" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9 12h6m-6 4h6m2 5H7a2 2 0 01-2-2V5a2 2 0 012-2h5.586a1 1 0 01.707.293l5.414 5.414a1 1 0 01.293.707V19a2 2 0 01-2 2z"/></svg>
                                        ${theme.chunk_count} chunks
                                    </span>
                                    <span>
                                        <svg width="14" height="14" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M7 7h.01M7 3h5c.512 0 1.024.195 1.414.586l7 7a2 2 0 010 2.828l-7 7a2 2 0 01-2.828 0l-7-7A1.994 1.994 0 013 12V7a4 4 0 014-4z"/></svg>
                                        ${theme.supporting_codes.length} codes
                                    </span>
                                    <span class="evidence-badge evidence-${theme.initial_analysis.evidence_strength.toLowerCase()}">${theme.initial_analysis.evidence_strength} Evidence</span>
                                </div>
                            </div>
                            <div class="theme-arrow">
                                <svg width="20" height="20" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><polyline points="9 18 15 12 9 6"/></svg>
                            </div>
                        </div>
                    `).join('')}
                </div>
            </div>
        `;
    }

    function renderThematicAnalysis() {
        const { themes } = analysisData;

        // Calculate evidence distribution
        const evidenceCounts = { strong: 0, moderate: 0, emerging: 0 };
        themes.forEach(t => {
            const strength = t.initial_analysis.evidence_strength.toLowerCase();
            if (evidenceCounts[strength] !== undefined) evidenceCounts[strength]++;
        });

        return `
            <div class="page-header">
                <h1>Thematic Analysis</h1>
                <p>Detailed view of ${themes.length} themes identified through ${analysisData.metadata.coding_approach} coding approach.</p>
            </div>

            <div class="stats-grid" style="margin-bottom: 1.5rem;">
                <div class="stat-card">
                    <div class="stat-icon" style="background: #274C77">
                        <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9 12l2 2 4-4m6 2a9 9 0 11-18 0 9 9 0 0118 0z"/></svg>
                    </div>
                    <div class="stat-content">
                        <h4>Strong Evidence</h4>
                        <div class="value">${evidenceCounts.strong}</div>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon" style="background: #3d5a80">
                        <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9 12l2 2 4-4m6 2a9 9 0 11-18 0 9 9 0 0118 0z"/></svg>
                    </div>
                    <div class="stat-content">
                        <h4>Moderate Evidence</h4>
                        <div class="value">${evidenceCounts.moderate}</div>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon" style="background: #6096BA">
                        <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9 12l2 2 4-4m6 2a9 9 0 11-18 0 9 9 0 0118 0z"/></svg>
                    </div>
                    <div class="stat-content">
                        <h4>Emerging Evidence</h4>
                        <div class="value">${evidenceCounts.emerging}</div>
                    </div>
                </div>
            </div>

            <div>
                ${themes.map(theme => {
                    const ia = theme.initial_analysis;
                    const da = theme.deep_analysis;
                    const themeName = theme.theme_name.replace(/^Theme \\d+:\\s*/, '');

                    return `
                        <div class="theme-analysis-card" onclick="openTheme(${theme.theme_id})">
                            <div class="theme-analysis-number">${theme.theme_id}</div>
                            <div class="theme-analysis-content">
                                <h3>${escapeHtml(themeName)}</h3>
                                <p class="core-concept">${escapeHtml(ia.core_concept)}</p>
                                <div class="key-finding">
                                    <strong>Key Finding:</strong> ${escapeHtml(ia.key_finding)}
                                </div>
                                <div class="sub-theme-tags">
                                    ${ia.sub_themes.map(sub => `
                                        <span class="sub-theme-tag">${escapeHtml(sub.name)}</span>
                                    `).join('')}
                                </div>
                                <div class="theme-analysis-meta">
                                    <span>
                                        <svg width="14" height="14" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9 12h6m-6 4h6m2 5H7a2 2 0 01-2-2V5a2 2 0 012-2h5.586a1 1 0 01.707.293l5.414 5.414a1 1 0 01.293.707V19a2 2 0 01-2 2z"/></svg>
                                        ${theme.chunk_count} chunks
                                    </span>
                                    <span>
                                        <svg width="14" height="14" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M7 7h.01M7 3h5c.512 0 1.024.195 1.414.586l7 7a2 2 0 010 2.828l-7 7a2 2 0 01-2.828 0l-7-7A1.994 1.994 0 013 12V7a4 4 0 014-4z"/></svg>
                                        ${theme.supporting_codes.length} codes
                                    </span>
                                    <span class="evidence-badge evidence-${ia.evidence_strength.toLowerCase()}">${ia.evidence_strength} Evidence</span>
                                </div>
                                <div class="supporting-codes-row">
                                    ${theme.supporting_codes.map(code => {
                                        const codeType = analysisData.codes.frequencies[code]?.type || 'deductive';
                                        return `<span class="code-chip ${codeType}" onclick="event.stopPropagation(); openCode('${code}')">${code}</span>`;
                                    }).join('')}
                                </div>
                            </div>
                            <div class="theme-analysis-arrow">
                                <svg width="20" height="20" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><polyline points="9 18 15 12 9 6"/></svg>
                            </div>
                        </div>
                    `;
                }).join('')}
            </div>
        `;
    }

    function renderThemeDetail() {
        const theme = analysisData.themes.find(t => t.theme_id === selectedTheme);
        if (!theme) return '<p>Theme not found</p>';

        const da = theme.deep_analysis || {};
        const ia = theme.initial_analysis;

        return `
            <button class="back-button" onclick="backToThemes()">
                <svg width="16" height="16" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M19 12H5M12 19l-7-7 7-7"/></svg>
                Back to Thematic Analysis
            </button>

            <div class="theme-detail-header">
                <h2>${escapeHtml(theme.theme_name)}</h2>
                <p style="color: var(--color-neutral); margin-bottom: 1rem;">${escapeHtml(da.overview || ia.core_concept || '')}</p>
                <div>
                    <span class="evidence-badge evidence-${ia.evidence_strength.toLowerCase()}">${ia.evidence_strength} Evidence</span>
                    <span style="font-size: 0.85rem; color: var(--color-neutral);">${theme.chunk_count} data chunks · ${theme.supporting_codes.length} supporting codes</span>
                </div>
            </div>

            <div class="tabs">
                <button class="tab ${activeTab === 'overview' ? 'active' : ''}" onclick="setTab('overview')">Overview</button>
                <button class="tab ${activeTab === 'voices' ? 'active' : ''}" onclick="setTab('voices')">Participant Perspective</button>
                <button class="tab ${activeTab === 'stakeholders' ? 'active' : ''}" onclick="setTab('stakeholders')">Stakeholders</button>
                <button class="tab ${activeTab === 'implications' ? 'active' : ''}" onclick="setTab('implications')">Implications</button>
            </div>

            <div id="tab-content">
                ${renderTabContent(theme)}
            </div>
        `;
    }

    function renderTabContent(theme) {
        const da = theme.deep_analysis || {};
        const ia = theme.initial_analysis;

        if (activeTab === 'overview') {
            return `
                <!-- Sub-themes -->
                <div class="card" style="margin-bottom: 1.5rem;">
                    <h3>Sub-Themes</h3>
                    ${ia.sub_themes.map((sub, i) => `
                        <div class="sub-theme">
                            <span class="sub-theme-letter">${String.fromCharCode(97 + i)})</span>
                            <div>
                                <h4>${escapeHtml(sub.name)}</h4>
                                <p>${escapeHtml(sub.description)}</p>
                            </div>
                        </div>
                    `).join('')}
                </div>

                <!-- Manifestations -->
                ${da.manifestations && da.manifestations.length > 0 ? `
                    <div style="margin-bottom: 1.5rem;">
                        <h3 style="font-family: var(--font-display); font-size: 1.1rem; color: var(--color-primary); margin-bottom: 1rem;">Key Manifestations</h3>
                        <div class="section-grid section-grid-2">
                            ${da.manifestations.map(m => `
                                <div class="manifestation">
                                    <h4>${escapeHtml(m.pattern)}</h4>
                                    <p>${escapeHtml(m.description)}</p>
                                </div>
                            `).join('')}
                        </div>
                    </div>
                ` : ''}

                <!-- Analytical Interpretation (Etic) -->
                ${da.analytical_interpretation ? `
                    <div class="interpretation-section etic">
                        <h3>
                            <svg width="20" height="20" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9.663 17h4.673M12 3v1m6.364 1.636l-.707.707M21 12h-1M4 12H3m3.343-5.657l-.707-.707m2.828 9.9a5 5 0 117.072 0l-.548.547A3.374 3.374 0 0014 18.469V19a2 2 0 11-4 0v-.531c0-.895-.356-1.754-.988-2.386l-.548-.547z"/></svg>
                            Analytical Interpretation
                            <span class="label">Etic</span>
                        </h3>
                        ${da.analytical_interpretation.core_concept ? `
                            <div class="interpretation-item">
                                <strong>Core Concept</strong>
                                <p>${escapeHtml(da.analytical_interpretation.core_concept)}</p>
                            </div>
                        ` : ''}
                        ${da.analytical_interpretation.key_finding ? `
                            <div class="interpretation-item">
                                <strong>Key Finding</strong>
                                <p>${escapeHtml(da.analytical_interpretation.key_finding)}</p>
                            </div>
                        ` : ''}
                    </div>
                ` : ''}

                <!-- Participant Perspective (Emic) -->
                ${da.participant_perspective ? `
                    <div class="interpretation-section emic">
                        <h3>
                            <svg width="20" height="20" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M17 20h5v-2a3 3 0 00-5.356-1.857M17 20H7m10 0v-2c0-.656-.126-1.283-.356-1.857M7 20H2v-2a3 3 0 015.356-1.857M7 20v-2c0-.656.126-1.283.356-1.857m0 0a5.002 5.002 0 019.288 0M15 7a3 3 0 11-6 0 3 3 0 016 0zm6 3a2 2 0 11-4 0 2 2 0 014 0zM7 10a2 2 0 11-4 0 2 2 0 014 0z"/></svg>
                            Participant Perspective
                            <span class="label">Emic</span>
                        </h3>
                        <div style="font-style: italic; color: #6b7280; line-height: 1.7;">${mdToHtml(da.participant_perspective || "")}</div>
                    </div>
                ` : ''}
            `;
        }

        if (activeTab === 'voices') {
            return `
                <!-- Participant Voices -->
                <div class="card" style="margin-bottom: 1.5rem;">
                    <h3 style="display: flex; align-items: center; gap: 0.5rem;">
                        <svg width="20" height="20" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M17 20h5v-2a3 3 0 00-5.356-1.857M17 20H7m10 0v-2c0-.656-.126-1.283-.356-1.857M7 20H2v-2a3 3 0 015.356-1.857M7 20v-2c0-.656.126-1.283.356-1.857m0 0a5.002 5.002 0 019.288 0M15 7a3 3 0 11-6 0 3 3 0 016 0zm6 3a2 2 0 11-4 0 2 2 0 014 0zM7 10a2 2 0 11-4 0 2 2 0 014 0z"/></svg>
                        Participant Perspective
                    </h3>
                    ${da.participant_voices && da.participant_voices.length > 0 ?
                        da.participant_voices.map(voice => `
                            <div class="quote-card">
                                <blockquote>"${escapeHtml(voice.quote)}"</blockquote>
                                <cite>— ${escapeHtml(voice.record_id)}</cite>
                            </div>
                        `).join('') :
                        (da.participant_perspective ?
                            '<div style="font-style: italic; color: #4a4a4a; line-height: 1.7;">' + mdToHtml(da.participant_perspective || '') + '</div>' :
                            '<p class="empty-state">No participant perspective data available for this theme.</p>'
                        )
                    }
                </div>

                <!-- Tensions & Contradictions -->
                ${da.tensions_contradictions ? `
                    <div class="tensions-card">
                        <h3>
                            <svg width="20" height="20" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M12 9v2m0 4h.01m-6.938 4h13.856c1.54 0 2.502-1.667 1.732-3L13.732 4c-.77-1.333-2.694-1.333-3.464 0L3.34 16c-.77 1.333.192 3 1.732 3z"/></svg>
                            Tensions & Contradictions
                        </h3>
                        <div style="line-height: 1.7;">${mdToHtml(da.tensions_contradictions || "")}</div>
                    </div>
                ` : ''}
            `;
        }

        if (activeTab === 'stakeholders') {
            return `
                <h3 style="font-family: var(--font-display); font-size: 1.1rem; color: var(--color-primary); margin-bottom: 0.5rem;">Stakeholder Perspectives</h3>
                <p style="color: var(--color-neutral); margin-bottom: 1.5rem;">How different groups experience and articulate this theme.</p>

                ${da.stakeholder_perspectives && da.stakeholder_perspectives.length > 0 ? `
                    <div class="section-grid section-grid-2">
                        ${da.stakeholder_perspectives.map(sp => `
                            <div class="stakeholder-card">
                                <h4>
                                    <svg width="18" height="18" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M16 7a4 4 0 11-8 0 4 4 0 018 0zM12 14a7 7 0 00-7 7h14a7 7 0 00-7-7z"/></svg>
                                    ${escapeHtml(sp.group)}
                                </h4>
                                <p>${escapeHtml(sp.perspective)}</p>
                            </div>
                        `).join('')}
                    </div>
                ` : '<p class="empty-state">No stakeholder perspectives documented for this theme.</p>'}
            `;
        }

        if (activeTab === 'implications') {
            return `
                <!-- Practical Implications -->
                <div class="card" style="margin-bottom: 1.5rem;">
                    <h3 style="display: flex; align-items: center; gap: 0.5rem;">
                        <svg width="20" height="20" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9.663 17h4.673M12 3v1m6.364 1.636l-.707.707M21 12h-1M4 12H3m3.343-5.657l-.707-.707m2.828 9.9a5 5 0 117.072 0l-.548.547A3.374 3.374 0 0014 18.469V19a2 2 0 11-4 0v-.531c0-.895-.356-1.754-.988-2.386l-.548-.547z"/></svg>
                        Practical Implications
                    </h3>
                    ${da.practical_implications && da.practical_implications.length > 0 ?
                        da.practical_implications.map(impl => `
                            <div style="margin-bottom: 0.75rem; padding-left: 1rem; border-left: 3px solid var(--color-secondary);">${mdToHtml(impl)}</div>
                        `).join('') :
                        '<p class="empty-state">No practical implications documented.</p>'
                    }
                </div>

                <!-- Needs, Barriers, Workarounds -->
                <div class="section-grid section-grid-3" style="margin-bottom: 1.5rem;">
                    <div class="nbw-card needs">
                        <h4>Expressed Needs</h4>
                        <ul style="list-style: none;">
                            ${(da.needs_barriers_workarounds?.needs || []).map(n => `<li>• ${escapeHtml(n)}</li>`).join('') || '<li>None documented</li>'}
                        </ul>
                    </div>
                    <div class="nbw-card barriers">
                        <h4>Barriers</h4>
                        <ul style="list-style: none;">
                            ${(da.needs_barriers_workarounds?.barriers || []).map(b => `<li>• ${escapeHtml(b)}</li>`).join('') || '<li>None documented</li>'}
                        </ul>
                    </div>
                    <div class="nbw-card workarounds">
                        <h4>Workarounds</h4>
                        <ul style="list-style: none;">
                            ${(da.needs_barriers_workarounds?.workarounds || []).map(w => `<li>• ${escapeHtml(w)}</li>`).join('') || '<li>None documented</li>'}
                        </ul>
                    </div>
                </div>

                <!-- Open Questions -->
                ${da.open_questions && da.open_questions.length > 0 ? `
                    <div class="card">
                        <h3>Open Questions for Further Exploration</h3>
                        ${da.open_questions.map(q => `
                            <div class="open-question">${escapeHtml(q)}</div>
                        `).join('')}
                    </div>
                ` : ''}
            `;
        }

        return '';
    }

    function renderCodeAnalytics() {
        const { codes, metadata } = analysisData;
        const codeData = Object.entries(codes.frequencies)
            .map(([name, data]) => ({ name, ...data }))
            .sort((a, b) => b.frequency - a.frequency);

        const maxFreq = Math.max(...codeData.map(c => c.frequency));
        const deductiveCount = codeData.filter(c => c.type === 'deductive').length;
        const inductiveCount = codeData.filter(c => c.type === 'inductive').length;

        return `
            <div class="page-header">
                <h1>Coding</h1>
                <p>Frequency distribution of inductive and deductive codes applied during analysis.</p>
            </div>

            <div class="stats-grid">
                <div class="stat-card">
                    <div class="stat-icon" style="background: #274C77">
                        <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9 12h6m-6 4h6m2 5H7a2 2 0 01-2-2V5a2 2 0 012-2h5.586a1 1 0 01.707.293l5.414 5.414a1 1 0 01.293.707V19a2 2 0 01-2 2z"/></svg>
                    </div>
                    <div class="stat-content">
                        <h4>Total Codes</h4>
                        <div class="value">${metadata.unique_codes}</div>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon" style="background: #4a6fa5">
                        <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M19.428 15.428a2 2 0 00-1.022-.547l-2.387-.477a6 6 0 00-3.86.517l-.318.158a6 6 0 01-3.86.517L6.05 15.21a2 2 0 00-1.806.547M8 4h8l-1 1v5.172a2 2 0 00.586 1.414l5 5c1.26 1.26.367 3.414-1.415 3.414H4.828c-1.782 0-2.674-2.154-1.414-3.414l5-5A2 2 0 009 10.172V5L8 4z"/></svg>
                    </div>
                    <div class="stat-content">
                        <h4>Deductive Codes</h4>
                        <div class="value">${deductiveCount}</div>
                    </div>
                </div>
                <div class="stat-card">
                    <div class="stat-icon" style="background: #6096BA">
                        <svg fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M9.663 17h4.673M12 3v1m6.364 1.636l-.707.707M21 12h-1M4 12H3m3.343-5.657l-.707-.707m2.828 9.9a5 5 0 117.072 0l-.548.547A3.374 3.374 0 0014 18.469V19a2 2 0 11-4 0v-.531c0-.895-.356-1.754-.988-2.386l-.548-.547z"/></svg>
                    </div>
                    <div class="stat-content">
                        <h4>Inductive Codes</h4>
                        <div class="value">${inductiveCount}</div>
                    </div>
                </div>
            </div>

            <div class="chart-container">
                <h3 style="font-family: var(--font-display); font-size: 1.1rem; color: var(--color-primary); margin-bottom: 1.5rem;">Top 15 Applied Codes</h3>
                ${codeData.slice(0, 15).map(code => `
                    <div class="code-bar">
                        <div class="name" title="${code.name}" style="cursor: pointer;" onclick="openCode('${code.name}')">${code.name.length > 28 ? code.name.substring(0, 28) + '...' : code.name}</div>
                        <div class="bar-container">
                            <div class="bar" style="width: ${(code.frequency / maxFreq) * 100}%; background: ${code.type === 'deductive' ? 'var(--color-deductive)' : 'var(--color-inductive)'}"></div>
                        </div>
                        <div class="freq">${code.frequency}</div>
                    </div>
                `).join('')}
                <div class="legend">
                    <div class="legend-item">
                        <div class="legend-dot" style="background: var(--color-deductive)"></div>
                        Deductive (Theory-driven)
                    </div>
                    <div class="legend-item">
                        <div class="legend-dot" style="background: var(--color-inductive)"></div>
                        Inductive (Data-driven)
                    </div>
                </div>
            </div>

            <div class="card">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 1rem;">
                    <h3>Code Matrix</h3>
                    <input type="text" class="search-input" placeholder="Search codes..." oninput="filterCodes(this.value)">
                </div>
                <table class="code-table" id="code-table">
                    <thead>
                        <tr>
                            <th>Code Name</th>
                            <th style="text-align: center;">Type</th>
                            <th style="text-align: center;">Frequency</th>
                            <th style="width: 40px;"></th>
                        </tr>
                    </thead>
                    <tbody>
                        ${codeData.map(code => `
                            <tr data-name="${code.name.toLowerCase()}" class="clickable" onclick="openCode('${code.name}')">
                                <td class="code-name">${code.name}</td>
                                <td style="text-align: center;"><span class="type-badge ${code.type}">${code.type}</span></td>
                                <td style="text-align: center; font-weight: 600;">${code.frequency}</td>
                                <td style="text-align: center;" class="row-arrow"><svg width="16" height="16" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><polyline points="9 18 15 12 9 6"/></svg></td>
                            </tr>
                        `).join('')}
                    </tbody>
                </table>
            </div>
        `;
    }

    function renderNetwork() {
        return `
            <div class="page-header">
                <h1>Concept Network</h1>
                <p>Co-occurrence relationships between codes. Thicker lines indicate stronger associations. Drag nodes to rearrange.</p>
            </div>

            <div id="network-container">
                <canvas id="network-canvas" style="width: 100%; height: 500px; cursor: grab;"></canvas>
            </div>

            <div class="legend" style="margin-top: 1rem; margin-bottom: 1.5rem;">
                <div class="legend-item">
                    <div class="legend-dot" style="background: var(--color-deductive)"></div>
                    Deductive (Theory-driven)
                </div>
                <div class="legend-item">
                    <div class="legend-dot" style="background: var(--color-inductive)"></div>
                    Inductive (Data-driven)
                </div>
            </div>

            <div class="card" style="margin-top: 1rem;">
                <h3>Top Code Co-occurrences</h3>
                <table class="code-table">
                    <thead>
                        <tr>
                            <th>Source Code</th>
                            <th>Target Code</th>
                            <th style="text-align: center;">Weight</th>
                        </tr>
                    </thead>
                    <tbody>
                        ${analysisData.codes.edges.slice(0, 10).map(edge => {
                            const sourceType = analysisData.codes.frequencies[edge.source]?.type || 'deductive';
                            const targetType = analysisData.codes.frequencies[edge.target]?.type || 'deductive';
                            const sourceColor = sourceType === 'deductive' ? '#274C77' : '#6096BA';
                            const targetColor = targetType === 'deductive' ? '#274C77' : '#6096BA';
                            return `
                            <tr>
                                <td class="code-name" style="color: ${sourceColor}; font-weight: 500;">${edge.source}</td>
                                <td class="code-name" style="color: ${targetColor}; font-weight: 500;">${edge.target}</td>
                                <td style="text-align: center; font-weight: 600;">${edge.weight}</td>
                            </tr>
                        `}).join('')}
                    </tbody>
                </table>
            </div>
        `;
    }

    function mdToHtml(text) {
        if (!text) return '';
        // Convert bold and italic
        let processed = escapeHtml(text)
            .replace(/\*\*([^*]+)\*\*/g, '<strong>$1</strong>')
            .replace(/(?<!\*)\*([^*]+)\*(?!\*)/g, '<em>$1</em>');
        // Split into paragraphs on double newlines
        return processed
            .split(/\n{2,}/)
            .map(p => p.trim())
            .filter(p => p)
            .map(p => {
                // Check if this paragraph is a bullet list
                const lines = p.split('\n');
                const isList = lines.every(l => /^[-*]\s/.test(l.trim()));
                if (isList) {
                    return '<ul style="margin: 0.5rem 0; padding-left: 1.5rem;">' +
                        lines.map(l => '<li style="margin-bottom: 0.25rem;">' + l.replace(/^[-*]\s+/, '') + '</li>').join('') +
                        '</ul>';
                }
                // Check if some lines are bullets (mixed paragraph + list)
                const hasListItems = lines.some(l => /^[-*]\s/.test(l.trim()));
                if (hasListItems) {
                    let html = '';
                    let listItems = [];
                    lines.forEach(l => {
                        if (/^[-*]\s/.test(l.trim())) {
                            listItems.push(l.replace(/^[-*]\s+/, '').trim());
                        } else {
                            if (listItems.length > 0) {
                                html += '<ul style="margin: 0.5rem 0; padding-left: 1.5rem;">' +
                                    listItems.map(li => '<li style="margin-bottom: 0.25rem;">' + li + '</li>').join('') + '</ul>';
                                listItems = [];
                            }
                            html += '<p>' + l + '</p>';
                        }
                    });
                    if (listItems.length > 0) {
                        html += '<ul style="margin: 0.5rem 0; padding-left: 1.5rem;">' +
                            listItems.map(li => '<li style="margin-bottom: 0.25rem;">' + li + '</li>').join('') + '</ul>';
                    }
                    return html;
                }
                return '<p>' + p.replace(/\n/g, '<br>') + '</p>';
            })
            .join('');
    }

    function renderInsights() {
        const insights = analysisData.executive_summary.cross_cutting_insights;
        if (!insights) return '<div class="page-header"><h1>Cross-Cutting Insights</h1><p>No cross-cutting insights available.</p></div>';

        // Remove the initial # header and split by ##
        const cleanedInsights = insights.replace(/^# (?!#)[^\n]+\n+/, '');
        const sections = cleanedInsights.split('## ').filter(s => s.trim());

        // Process each section, removing --- dividers
        const processedSections = sections.map(section => {
            return section.split('---')[0].trim();
        }).filter(s => s);

        let html = `
            <div class="page-header">
                <h1>Cross-Cutting Insights</h1>
                <p>Meta-level patterns and connections observed across all themes.</p>
            </div>
        `;

        processedSections.forEach(section => {
            const lines = section.split('\n');
            const title = lines[0].trim();
            const content = lines.slice(1).join('\n');

            // Parse bold items
            const items = content.split('**').filter((_, idx) => idx % 2 === 1);
            const descriptions = content.split('**').filter((_, idx) => idx % 2 === 0 && idx > 0);

            html += '<div class="insight-section">';
            html += '<h2>' + escapeHtml(title.replace(/\*\*/g, '')) + '</h2>';

            // If no bold items found, render the whole content as prose
            if (items.length === 0) {
                html += '<div class="insight-item">' + mdToHtml(content) + '</div>';
            } else {
                items.forEach((item, i) => {
                    html += '<div class="insight-item">';
                    html += '<h3>' + escapeHtml(item) + '</h3>';
                    html += mdToHtml((descriptions[i] || '').trim());
                    html += '</div>';
                });
            }

            html += '</div>';
        });

        return html;
    }

    // ============================================================
    // CANVAS-BASED NETWORK VISUALIZATION (No external dependencies)
    // ============================================================

    function drawNetwork() {
        const canvas = document.getElementById('network-canvas');
        if (!canvas) return;

        const container = document.getElementById('network-container');
        const rect = container.getBoundingClientRect();
        const width = rect.width - 32;
        const height = 600;

        // Set canvas size with device pixel ratio for sharp rendering
        const dpr = window.devicePixelRatio || 1;
        canvas.width = width * dpr;
        canvas.height = height * dpr;
        canvas.style.width = width + 'px';
        canvas.style.height = height + 'px';

        const ctx = canvas.getContext('2d');
        ctx.scale(dpr, dpr);

        const { codes } = analysisData;
        const edges = codes.edges;

        // Build nodes from edges
        const nodeMap = new Map();
        let nodeIndex = 0;
        const nodeCount = new Set([...edges.map(e => e.source), ...edges.map(e => e.target)]).size;

        edges.forEach(e => {
            if (!nodeMap.has(e.source)) {
                // Circular initial layout for stability
                const angle = (nodeIndex / nodeCount) * Math.PI * 2;
                const radius = Math.min(width, height) * 0.42;
                nodeMap.set(e.source, {
                    id: e.source,
                    frequency: codes.frequencies[e.source]?.frequency || 1,
                    type: codes.frequencies[e.source]?.type || 'deductive',
                    x: width / 2 + Math.cos(angle) * radius,
                    y: height / 2 + Math.sin(angle) * radius,
                    vx: 0,
                    vy: 0
                });
                nodeIndex++;
            }
            if (!nodeMap.has(e.target)) {
                const angle = (nodeIndex / nodeCount) * Math.PI * 2;
                const radius = Math.min(width, height) * 0.42;
                nodeMap.set(e.target, {
                    id: e.target,
                    frequency: codes.frequencies[e.target]?.frequency || 1,
                    type: codes.frequencies[e.target]?.type || 'deductive',
                    x: width / 2 + Math.cos(angle) * radius,
                    y: height / 2 + Math.sin(angle) * radius,
                    vx: 0,
                    vy: 0
                });
                nodeIndex++;
            }
        });

        const nodes = Array.from(nodeMap.values());
        const links = edges.map(e => ({
            source: nodeMap.get(e.source),
            target: nodeMap.get(e.target),
            weight: e.weight
        }));

        // Physics simulation parameters - tuned for smooth settling
        const repulsion = 8000;
        const attraction = 0.005;
        const damping = 0.92;
        const centerPull = 0.003;

        let animating = true;
        let iterations = 0;
        const maxIterations = 200;
        const warmupIterations = 100; // Pre-simulate before displaying

        // Dragging state
        let dragNode = null;
        let isDragging = false;

        function getNodeRadius(node) {
            return Math.sqrt(node.frequency) * 3 + 8;
        }

        function simulate() {
            // Apply forces
            nodes.forEach(node => {
                // Center pull
                node.vx += (width / 2 - node.x) * centerPull;
                node.vy += (height / 2 - node.y) * centerPull;

                // Repulsion from other nodes
                nodes.forEach(other => {
                    if (node === other) return;
                    const dx = node.x - other.x;
                    const dy = node.y - other.y;
                    const dist = Math.sqrt(dx * dx + dy * dy) || 1;
                    const force = repulsion / (dist * dist);
                    node.vx += (dx / dist) * force;
                    node.vy += (dy / dist) * force;
                });
            });

            // Attraction along links
            links.forEach(link => {
                const dx = link.target.x - link.source.x;
                const dy = link.target.y - link.source.y;
                const dist = Math.sqrt(dx * dx + dy * dy) || 1;
                const force = dist * attraction * (link.weight / 5);

                link.source.vx += (dx / dist) * force;
                link.source.vy += (dy / dist) * force;
                link.target.vx -= (dx / dist) * force;
                link.target.vy -= (dy / dist) * force;
            });

            // Update positions with adaptive damping (more damping over time)
            const adaptiveDamping = damping - (iterations / maxIterations) * 0.1;
            nodes.forEach(node => {
                if (node === dragNode) return;

                node.vx *= adaptiveDamping;
                node.vy *= adaptiveDamping;
                node.x += node.vx;
                node.y += node.vy;

                // Keep in bounds
                const r = getNodeRadius(node);
                node.x = Math.max(r + 10, Math.min(width - r - 10, node.x));
                node.y = Math.max(r + 10, Math.min(height - r - 10, node.y));
            });
        }

        function draw() {
            ctx.clearRect(0, 0, width, height);

            // Draw links
            links.forEach(link => {
                ctx.beginPath();
                ctx.moveTo(link.source.x, link.source.y);
                ctx.lineTo(link.target.x, link.target.y);
                ctx.strokeStyle = 'rgba(39, 76, 119, 0.25)';
                ctx.lineWidth = Math.sqrt(link.weight) * 0.8;
                ctx.stroke();
            });

            // Draw nodes
            nodes.forEach(node => {
                const r = getNodeRadius(node);

                // Node circle - blue palette
                ctx.beginPath();
                ctx.arc(node.x, node.y, r, 0, Math.PI * 2);
                ctx.fillStyle = node.type === 'deductive' ? '#274C77' : '#6096BA';
                ctx.fill();
                ctx.strokeStyle = '#fff';
                ctx.lineWidth = 2;
                ctx.stroke();

                // Label
                ctx.fillStyle = '#274C77';
                ctx.font = '9px "IBM Plex Mono", monospace';
                ctx.textAlign = 'center';
                const label = node.id.length > 18 ? node.id.substring(0, 18) + '...' : node.id;
                ctx.fillText(label, node.x, node.y + r + 12);
            });
        }

        // Pre-simulate (warmup) before displaying
        for (let i = 0; i < warmupIterations; i++) {
            simulate();
            iterations++;
        }

        function animate() {
            if (!animating) return;

            simulate();
            draw();
            iterations++;

            if (iterations < maxIterations) {
                requestAnimationFrame(animate);
            } else {
                draw(); // Final draw
            }
        }

        // Mouse interaction
        function getMousePos(e) {
            const rect = canvas.getBoundingClientRect();
            return {
                x: e.clientX - rect.left,
                y: e.clientY - rect.top
            };
        }

        function findNodeAt(pos) {
            for (let i = nodes.length - 1; i >= 0; i--) {
                const node = nodes[i];
                const r = getNodeRadius(node);
                const dx = pos.x - node.x;
                const dy = pos.y - node.y;
                if (dx * dx + dy * dy < r * r) {
                    return node;
                }
            }
            return null;
        }

        canvas.addEventListener('mousedown', (e) => {
            const pos = getMousePos(e);
            dragNode = findNodeAt(pos);
            if (dragNode) {
                isDragging = true;
                canvas.style.cursor = 'grabbing';
            }
        });

        canvas.addEventListener('mousemove', (e) => {
            const pos = getMousePos(e);

            if (isDragging && dragNode) {
                dragNode.x = pos.x;
                dragNode.y = pos.y;
                dragNode.vx = 0;
                dragNode.vy = 0;
                draw();
            } else {
                const hoverNode = findNodeAt(pos);
                canvas.style.cursor = hoverNode ? 'grab' : 'default';
                canvas.title = hoverNode ? `${hoverNode.id}\\nFrequency: ${hoverNode.frequency}\\nType: ${hoverNode.type}` : '';
            }
        });

        canvas.addEventListener('mouseup', () => {
            isDragging = false;
            dragNode = null;
            canvas.style.cursor = 'grab';
        });

        canvas.addEventListener('mouseleave', () => {
            isDragging = false;
            dragNode = null;
        });

        // Start animation
        animate();
    }

    // ============================================================
    // NAVIGATION & INTERACTION
    // ============================================================

    function navigate(page) {
        currentPage = page;
        activeTab = 'overview';
        selectedTheme = null;
        render();
        updateNav();
    }

    function openTheme(themeId) {
        selectedTheme = themeId;
        currentPage = 'theme-detail';
        activeTab = 'overview';
        render();
        updateNav();
    }

    function backToThemes() {
        currentPage = 'themes';
        selectedTheme = null;
        render();
        updateNav();
    }

    function setTab(tab) {
        activeTab = tab;
        const theme = analysisData.themes.find(t => t.theme_id === selectedTheme);
        if (theme) {
            document.getElementById('tab-content').innerHTML = renderTabContent(theme);
        }
        // Update tab buttons
        document.querySelectorAll('.tab').forEach(btn => {
            btn.classList.toggle('active', btn.textContent.toLowerCase().includes(tab) ||
                (tab === 'overview' && btn.textContent === 'Overview') ||
                (tab === 'voices' && btn.textContent === 'Participant Perspective') ||
                (tab === 'stakeholders' && btn.textContent === 'Stakeholders') ||
                (tab === 'implications' && btn.textContent === 'Implications'));
        });
    }

    // ============================================================
    // CODE DETAIL PAGE
    // ============================================================

    function openCode(codeName) {
        selectedCode = codeName;
        currentPage = 'code-detail';
        render();
        updateNav();
    }

    function backToCodeAnalytics() {
        currentPage = 'codes';
        selectedCode = null;
        render();
        updateNav();
    }

    function renderCodeDetail() {
        const codeName = selectedCode;
        const codeData = analysisData.codes.frequencies[codeName];
        if (!codeData) return '<p>Code not found</p>';

        // Get associated themes
        const associatedThemes = analysisData.themes.filter(theme =>
            theme.supporting_codes && theme.supporting_codes.includes(codeName)
        );

        // Get co-occurring codes from edges
        const cooccurrences = [];
        analysisData.codes.edges.forEach(edge => {
            if (edge.source === codeName) {
                cooccurrences.push({ code: edge.target, weight: edge.weight });
            } else if (edge.target === codeName) {
                cooccurrences.push({ code: edge.source, weight: edge.weight });
            }
        });
        // Sort by weight and deduplicate
        cooccurrences.sort((a, b) => b.weight - a.weight);
        const uniqueCooccurrences = cooccurrences.filter((item, index, self) =>
            index === self.findIndex(t => t.code === item.code)
        ).slice(0, 10);

        const typeColor = codeData.type === 'deductive' ? '#274C77' : '#6096BA';
        const typeLabel = codeData.type === 'deductive' ? 'Deductive (Theory-driven)' : 'Inductive (Data-driven)';

        return `
            <button class="back-button" onclick="backToCodeAnalytics()">
                <svg width="16" height="16" fill="none" stroke="currentColor" stroke-width="2" viewBox="0 0 24 24"><path d="M19 12H5M12 19l-7-7 7-7"/></svg>
                Back to Coding
            </button>

            <div class="code-detail-header">
                <h2>${codeName}</h2>
                <p style="color: var(--color-neutral); margin-bottom: 0.5rem;">${typeLabel}</p>
                <div class="code-stats-row">
                    <div class="code-stat-item">
                        <div class="value">${codeData.frequency}</div>
                        <div class="label">Applications</div>
                    </div>
                    <div class="code-stat-item">
                        <div class="value">${associatedThemes.length}</div>
                        <div class="label">Associated Themes</div>
                    </div>
                    <div class="code-stat-item">
                        <div class="value">${uniqueCooccurrences.length}</div>
                        <div class="label">Co-occurring Codes</div>
                    </div>
                </div>
            </div>

            <!-- Associated Themes -->
            <div class="card" style="margin-bottom: 1.5rem;">
                <h3 style="margin-bottom: 1rem;">Associated Themes</h3>
                ${associatedThemes.length > 0 ? `
                    <p style="color: var(--color-neutral); font-size: 0.9rem; margin-bottom: 1rem;">This code supports the following themes:</p>
                    <div>
                        ${associatedThemes.map(theme => {
                            const themeName = theme.theme_name.replace(/^Theme \\d+:\\s*/, '');
                            return `<span class="theme-chip" onclick="openTheme(${theme.theme_id})">
                                <strong>${theme.theme_id}.</strong> ${escapeHtml(themeName)}
                            </span>`;
                        }).join('')}
                    </div>
                ` : `
                    <p style="color: var(--color-neutral); font-style: italic;">This code is not a primary supporting code for any theme, but may appear in chunk-level coding.</p>
                `}
            </div>

            <!-- Co-occurring Codes -->
            <div class="card">
                <h3 style="margin-bottom: 1rem;">Co-occurring Codes</h3>
                ${uniqueCooccurrences.length > 0 ? `
                    <p style="color: var(--color-neutral); font-size: 0.9rem; margin-bottom: 1rem;">Codes that frequently appear alongside this code in the same data chunks:</p>
                    <div>
                        ${uniqueCooccurrences.map(item => {
                            const itemType = analysisData.codes.frequencies[item.code]?.type || 'deductive';
                            const itemColor = itemType === 'deductive' ? '#274C77' : '#6096BA';
                            return `
                                <div class="cooccurrence-item">
                                    <span class="cooccurrence-code" style="color: ${itemColor};" onclick="openCode('${item.code}')">${item.code}</span>
                                    <span class="cooccurrence-weight">${item.weight} co-occurrences</span>
                                </div>
                            `;
                        }).join('')}
                    </div>
                ` : `
                    <p style="color: var(--color-neutral); font-style: italic;">No co-occurrence data available for this code.</p>
                `}
            </div>
        `;
    }

    function filterCodes(searchTerm) {
        const rows = document.querySelectorAll('#code-table tbody tr');
        const term = searchTerm.toLowerCase();
        rows.forEach(row => {
            const name = row.getAttribute('data-name');
            row.style.display = name.includes(term) ? '' : 'none';
        });
    }

    function updateNav() {
        document.querySelectorAll('.nav-item').forEach(item => {
            const page = item.getAttribute('data-page');
            item.classList.toggle('active',
                page === currentPage ||
                (page === 'themes' && currentPage === 'theme-detail') ||
                (page === 'codes' && currentPage === 'code-detail'));
        });
    }

    function render() {
        const content = document.getElementById('page-content');

        switch(currentPage) {
            case 'dashboard':
                content.innerHTML = renderDashboard();
                break;
            case 'themes':
                content.innerHTML = renderThematicAnalysis();
                break;
            case 'theme-detail':
                content.innerHTML = renderThemeDetail();
                break;
            case 'codes':
                content.innerHTML = renderCodeAnalytics();
                break;
            case 'code-detail':
                content.innerHTML = renderCodeDetail();
                break;
            case 'network':
                content.innerHTML = renderNetwork();
                setTimeout(drawNetwork, 100);
                break;
            case 'insights':
                content.innerHTML = renderInsights();
                break;
            default:
                content.innerHTML = renderDashboard();
        }
    }

    // ============================================================
    // INITIALIZATION
    // ============================================================

    document.addEventListener('DOMContentLoaded', () => {
        // Set up navigation
        document.querySelectorAll('.nav-item').forEach(item => {
            item.addEventListener('click', () => {
                const page = item.getAttribute('data-page');
                navigate(page);
            });
        });

        // Initial render
        render();
    });
    </script>
</body>
</html>
'''

        # Combine HTML with embedded data
        html_content = html_start + "    const analysisData = " + json_data + ";\n" + html_end

        # Write the file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(html_content)

        print(f"✅ HTML dashboard exported: {filename}")


    def export_all(self, output_subfolder=None):
        """Export all formats: Word, XLSX, JSON, and HTML dashboard."""
        import os
        timestamp = config['timestamp']
        # Use separate folder for thematic analysis outputs
        output_folder = '2_thematic_analysis'
        if output_subfolder:
            output_folder = os.path.join(output_folder, output_subfolder)

        # Create folder if it doesn't exist
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        print("\n💾 EXPORTING THEMATIC ANALYSIS")
        print("-" * 40)

        word_file = f"{output_folder}/thematic_analysis_report_{timestamp}.docx"
        self.export_word_document(word_file)
        print(f"Word document: {word_file}")

        xlsx_file = f"{output_folder}/coded_data_with_themes_{timestamp}.xlsx"
        self.export_themed_data(xlsx_file)
        print(f"Themed data XLSX: {xlsx_file}")

        json_file = f"{output_folder}/thematic_analysis_{timestamp}.json"
        self.export_json(json_file)
        print(f"JSON file: {json_file}")

        html_file = f"{output_folder}/qualitative_analysis_dashboard_{timestamp}.html"
        self.export_html_dashboard(html_file)
        print(f"HTML dashboard: {html_file}")

        return {
            'word': word_file,
            'xlsx': xlsx_file,
            'json': json_file,
            'html': html_file
        }


def run_thematic_analysis(results):
    """
    Run thematic analysis on coding results.
    Handles both single-stance and multi-stance results.
    """
    if results is None:
        print("No results to analyze. Run run_complete_analysis() first.")
        return None

    # ── Multi-stance mode ──
    if isinstance(results, dict) and results.get('multi_stance'):
        stance_results = results['stance_results']
        all_deep = {}

        # Run thematic analysis per stance sequentially so console output stays readable
        for sk, res in stance_results.items():
            print(f"\n🎯 Running thematic analysis for stance: {sk}")
            try:
                analyzer = ThematicAnalyzer(
                    df_coded=res['df_coded'],
                    themes_results=res['themes'],
                    code_patterns=res['code_patterns'],
                    integration_stats=res['integration_stats'],
                    inductive_results=res['inductive_results'],
                    coder=None,
                    api_key=config['api_key']
                )
                analyzer.run_deep_analysis()
                export_files = analyzer.export_all(output_subfolder=f"stance_{sk}")
                all_deep[sk] = {
                    'analyzer': analyzer,
                    'theme_mappings': analyzer.theme_mappings,
                    'detailed_analyses': analyzer.detailed_analyses,
                    'export_files': export_files
                }
                print(f"✅ Thematic analysis complete for stance: {sk}")
            except Exception as e:
                print(f"❌ Thematic analysis failed for stance {sk}: {e}")

        # Run cross-stance thematic comparison
        cross_thematic = run_cross_stance_thematic_comparison(all_deep, config['api_key'])

        # Run cross-stance coding comparison
        cross_exports = run_cross_stance_comparison(stance_results)

        print("\n" + "=" * 50)
        print("🎯 MULTI-STANCE THEMATIC ANALYSIS COMPLETE")
        print("=" * 50)
        print(f"\nPer-stance reports exported to: 2_thematic_analysis/stance_*/")
        print(f"Cross-stance reports exported to: {config['output_folder']}/cross_stance/")

        return {
            'multi_stance': True,
            'stance_analyses': all_deep,
            'cross_thematic': cross_thematic,
            'cross_exports': cross_exports
        }

    # ── Single-stance mode (original behavior) ──
    analyzer = ThematicAnalyzer(
        df_coded=results['df_coded'],
        themes_results=results['themes'],
        code_patterns=results['code_patterns'],
        integration_stats=results['integration_stats'],
        inductive_results=results['inductive_results'],
        coder=None,
        api_key=config['api_key']
    )

    analyzer.run_deep_analysis()
    export_files = analyzer.export_all()

    print("\n" + "=" * 50)
    print("🎯 THEMATIC ANALYSIS COMPLETE")
    print("=" * 50)
    print(f"\nFiles exported to: 2_thematic_analysis/")

    return {
        'analyzer': analyzer,
        'theme_mappings': analyzer.theme_mappings,
        'detailed_analyses': analyzer.detailed_analyses,
        'export_files': export_files
    }


def run_cross_stance_thematic_comparison(stance_analyses, api_key):
    """
    Compare themes across stances using Claude to identify recurring,
    stance-specific, and convergent/divergent themes.
    """
    if len(stance_analyses) < 2:
        print("⚠️ Cross-stance thematic comparison requires at least 2 stances. Skipping.")
        return None

    print("\n🔀 CROSS-STANCE THEMATIC COMPARISON")
    print("=" * 50)

    client = anthropic.Anthropic(api_key=api_key)

    # Build theme summaries per stance
    stance_theme_text = []
    for sk, deep in stance_analyses.items():
        mappings = deep.get('theme_mappings', {})
        themes_list = []
        for tname, tinfo in mappings.items():
            insight = tinfo.get('core_insight', '')[:200]
            themes_list.append(f"  - {tname}: {insight}")
        stance_theme_text.append(
            f"STANCE: {sk} ({len(mappings)} themes)\n" +
            "\n".join(themes_list)
        )

    prompt = f"""You are a qualitative research methodologist comparing thematic analysis results across epistemological stances.

{chr(10).join(stance_theme_text)}

Provide a structured comparison:

## RECURRING THEMES
Themes or theme clusters that appear across multiple stances (possibly with different names but similar substance).

## STANCE-SPECIFIC THEMES
Themes that only emerged under one particular epistemic lens, and what this reveals about that stance's analytic affordances.

## CONVERGENCE PATTERNS
Where do themes from different stances point to the same underlying phenomenon? What does this multi-perspective convergence suggest?

## DIVERGENCE PATTERNS
Where do stances produce genuinely different or contradictory thematic interpretations? What does this tell us?

## SYNTHESIS
A 3-5 sentence synthesis of what multi-stance thematic analysis reveals that no single stance could capture alone.

Be specific — reference theme names and stances from the data above."""

    try:
        response = api_call_with_backoff(
            client,
            model=config['ai_model'],
            max_tokens=3000,
            temperature=0.3,
            messages=[{"role": "user", "content": prompt}]
        )
        comparison_text = response.content[0].text

        # Export to Word
        cross_folder = os.path.join(config['output_folder'], 'cross_stance')
        os.makedirs(cross_folder, exist_ok=True)
        word_path = f"{cross_folder}/cross_stance_thematic_comparison_{config['timestamp']}.docx"

        doc = Document()
        title = doc.add_heading('Cross-Stance Thematic Comparison', 0)
        title.alignment = WD_ALIGN_PARAGRAPH.CENTER
        doc.add_paragraph()
        p = doc.add_paragraph()
        p.add_run('Stances compared: ').bold = True
        p.add_run(', '.join(stance_analyses.keys()))
        doc.add_page_break()
        _add_md_content(doc, comparison_text)
        doc.save(word_path)
        print(f"✅ Cross-stance thematic comparison: {word_path}")

        return comparison_text

    except Exception as e:
        print(f"❌ Cross-stance thematic comparison error: {e}")
        return None


print("Thematic analysis functions loaded!")

## Execute

*Run the coding analysis followed by thematic analysis. In single-codebook mode, this analyzes one codebook against the transcript. In multi-stance mode, each codebook is processed in parallel, followed by cross-stance comparison and synthesis. Outputs include Word, Excel, JSON, and HTML per stance, plus cross-stance comparison reports.*

In [ ]:
# Run the complete analysis
results = run_complete_analysis()

In [ ]:
# Run thematic analysis (generates Word, CSV, JSON, and HTML outputs)
# In multi-stance mode, this also runs cross-stance thematic comparison
deep_results = run_thematic_analysis(results)

## Download Your Results

*Package the outputs from both analysis folders into a single zip archive. In Google Colab the download starts automatically; otherwise the archive is saved to the working directory.*

In [ ]:
# Package all analysis outputs into a single zip archive
import zipfile

zip_filename = 'qualitative_analysis_results.zip'
output_folders = ['1_coding_and_initial_themes', '2_thematic_analysis']

file_count = 0
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in output_folders:
        if not os.path.isdir(folder):
            print(f"⚠️ Folder not found (skipped): {folder}")
            continue
        for root, dirs, files in os.walk(folder):
            for name in files:
                zf.write(os.path.join(root, name))
                file_count += 1

print(f"✅ Zip archive created: {zip_filename} ({file_count} files)")

try:
    from google.colab import files
    files.download(zip_filename)
except ImportError:
    print(f"📁 Download manually from: {os.path.abspath(zip_filename)}")